# snake-arena · ACKTR sem calibrar a região de confiança — o que se perde

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/voaneves/snake-arena/blob/main/notebooks/98_acktr_kl_nominal.ipynb)

O braço de controle da calibração: `kl_max` volta a ser um alvo **nominal** de 0,002, e o que a rede entrega é ~0,014 — o fator sistemático entre a Fisher aproximada e a KL da política de verdade. Foi essa a configuração até agosto, e ela produziu 83,91 num Colab e 64,53 num Kaggle **com a mesma semente**: o fator não controlado muda com o hardware. Aqui a medição fica registrada em vez de virar anedota. Compare com `08_acktr` na mesma semente.

**Este notebook é autocontido.** Não precisa clonar nada: o ambiente, a rede, o protocolo
de avaliação e o agente estão todos aqui dentro. O código do núcleo é **gerado a partir do
pacote** ([`voaneves/snake-arena`](https://github.com/voaneves/snake-arena)) e é byte a byte igual
em todos os notebooks — é isso que torna as curvas comparáveis.

`Runtime → Change runtime type → GPU (T4)` antes de rodar.

Assinatura do código gerado: `9e11e1827d746f73`


In [ ]:
# @title Ambiente
import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import json, math, time, glob, csv, platform, subprocess, sys, shutil, argparse
from dataclasses import dataclass, field, asdict

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers, ops, regularizers

print("TensorFlow", tf.__version__, "| Keras", keras.__version__,
      "| backend", keras.backend.backend())
GPUS = tf.config.list_physical_devices("GPU")
print("GPU:", GPUS or "nenhuma — vai rodar em CPU, muito mais lento")
for g in GPUS:
    tf.config.experimental.set_memory_growth(g, True)


## O núcleo, gerado a partir do pacote

A célula abaixo é **gerada**. Editá-la aqui não muda o repositório e faz o teste
`tests/test_notebooks.py` acusar divergência — o que é de propósito: é o que garante que
os 19 notebooks rodem exatamente o mesmo jogo, com a mesma régua.

Para mudar algo aqui, mude no pacote e rode `python tools/gerar_notebooks.py`.


In [ ]:
# ==== GERADO A PARTIR DO PACOTE — NÃO EDITE AQUI ====
# assinatura: 9e11e1827d746f73

from __future__ import annotations

# --- snakeai/plataforma.py ---
"""Onde o notebook está rodando — Colab, Kaggle ou máquina local.

Por que isto existe
-------------------
O mesmo `.ipynb` precisa rodar nos dois serviços gratuitos, e eles diferem exatamente nos
três pontos que decidem se um treino de horas sobrevive:

======================  ==============================  ==============================
                        Colab                           Kaggle
======================  ==============================  ==============================
pasta que persiste      Google Drive, montado à mão     ``/kaggle/working``, automático
retomar depois da queda  o Drive continua lá             anexar a saída da execução
                                                        anterior em ``/kaggle/input``
baixar o resultado      ``google.colab.files.download``  painel *Output*, sem código
======================  ==============================  ==============================

A detecção é por **capacidade observada**, não por variável de ambiente decorada: `kaggle`
só se `/kaggle/working` for gravável, `colab` só se além de `google.colab` importar
existir um `/content` gravável. Um notebook rodando em qualquer outro lugar cai no caso
`local` e continua funcionando — o que também é o que faz a suíte de testes conseguir
exercitar isto aqui.

E a ordem importa, por um motivo que só aparece rodando: **o Kaggle também consegue
importar `google.colab`**. Ele traz um módulo de compatibilidade cujo `drive.mount` existe
e levanta `NotImplementedError: Mounting drive is unsupported in this environment`. Uma
detecção que perguntasse "`google.colab` importa?" primeiro chamaria o Kaggle de Colab,
tentaria montar o Drive, falharia, e cairia no fallback `/content` — que no Kaggle não é
só volátil, é **invisível**: o painel *Output* mostra `/kaggle/working` e mais nada. O
treino roda até o fim e o resultado não existe em lugar nenhum. Daí a regra: quem tem
pasta persistente própria é perguntado primeiro, e importabilidade de um stub nunca conta
como capacidade.

O problema que o Kaggle resolve
-------------------------------
No Colab a sessão cai por inatividade e o teto de uso é opaco. O Kaggle tem cota semanal
de GPU declarada e um caminho **headless**: *Save Version → Save & Run All* roda o notebook
inteiro sem aba aberta, e a saída vira um artefato versionado. Para um treino de 5 M passos
que leva ~40 minutos, isso é a diferença entre "torcer para não cair" e "enfileirar e
buscar depois".

A contrapartida é que `/kaggle/working` **não** volta sozinho na sessão seguinte: ele vira
a *saída* daquela versão. Para continuar de onde parou, anexe a saída anterior como
entrada (*Add Input → Your Work → Notebook Output*) e `semear_checkpoints` faz o resto.
"""


import os
import shutil

__all__ = ["detecta", "pasta_de_trabalho", "semear_checkpoints", "entregar_arquivo",
           "resumo_plataforma", "COLAB", "KAGGLE", "LOCAL"]

COLAB, KAGGLE, LOCAL = "colab", "kaggle", "local"


def _gravavel(caminho):
    return os.path.isdir(caminho) and os.access(caminho, os.W_OK)


def detecta():
    """`"colab"`, `"kaggle"` ou `"local"`, por capacidade observada.

    O Kaggle é testado **primeiro** de propósito: ele importa `google.colab` (um módulo de
    compatibilidade cujo `drive.mount` só levanta `NotImplementedError`), então perguntar
    pelo Colab antes o classificaria errado e mandaria o treino escrever em `/content` —
    fora do painel *Output*, ou seja, resultado nenhum no fim. Importar não é capacidade;
    ter pasta persistente é.
    """
    if _gravavel("/kaggle/working"):
        return KAGGLE
    try:
        import google.colab  # noqa: F401,PLC0415
    except Exception:
        return LOCAL
    return COLAB if _gravavel("/content") else LOCAL


def pasta_de_trabalho(usar_drive="auto", nome="snake-arena", verbose=True):
    """A pasta onde checkpoints, `runs/` e export vão viver. **Sem nada para configurar.**

    `usar_drive="auto"` é o padrão e resolve tudo sozinho: no Colab tenta montar o Drive,
    no Kaggle usa `/kaggle/working`, no local usa o diretório atual. Passar `True` ou
    `False` força o comportamento no Colab e não faz diferença nos outros dois — o mesmo
    notebook roda nos três sem editar célula, que é o ponto.

    Se a montagem do Drive falhar (o usuário recusa a autorização, ou a sessão não tem
    navegador), cai para `/content` **avisando alto**: ali o treino roda, mas a queda da
    sessão leva os checkpoints junto, e descobrir isso depois de três horas é pior do que
    ler um aviso agora.
    """
    onde = detecta()

    if onde == COLAB and usar_drive is not False:
        try:
            from google.colab import drive  # noqa: PLC0415

            drive.mount("/content/drive")
            raiz = os.path.join("/content/drive/MyDrive", nome)
        except Exception as e:
            if usar_drive is True:
                raise
            print(f"AVISO: não consegui montar o Drive ({type(e).__name__}: {e}).")
            print("       Usando /content, que NÃO sobrevive à queda da sessão —")
            print("       se o treino cair, ele recomeça do zero.")
            raiz = os.path.join("/content", nome)
    elif onde == COLAB:
        raiz = os.path.join("/content", nome)
    elif onde == KAGGLE:
        raiz = os.path.join("/kaggle/working", nome)
    else:
        raiz = os.path.abspath(nome)

    os.makedirs(raiz, exist_ok=True)
    if verbose:
        print(f"plataforma: {onde} · pasta: {raiz}")
        if onde == KAGGLE:
            print("  lembre: /kaggle/working vira a SAÍDA desta versão. Para continuar "
                  "depois,\n  anexe esta saída como entrada da próxima execução.")
    return raiz


def semear_checkpoints(ckpt_dir, verbose=True):
    """Traz checkpoints de execuções anteriores anexadas em `/kaggle/input`.

    É isto que faz "retomar" funcionar no Kaggle. A sessão nova nasce com
    `/kaggle/working` vazio; o que sobreviveu está montado **somente leitura** em
    `/kaggle/input/<algum-nome>/`. Copiamos para `ckpt_dir` só o que ainda não existe lá —
    um checkpoint desta sessão sempre vence o de uma anterior, senão retomar andaria para
    trás.

    Devolve a lista do que foi copiado. Fora do Kaggle, lista vazia e nenhum efeito.
    """
    if detecta() != KAGGLE or not os.path.isdir("/kaggle/input"):
        return []

    os.makedirs(ckpt_dir, exist_ok=True)
    copiados = []
    for raiz, _, arquivos in os.walk("/kaggle/input"):
        if os.path.basename(raiz) != "checkpoints":
            continue
        for nome in arquivos:
            if not nome.endswith((".keras", ".json")):
                continue
            destino = os.path.join(ckpt_dir, nome)
            if os.path.exists(destino):
                continue
            shutil.copyfile(os.path.join(raiz, nome), destino)
            copiados.append(destino)

    if verbose and copiados:
        print(f"  [retomada] {len(copiados)} arquivo(s) de checkpoint vieram de "
              f"/kaggle/input")
    return copiados


def entregar_arquivo(caminho, verbose=True):
    """Entrega o arquivo ao usuário, do jeito que a plataforma permite.

    No Colab dispara o download pelo navegador — que só funciona com a aba aberta. No
    Kaggle não há o que disparar: o que está em `/kaggle/working` aparece sozinho no painel
    *Output*, e é justamente por isso que o Kaggle aguenta execução headless. No local, o
    arquivo já está no disco.

    Devolve `True` só quando um download foi realmente disparado.
    """
    onde = detecta()
    if onde == COLAB:
        try:
            from google.colab import files  # noqa: PLC0415

            files.download(caminho)
            return True
        except Exception as e:                       # aba fechada, sessão sem navegador
            if verbose:
                print(f"download automático não rolou ({type(e).__name__}: {e})")
    elif onde == KAGGLE and verbose:
        print("no Kaggle não há download automático: o arquivo já está no painel "
              "**Output**,\nà direita, e é baixável de lá mesmo com a aba fechada.")
    if verbose:
        print(f"arquivo: {caminho}")
    return False


def resumo_plataforma():
    """Dicionário com plataforma e aceleradores visíveis — vai para o `meta` do registro.

    O nome é longo de propósito. No notebook gerado todos os módulos viram **um espaço de
    nomes só**, e um `resumo()` aqui colidiria com o `resumo()` de `snakeai/nets/registry.py`
    — o último inlinado venceria e o outro sumiria sem erro nenhum.
    `tests/test_notebooks.py::test_no_two_inlined_modules_define_the_same_name` tranca isso.
    """
    info = {"plataforma": detecta()}
    try:
        import tensorflow as tf  # noqa: PLC0415

        gpus = tf.config.list_physical_devices("GPU")
        info["gpus"] = [g.name for g in gpus]
        info["n_gpus"] = len(gpus)
    except Exception:
        info["gpus"], info["n_gpus"] = [], 0
    return info


# --- snakeai/env/vec_snake.py ---
"""`VecSnake` — Snake vetorizado, N tabuleiros independentes evoluindo em lote.

Este módulo é **a fonte única de verdade do ambiente**. Todo algoritmo do `snake-arena`
treina e é avaliado aqui, sem exceção — é isso que torna as curvas comparáveis. Ele não
importa TensorFlow nem Keras: é NumPy puro, roda em qualquer lugar e é rápido o bastante
para que o gargalo do treino seja a GPU, não o jogo.

O truque que faz ser rápido: em vez de uma lista de posições por cobra, guardamos uma
grade `occ` de inteiros onde `occ[n, y, x]` é **quantos passos faltam para aquela célula
ficar livre**. A cabeça recebe `occ = comprimento`; a cada passo o mundo inteiro decrementa
em 1 e a cauda some sozinha. Tudo vira operação NumPy em lote sobre `(N, B, B)` — nada de
laço Python por cobra.

Como bônus, essa grade *já é* a feature mais informativa que existe para Snake: normalizada
por comprimento, ela diz à rede **quando** cada célula vai desocupar, que é exatamente a
informação necessária para a cobra passar rente ao próprio corpo sem se prender.

Convenções fixadas pelo contrato de comparabilidade (`docs/COMPARABILITY.md`):

* tabuleiro 10x10, `starve_base = 100`;
* observação `(N, B, B, 5)` egocêntrica;
* 3 ações relativas com máscara de morte imediata;
* recompensa `+1` comer, `-1` morrer, `0` passo;
* **score = comida comida**, começando em zero. Nunca comprimento.
"""


import numpy as np

__all__ = ["VecSnake", "DIRS", "TURN", "N_ACTIONS", "N_CHANNELS",
           "N_CHANNELS_COM_FOME", "DEFAULT_SEED"]

# Direções: 0=cima(-y), 1=direita(+x), 2=baixo(+y), 3=esquerda(-x)  (sentido horário)
DIRS = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]], dtype=np.int32)
# Ações relativas: 0=vira à esquerda, 1=segue reto, 2=vira à direita
TURN = np.array([-1, 0, 1], dtype=np.int32)

N_ACTIONS = 3

#: O contrato oficial. Não mude este número — versione o contrato.
N_CHANNELS = 5

#: Com o canal de fome ligado (`VecSnake(canal_fome=True)`). **Fora do contrato**, e é para
#: ser: qualquer execução assim tem que ir para a arena com `comparable=False`.
N_CHANNELS_COM_FOME = 6

DEFAULT_SEED = 42


class VecSnake:
    """`num_envs` tabuleiros independentes de Snake evoluindo em lote.

    Observação: `(num_envs, B, B, 5)` float32, **egocêntrica** — o tabuleiro é rotacionado
    para que a cobra sempre olhe para cima. Isso colapsa as 4 simetrias de rotação e deixa
    a rede ~4x mais eficiente em amostras.

    Canais
    ------
    0. corpo (binário, sem a cabeça)
    1. cabeça
    2. decaimento da cauda: `occ / comprimento` em (0, 1]
    3. comida
    4. plano constante = comprimento / B**2  (a rede precisa saber o quão longa está)

    Parâmetros
    ----------
    num_envs : int
        Quantos tabuleiros correm em paralelo.
    board_size : int
        Lado do tabuleiro. O contrato oficial usa 10.
    starve_base : int, opcional
        Paciência base antes de morrer de fome; o limite efetivo é
        `starve_base + 2 * comprimento`. Padrão: `board_size ** 2`.
    rng : np.random.Generator, opcional
        Gerador próprio. Passe um com semente fixa para reprodutibilidade.
    canal_fome : bool, opcional
        Liga um **sexto** canal com o relógio da fome. Padrão `False`, que é o contrato.

        Existe porque os 5 canais do contrato não contêm o contador de fome, e o limite é
        `starve_base + 2·comprimento` passos sem comer. Ou seja: dois estados visualmente
        idênticos, um com fome 5 e outro com fome 105, valem coisas diferentes e a rede não
        tem como saber. Os algoritmos sem modelo toleram — o retorno **real** pune andar em
        círculo, mesmo que o crítico não veja a causa. Um modelo do mundo não tem essa
        sorte: ele só pode sonhar o que a observação carrega, e inanição simplesmente não
        existe no sonho.

        Ligar isto **quebra a comparabilidade** com todas as curvas de 5 canais: a entrada
        da rede muda. Use apenas em execuções marcadas `comparable=False`, com o motivo em
        `caveat`.
    """

    def __init__(self, num_envs=256, board_size=10, starve_base=None, rng=None,
                 canal_fome=False):
        if board_size < 6:
            raise ValueError("tabuleiro pequeno demais para o corpo inicial (mínimo 6)")
        self.canal_fome = bool(canal_fome)
        self.n_channels = N_CHANNELS_COM_FOME if self.canal_fome else N_CHANNELS
        self.n = int(num_envs)
        self.b = int(board_size)
        self.cells = self.b * self.b
        self.starve_base = self.cells if starve_base is None else int(starve_base)
        self.rng = rng if rng is not None else np.random.default_rng(DEFAULT_SEED)

        self.occ = np.zeros((self.n, self.b, self.b), dtype=np.int32)
        self.head = np.zeros((self.n, 2), dtype=np.int32)
        self.food = np.zeros((self.n, 2), dtype=np.int32)
        self.dir = np.zeros(self.n, dtype=np.int32)
        self.length = np.zeros(self.n, dtype=np.int32)
        self.steps = np.zeros(self.n, dtype=np.int32)
        self.hunger = np.zeros(self.n, dtype=np.int32)
        self.score = np.zeros(self.n, dtype=np.int32)

        self._reset_idx(np.arange(self.n))

    # ------------------------------------------------------------------- reset
    def _reset_idx(self, idx):
        """Reinicia apenas os ambientes em `idx`, em lote."""
        if idx.size == 0:
            return
        k = idx.size
        b = self.b
        self.occ[idx] = 0
        # cabeça longe das bordas para caber o corpo inicial de 3
        self.head[idx] = self.rng.integers(2, b - 2, size=(k, 2), dtype=np.int32)
        self.dir[idx] = self.rng.integers(0, 4, size=k, dtype=np.int32)
        self.length[idx] = 3
        self.steps[idx] = 0
        self.hunger[idx] = 0
        self.score[idx] = 0

        d = DIRS[self.dir[idx]]                       # (k, 2)
        for back, ttl in ((0, 3), (1, 2), (2, 1)):    # cabeça, meio, cauda
            p = self.head[idx] - back * d
            np.clip(p, 0, b - 1, out=p)
            self.occ[idx, p[:, 0], p[:, 1]] = ttl

        self._spawn_food(idx)

    def _spawn_food(self, idx):
        """Sorteia comida uniformemente entre as células livres (vetorizado)."""
        if idx.size == 0:
            return
        free = self.occ[idx].reshape(idx.size, -1) == 0
        r = self.rng.random((idx.size, self.cells))
        r[~free] = -1.0
        flat = r.argmax(axis=1)
        self.food[idx, 0] = flat // self.b
        self.food[idx, 1] = flat % self.b

    def reset(self):
        """Reinicia todos os ambientes. Retorna `(obs, mask)`."""
        self._reset_idx(np.arange(self.n))
        return self.obs(), self.action_mask()

    # -------------------------------------------------------------- observação
    def limite_de_fome(self):
        """`(N,)` passos sem comer que matam. Cresce com o corpo: comer fica mais difícil."""
        return self.starve_base + 2 * self.length

    def _raw_planes(self):
        """Os 5 canais no referencial do tabuleiro (6 com `canal_fome`), antes da rotação."""
        b, n = self.b, self.n
        occ = self.occ
        body = (occ > 0).astype(np.float32)
        head = np.zeros((n, b, b), dtype=np.float32)
        rows = np.arange(n)
        head[rows, self.head[:, 0], self.head[:, 1]] = 1.0
        body -= head                                   # cabeça sai do canal de corpo
        decay = occ.astype(np.float32) / self.length[:, None, None].astype(np.float32)
        food = np.zeros((n, b, b), dtype=np.float32)
        food[rows, self.food[:, 0], self.food[:, 1]] = 1.0
        lenpl = np.broadcast_to(
            (self.length.astype(np.float32) / self.cells)[:, None, None], (n, b, b)
        )
        planos = [body, head, decay, food, lenpl]

        if self.canal_fome:
            # Plano constante com a fração do relógio já gasta: 0 = acabou de comer,
            # 1 = morre de fome neste passo. Normalizado pelo limite **efetivo**, que
            # cresce com o corpo — assim o número significa a mesma coisa do começo ao fim,
            # que é a mesma razão de o canal de comprimento existir.
            fome = self.hunger.astype(np.float32) / np.maximum(
                self.limite_de_fome().astype(np.float32), 1.0)
            planos.append(np.broadcast_to(fome[:, None, None], (n, b, b)))

        return np.stack(planos, axis=-1)

    def obs(self):
        """Planos rotacionados para o referencial da cabeça (sempre olhando p/ cima)."""
        raw = self._raw_planes()
        out = np.empty_like(raw)
        for k in range(4):
            m = self.dir == k
            if m.any():
                out[m] = np.rot90(raw[m], k=k, axes=(1, 2))
        return out

    # ----------------------------------------------------------------- máscara
    def _next_head(self, actions):
        """Posição e direção da cabeça se `actions` fosse aplicada agora."""
        nd = (self.dir + TURN[actions]) % 4
        return self.head + DIRS[nd], nd

    def _lethal(self, pos):
        """True onde a posição mata (parede ou corpo que ainda não desocupou)."""
        b = self.b
        oob = (pos[:, 0] < 0) | (pos[:, 0] >= b) | (pos[:, 1] < 0) | (pos[:, 1] >= b)
        safe_pos = np.where(oob[:, None], 0, pos)
        # a cauda vai embora neste passo -> célula com occ<=1 estará livre
        hit = self.occ[np.arange(self.n), safe_pos[:, 0], safe_pos[:, 1]] > 1
        return oob | (hit & ~oob)

    def _raw_mask(self):
        """`(N, 3)` bool sem o *override* de beco sem saída — a verdade nua."""
        mask = np.empty((self.n, N_ACTIONS), dtype=bool)
        for a in range(N_ACTIONS):
            pos, _ = self._next_head(np.full(self.n, a, dtype=np.int32))
            mask[:, a] = ~self._lethal(pos)
        return mask

    def dead_ends(self):
        """`(N,)` bool: True onde **todas** as três ações matam.

        Existe porque `action_mask()` não permite descobrir isso — lá, um beco sem saída
        aparece como "tudo liberado". Quem precisa distinguir (testes, diagnóstico, o
        filtro de segurança) pergunta aqui.
        """
        return ~self._raw_mask().any(axis=1)

    def action_mask(self):
        """`(N, 3)` bool: True = ação não mata imediatamente.

        Se as três matam, liberamos todas (a cobra morreu de qualquer jeito) — assim a
        distribuição nunca fica sem suporte e o log-prob não vira NaN. Use `dead_ends()`
        para saber quando esse caso ocorreu.
        """
        mask = self._raw_mask()
        mask[~mask.any(axis=1)] = True
        return mask

    # -------------------------------------------------------------------- step
    def step(self, actions, shaping_coef=0.0, gamma=0.99):
        """Avança todos os ambientes um passo.

        Retorna `(obs, mask, reward, done, info)`. Ambientes terminados são resetados
        automaticamente; `obs` já é o do episódio novo, e `info` guarda as estatísticas
        do episódio que acabou.

        `info` contém:
            scores      : score final dos episódios encerrados neste passo
            lengths     : duração em passos desses episódios
            wins        : quantos encheram o tabuleiro
            deaths      : quantos morreram por colisão
            starved     : quantos foram truncados por fome
            trunc_idx   : índices dos truncados por fome
            final_obs   : observação terminal dos truncados (para bootstrap do valor)
            final_mask  : máscara terminal dos truncados
        """
        n, b = self.n, self.b
        rows = np.arange(n)
        actions = np.asarray(actions, dtype=np.int32)

        d_old = np.abs(self.head - self.food).sum(axis=1).astype(np.float32)

        new_head, new_dir = self._next_head(actions)
        dead = self._lethal(new_head)
        new_head = np.where(dead[:, None], self.head, new_head)  # congela quem morreu

        ate = (
            (~dead)
            & (new_head[:, 0] == self.food[:, 0])
            & (new_head[:, 1] == self.food[:, 1])
        )

        # cauda anda quando não comeu
        moved = ~ate & ~dead
        self.occ[moved] = np.maximum(self.occ[moved] - 1, 0)

        self.length += ate.astype(np.int32)
        self.score += ate.astype(np.int32)
        alive = ~dead
        self.head[alive] = new_head[alive]
        self.dir[alive] = new_dir[alive]
        self.occ[rows[alive], self.head[alive, 0], self.head[alive, 1]] = self.length[alive]

        self.steps += 1
        self.hunger = np.where(ate, 0, self.hunger + 1)

        won = self.length >= self.cells
        need_food = ate & ~won
        if need_food.any():
            self._spawn_food(np.nonzero(need_food)[0])

        starved = (self.hunger >= self.limite_de_fome()) & ~dead & ~won

        # ---- recompensa
        reward = np.zeros(n, dtype=np.float32)
        reward += ate.astype(np.float32)
        reward -= dead.astype(np.float32)
        reward += won.astype(np.float32) * 2.0
        reward -= starved.astype(np.float32) * 0.5
        if shaping_coef > 0.0:
            # o delta só faz sentido quando a comida não mudou de lugar
            d_new = np.abs(self.head - self.food).sum(axis=1).astype(np.float32)
            phi_old = -d_old / b
            phi_new = -d_new / b
            delta = np.where(dead | won | ate, 0.0, gamma * phi_new - phi_old)
            reward += shaping_coef * delta

        done = dead | won | starved

        # Truncamento por fome: o episódio *continuaria*, então precisamos do valor do
        # estado final para fazer bootstrap. Como o env reseta sozinho, guardamos a
        # observação terminal antes do reset (custa uma passada extra, só quando ocorre).
        starved_idx = np.nonzero(starved)[0]
        final_obs = final_mask = None
        if starved_idx.size:
            final_obs = self.obs()[starved_idx]
            final_mask = self.action_mask()[starved_idx]

        info = {
            "scores": self.score[done].copy(),
            "lengths": self.steps[done].copy(),
            "wins": int(won.sum()),
            "deaths": int(dead.sum()),
            "starved": int(starved.sum()),
            "trunc_idx": starved_idx,
            "final_obs": final_obs,
            "final_mask": final_mask,
        }
        self._reset_idx(np.nonzero(done)[0])
        return self.obs(), self.action_mask(), reward, done, info

    # -------------------------------------------------------------- utilidades
    def free_space_from(self, env_i, pos):
        """Flood-fill: quantas células livres são alcançáveis a partir de `pos`.

        Usado só no filtro de segurança da inferência, nunca no treino.
        """
        b = self.b
        occ = self.occ[env_i]
        seen = np.zeros((b, b), dtype=bool)
        stack = [(int(pos[0]), int(pos[1]))]
        seen[pos[0], pos[1]] = True
        count = 0
        while stack:
            y, x = stack.pop()
            count += 1
            for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                ny, nx = y + dy, x + dx
                if 0 <= ny < b and 0 <= nx < b and not seen[ny, nx] and occ[ny, nx] <= 1:
                    seen[ny, nx] = True
                    stack.append((ny, nx))
        return count

    # -------------------------------------------------------- estado serializável
    #: Os campos que definem completamente o estado do jogo. Nada fora desta lista
    #: influencia o futuro — é o que torna a busca em árvore possível.
    CAMPOS_ESTADO = ("occ", "head", "food", "dir", "length", "steps", "hunger", "score")

    def get_state(self):
        """Cópia do estado de todos os ambientes, como dicionário de arrays.

        Existe para a busca em árvore: o MCTS precisa voltar a um nó anterior, e a única
        forma honesta de fazer isso é restaurar o estado exato. Snake é determinístico e de
        informação perfeita, então este dicionário *é* o nó da árvore.
        """
        return {c: getattr(self, c).copy() for c in self.CAMPOS_ESTADO}

    def set_state(self, estado):
        """Restaura o estado. Não valida por desempenho — use `check_invariants` em teste."""
        for c in self.CAMPOS_ESTADO:
            getattr(self, c)[...] = estado[c]
        return self

    def estado_de(self, i):
        """O estado de um único ambiente, como dicionário de arrays sem eixo de lote."""
        return {c: getattr(self, c)[i].copy() for c in self.CAMPOS_ESTADO}

    def escrever_estado(self, i, estado_unico):
        for c in self.CAMPOS_ESTADO:
            getattr(self, c)[i] = estado_unico[c]

    # ------------------------------------------------------------- introspecção
    def check_invariants(self):
        """Levanta `AssertionError` se o estado interno estiver inconsistente.

        Barato o bastante para rodar em testes e em depuração; nunca no laço de treino.
        """
        assert (self.occ >= 0).all(), "occ negativo"
        assert ((self.occ > 0).sum(axis=(1, 2)) == self.length).all(), \
            "número de células ocupadas não bate com o comprimento"
        assert (self.occ.reshape(self.n, -1).max(axis=1) == self.length).all(), \
            "a cabeça deveria ser a célula de maior occ"
        rows = np.arange(self.n)
        assert (self.occ[rows, self.head[:, 0], self.head[:, 1]] == self.length).all(), \
            "occ na posição da cabeça não é o comprimento"
        occupied_food = self.occ[rows, self.food[:, 0], self.food[:, 1]] > 0
        assert not occupied_food.any() or (self.length >= self.cells).any(), \
            "comida dentro do corpo"
        assert (self.score == self.length - 3).all(), \
            "score deve ser comprimento - 3"

    def __repr__(self):
        extra = ", canal_fome=True" if self.canal_fome else ""
        return (
            f"VecSnake(num_envs={self.n}, board_size={self.b}, "
            f"starve_base={self.starve_base}{extra})"
        )


# --- snakeai/otimizadores.py ---
"""Otimizadores — o eixo de ablação de primeira ordem.

Onde foi parar o K-FAC
----------------------
Quatro notebooks do `colab-rl` tentaram K-FAC e nenhum roda: dependiam de
`tensorflow.contrib.kfac`, que não existe desde o TensorFlow 2. Ele **voltou**, mas não
para cá: mora em `snakeai/kfac.py` e é usado pelo `ACKTR` (`snakeai/agents/acktr.py`).

O motivo de não estar neste eixo é estrutural, não histórico. `cria_otimizador` recebe um
nome e um learning rate; um `keras.optimizers.Optimizer` recebe pares `(gradiente,
variável)`. O K-FAC precisa das **ativações de entrada** e dos **gradientes de
pré-ativação** de cada camada — coisas que só existem durante o forward/backward e que
nenhum otimizador do Keras enxerga. Espremê-lo nesta assinatura exigiria refazer o forward
por dentro do otimizador, que foi o que a API Keras do `tensorflow/kfac` fazia (arquivada
em 19/04/2026).

A **pergunta** por trás daqueles notebooks continua sendo boa: *o otimizador importa?* Este
módulo é a resposta de primeira ordem — um eixo de ablação com otimizadores que existem,
funcionam em Keras 3 e cobrem escolhas de projeto diferentes. A resposta de segunda ordem é
a curva do ACKTR ao lado da do A2C, que é o mesmo algoritmo com a curvatura ligada:

===========  ==============================================================
nome         o que muda
===========  ==============================================================
``rmsprop``  o que o repositório antigo usava na maioria dos experimentos
``adam``     momento + escala adaptativa; o padrão de fato em RL
``adamw``    Adam com decaimento de peso desacoplado — regulariza sem mexer
             na escala adaptativa, ao contrário do `weight_decay` clássico
``lion``     só o **sinal** do momento; usa muito menos memória de estado e
             costuma preferir LR ~10x menor
``sgd``      o controle: momento e nada mais. Se o eixo não separar nada,
             este aqui denuncia
===========  ==============================================================

Todos entram pelo mesmo lugar: `cfg.optimizer = "adamw"`. O resto do experimento não muda,
que é o que torna a comparação uma ablação e não uma anedota.
"""


import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras

__all__ = ["OTIMIZADORES", "cria_otimizador", "LR_SUGERIDO"]

OTIMIZADORES = ("adam", "adamw", "rmsprop", "lion", "sgd")

#: Multiplicador de learning rate típico de cada otimizador, relativo ao Adam. O Lion usa
#: só o sinal do momento, então o passo tem magnitude constante e o LR precisa ser bem
#: menor; o SGD, sem escala adaptativa, precisa de bem maior. Comparar otimizadores com o
#: mesmo LR não mede otimizador — mede quem tolera aquele LR específico.
LR_SUGERIDO = {"adam": 1.0, "adamw": 1.0, "rmsprop": 1.0, "lion": 0.1, "sgd": 30.0}


def cria_otimizador(nome, learning_rate, clipnorm=None, weight_decay=1e-4, **kw):
    """Devolve um `keras.optimizers.Optimizer` pelo nome.

    `learning_rate` é o valor **base**; aplique `LR_SUGERIDO[nome]` por fora se quiser a
    escala típica de cada um. Deixar isso explícito é de propósito: um experimento que
    ajusta o LR junto com o otimizador está medindo os dois ao mesmo tempo, e precisa
    dizer isso.
    """
    nome = nome.lower()
    comum = {"learning_rate": learning_rate}
    if clipnorm is not None:
        comum["clipnorm"] = clipnorm

    if nome == "adam":
        return keras.optimizers.Adam(epsilon=1e-5, **comum, **kw)
    if nome == "adamw":
        return keras.optimizers.AdamW(epsilon=1e-5, weight_decay=weight_decay,
                                      **comum, **kw)
    if nome == "rmsprop":
        return keras.optimizers.RMSprop(rho=0.95, epsilon=1e-5, **comum, **kw)
    if nome == "lion":
        return keras.optimizers.Lion(beta_1=0.9, beta_2=0.99, **comum, **kw)
    if nome == "sgd":
        return keras.optimizers.SGD(momentum=0.9, nesterov=True, **comum, **kw)
    raise ValueError(f"otimizador desconhecido: {nome!r}. Use um de {OTIMIZADORES}")


# --- snakeai/eval.py ---
"""Avaliação — o protocolo oficial do benchmark.

Este módulo responde à única pergunta que importa: **quanto esse agente tira, de verdade?**
Ele é deliberadamente independente de TensorFlow e Keras — recebe uma *função de política*,
não um modelo. Isso permite avaliar qualquer coisa pelo mesmo caminho: uma rede Keras, uma
tabela, uma heurística escrita à mão, ou a política aleatória que define o piso. E permite
testar a avaliação sem GPU.

Protocolo fixado pelo contrato de comparabilidade (`docs/COMPARABILITY.md`):

* 1.000 episódios, tabuleiro 10x10;
* política **greedy** (sem exploração);
* `seed = 123`;
* **sem** filtro de segurança na curva principal;
* métrica = `score` (comida comida), nunca comprimento.

Sobre o viés que este módulo corrige
------------------------------------
A forma ingênua de avaliar é rodar N ambientes em paralelo e parar assim que 1.000
episódios terminarem. Isso **subestima o agente**: episódios curtos terminam primeiro e
entram na amostra, enquanto os longos — que são justamente os bons — ainda estão correndo
quando a contagem fecha. Quanto melhor o agente, pior o viés.

A correção é simples: cada ambiente contribui com o mesmo número de episódios (os
primeiros que ele terminar), em vez de a amostra ser "os primeiros a terminar no total".
"""


import math

import numpy as np


__all__ = [
    "MASK_NEG",
    "evaluate",
    "random_baseline",
    "random_policy",
    "keras_policy",
    "apply_safety_filter",
    "verdict",
]

MASK_NEG = -1e9

#: Piso documentado no README: política aleatória com máscara, 1.000 episódios, 10x10.
PISO_ALEATORIO_10X10 = 1.08


# --------------------------------------------------------------------- políticas
def random_policy(rng=None):
    """Política uniforme sobre as ações permitidas — o piso do benchmark.

    Não é "aleatória pura": ela respeita a máscara, ou seja, já evita a morte imediata.
    É o piso honesto, porque qualquer agente do benchmark também tem a máscara.
    """
    rng = rng if rng is not None else np.random.default_rng(0)

    def politica(obs, mask):
        return np.where(mask, rng.random(mask.shape), -np.inf).astype(np.float32)

    return politica


def keras_policy(model, batch_size=None):
    """Embrulha um modelo Keras (actor-critic) numa função de política.

    O import de TensorFlow acontece aqui dentro, de propósito: quem só quer avaliar uma
    heurística não precisa ter TF instalado.
    """
    import tensorflow as tf  # noqa: PLC0415  (lazy de propósito)

    @tf.function(reduce_retracing=True)
    def _forward(obs, mask):
        saida = model(obs, training=False)
        logits = saida[0] if isinstance(saida, (list, tuple)) else saida
        return tf.where(mask, logits, tf.fill(tf.shape(logits), MASK_NEG))

    def politica(obs, mask):
        return _forward(
            tf.convert_to_tensor(obs), tf.convert_to_tensor(mask)
        ).numpy()

    return politica


# ------------------------------------------------------------- filtro de segurança
def apply_safety_filter(env: VecSnake, logits, margin=1.0, penalty=50.0):
    """Penaliza ações que deixariam a cobra num bolso menor que o próprio corpo.

    Pós-processamento de inferência, **não aprendido**: entre as ações que a rede
    considera boas, desencoraja as que se fecham num espaço sem saída (flood-fill a partir
    da nova cabeça). Por isso ele nunca entra na curva principal do benchmark — vira
    coluna separada da tabela.

    A penalidade é grande mas finita: se *todas* as opções forem ruins, a ordem relativa
    que a rede preferia é preservada e o agente escolhe a menos pior.
    """
    out = np.array(logits, dtype=np.float32, copy=True)
    for a in range(N_ACTIONS):
        pos, _ = env._next_head(np.full(env.n, a, dtype=np.int32))
        lethal = env._lethal(pos)
        for i in range(env.n):
            if lethal[i]:
                out[i, a] = MASK_NEG
            elif env.free_space_from(i, pos[i]) < margin * env.length[i]:
                out[i, a] -= penalty
    return out


# ------------------------------------------------------------------- avaliação
def evaluate(
    policy,
    board_size=10,
    episodes=1000,
    num_envs=250,
    greedy=True,
    safety=False,
    seed=123,
    max_steps=200_000,
    rng=None,
    canal_fome=False,
):
    """Roda o protocolo oficial e devolve `(stats, scores)`.

    Parâmetros
    ----------
    policy : callable
        `policy(obs, mask) -> logits (N, 3)`. Já deve aplicar a máscara aos logits;
        `evaluate` não confia nisso e reaplica de qualquer forma.
    episodes : int
        Quantos episódios compõem a amostra. O contrato usa 1.000.
    greedy : bool
        `True` = argmax (o padrão do benchmark). `False` = amostra da softmax.
    safety : bool
        Liga o flood-fill. Fora da curva principal, por construção.
    seed : int
        Semente do ambiente. Fixa em 123 no contrato, para que a sequência de comidas
        seja a mesma para todos os algoritmos.
    canal_fome : bool
        Constrói o ambiente de avaliação com o 6º canal. **Fora do contrato** — só existe
        para que uma política treinada com `canal_fome=True` receba a observação que a
        rede espera. Se ficar `False` com uma rede de 6 canais, o erro aparece na primeira
        chamada da política, como incompatibilidade de forma (5 vs 6). A dinâmica do jogo
        é idêntica nos dois casos: o canal só muda a observação, não o ambiente, então o
        número continua sendo do mesmo protocolo — o que muda é a entrada da rede, e por
        isso a curva não é comparável com as de 5 canais.

    Cada ambiente contribui com o mesmo número de episódios — ver a nota sobre viés no
    topo do módulo.
    """
    env = VecSnake(num_envs, board_size, rng=np.random.default_rng(seed),
                   canal_fome=canal_fome)
    rng = rng if rng is not None else np.random.default_rng(seed + 1)
    obs, mask = env.reset()
    apos_passo = getattr(policy, "apos_passo", None)

    por_env = math.ceil(episodes / num_envs)
    coletados = [[] for _ in range(num_envs)]
    #: Por que cada episódio da amostra terminou. Score sozinho não distingue "o agente
    #: joga mal" de "o agente anda em círculo": um DQN greedy no começo do treino tira
    #: 0,05 morrendo **100% por fome**, e a leitura correta disso não é "não aprendeu", é
    #: "a política determinística entrou em ciclo". São problemas diferentes.
    motivos = {"fome": 0, "colisao": 0, "tabuleiro_cheio": 0}
    faltam = num_envs
    passos = 0

    while faltam > 0 and passos < max_steps:
        logits = np.asarray(policy(obs, mask), dtype=np.float32)
        logits = np.where(mask, logits, MASK_NEG)
        if safety:
            logits = apply_safety_filter(env, logits)

        if greedy:
            acoes = logits.argmax(axis=1).astype(np.int32)
        else:
            z = logits - logits.max(axis=1, keepdims=True)
            p = np.exp(z)
            p /= p.sum(axis=1, keepdims=True)
            acoes = (p.cumsum(axis=1) > rng.random((num_envs, 1))).argmax(axis=1).astype(np.int32)

        obs, mask, r, done, info = env.step(acoes)
        passos += 1

        # Políticas com estado recorrente (DreamerV3) precisam saber o que de fato
        # aconteceu: a ação escolhida — que pode não ser o argmax, se o filtro de
        # segurança agiu — e onde o episódio terminou, para zerar o estado latente ali.
        # Políticas sem memória simplesmente não expõem este método.
        if apos_passo is not None:
            apos_passo(acoes, done)

        # `info["scores"]` é o score **final** do episódio, já contando a comida do
        # último passo. Ler `env.score` antes do passo perde exatamente um ponto nos
        # episódios que terminam comendo — que são precisamente as vitórias. Ver
        # `test_eval.py::test_a_winning_episode_scores_the_last_apple`.
        truncados = set(info["trunc_idx"].tolist())
        for j, i in enumerate(np.nonzero(done)[0]):
            if len(coletados[i]) < por_env:
                s_final = int(info["scores"][j])
                coletados[i].append(s_final)
                if i in truncados:
                    motivos["fome"] += 1
                elif s_final == board_size * board_size - 3:
                    motivos["tabuleiro_cheio"] += 1
                else:
                    motivos["colisao"] += 1
                if len(coletados[i]) == por_env:
                    faltam -= 1

    scores = np.array([s for lista in coletados for s in lista][:episodes], dtype=np.int32)
    if scores.size == 0:
        raise RuntimeError("nenhum episódio terminou — aumente `max_steps`")

    perfeito = board_size * board_size - 3
    # A taxa de vitória sai da **amostra coletada**, não de um contador do laço: o laço
    # continua rodando os ambientes que já cumpriram a cota, e somar as vitórias deles
    # daria uma taxa que não corresponde aos episódios de fato medidos.
    stats = {
        "episodes": int(scores.size),
        "score_mean": float(scores.mean()),
        "score_median": float(np.median(scores)),
        "score_std": float(scores.std()),
        "score_max": int(scores.max()),
        "score_p95": float(np.percentile(scores, 95)),
        "win_rate": float((scores == perfeito).mean()),
        "perfect_possible": perfeito,
        "env_steps_used": int(passos),
        "completo": bool(faltam == 0),
    }
    total_motivos = max(1, sum(motivos.values()))
    stats.update({f"fim_{k}": v / total_motivos for k, v in motivos.items()})
    return stats, scores


def random_baseline(board_size=10, episodes=1000, num_envs=250, seed=123):
    """O piso: política uniforme sobre as ações permitidas.

    É o número contra o qual todo resultado do benchmark é lido. Num 10x10 ele vale
    ~1,08 — qualquer coisa que não esteja bem acima disso não aprendeu nada.
    """
    stats, _ = evaluate(
        random_policy(np.random.default_rng(seed)),
        board_size=board_size,
        episodes=episodes,
        num_envs=num_envs,
        greedy=False,
        seed=seed,
    )
    return stats["score_mean"]


# ---------------------------------------------------------------------- veredito
def verdict(policy, board_size=10, episodes=1000, num_envs=250, com_filtro=True, seed=123,
            canal_fome=False):
    """A resposta objetiva para "aprendeu mesmo?".

    Roda, na mesma execução, três regimes e devolve a tabela:

    ===========================  =============================================
    regime                       o que mede
    ===========================  =============================================
    aleatório com máscara        o piso — quanto se tira sem aprender nada
    agente (greedy)              a política pura, sem nenhuma ajuda externa
    agente + filtro de segurança o teto prático, com o flood-fill ligado
    ===========================  =============================================

    Se a linha do meio não estiver bem acima do piso, não aprendeu — e aí o problema é de
    hiperparâmetro ou de tempo de treino, não do código.

    `canal_fome` acompanha o ambiente de treino — ver `evaluate`. O piso não muda: a
    política aleatória não olha a observação, e o canal extra não altera a dinâmica.
    """
    linhas = []

    piso = random_baseline(board_size, episodes, num_envs, seed)
    linhas.append({"regime": "aleatório com máscara", "score_mean": piso})

    st, sc = evaluate(policy, board_size=board_size, episodes=episodes,
                      num_envs=num_envs, greedy=True, seed=seed,
                      canal_fome=canal_fome)
    linhas.append({"regime": "agente (greedy)", "scores": sc, **st})

    if com_filtro:
        # o flood-fill é laço Python: menos ambientes, para não ficar lento
        stf, scf = evaluate(policy, board_size=board_size, episodes=episodes,
                            num_envs=min(num_envs, 64), greedy=True, safety=True,
                            seed=seed, canal_fome=canal_fome)
        linhas.append({"regime": "agente + filtro de segurança", "scores": scf, **stf})

    return {
        "piso": piso,
        "perfeito": board_size * board_size - 3,
        "ganho_sobre_o_piso": linhas[1]["score_mean"] / max(piso, 1e-9),
        "linhas": linhas,
    }


def format_verdict(resultado):
    """Formata o retorno de `verdict` como tabela de texto."""
    larg = 30
    out = [f"{'regime':<{larg}}{'média':>8}{'mediana':>9}{'máx':>6}{'cheio':>8}", "-" * (larg + 31)]
    for ln in resultado["linhas"]:
        med = f"{ln['score_median']:.0f}" if "score_median" in ln else "-"
        mx = f"{ln['score_max']}" if "score_max" in ln else "-"
        wr = f"{ln['win_rate']:.1%}" if "win_rate" in ln else "-"
        out.append(f"{ln['regime']:<{larg}}{ln['score_mean']:>8.2f}{med:>9}{mx:>6}{wr:>8}")
    out.append("-" * (larg + 31))
    out.append(
        f"score perfeito: {resultado['perfeito']}   |   "
        f"ganho sobre o piso: {resultado['ganho_sobre_o_piso']:.1f}x"
    )
    ag = resultado["linhas"][1]
    if "fim_fome" in ag:
        out.append(
            f"como terminou: fome {ag['fim_fome']:.0%} · colisão {ag['fim_colisao']:.0%}"
            f" · tabuleiro cheio {ag['fim_tabuleiro_cheio']:.0%}"
        )
        # Morrer de fome é o fim NORMAL aqui: a máscara de morte impede a colisão, então
        # até a política aleatória termina 85% dos episódios por fome. O que denuncia o
        # ciclo é a combinação — quase nenhuma colisão **e** score abaixo do piso, ou seja,
        # a cobra anda para sempre sem nunca comer.
        if ag["fim_colisao"] < 0.05 and ag["score_mean"] < resultado["piso"]:
            out.append(
                "  ⚠ nunca colide e não come: a política determinística entrou em ciclo.\n"
                "    Não é 'jogou mal' — é falta de exploração na hora de agir. Normal cedo\n"
                "    num DQN greedy, e é por isso que o score de TREINO (ε-greedy) fica\n"
                "    acima do de AVALIAÇÃO (greedy) nesta fase."
            )
    return "\n".join(out)


# --- snakeai/record.py ---
"""Registro de execuções — o esquema do `history.json` e o validador do contrato.

Este módulo é o porteiro do benchmark. Toda execução de todo algoritmo escreve o mesmo
arquivo, com os mesmos campos, e passa pela mesma validação antes de virar uma linha no
gráfico. **Um resultado que não valida não entra na arena** — não porque seja ruim, mas
porque não é comparável, que é pior.

A regra vale inclusive para as curvas históricas do `colab-rl`: elas são convertidas para
este mesmo esquema, mas com `comparable=False` e o motivo registrado em `caveat`. Assim
elas aparecem no gráfico como contexto (tracejado cinza) sem nunca serem confundidas com
um competidor.

Sem dependências além da biblioteca padrão e do NumPy — o validador roda no CI em segundos.
"""


import json
import os
import platform
import subprocess
import sys
import time
from dataclasses import asdict, dataclass, field

import numpy as np

__all__ = [
    "SCHEMA_VERSION",
    "CONTRATO",
    "ORCAMENTO_OFICIAL",
    "SEMENTES_OFICIAIS",
    "ContractViolation",
    "RunRecord",
    "Recorder",
    "validate",
    "save",
    "load",
    "load_all",
    "configuracoes_incompletas",
    "from_legacy_csv",
]

SCHEMA_VERSION = 1

#: Os valores que **todos** os resultados oficiais precisam compartilhar.
#: Espelha a tabela do README; mudar aqui é mudar o contrato, e invalida o histórico.
CONTRATO = {
    "env": "VecSnake",
    "board_size": 10,
    "starve_base": 100,
    "n_channels": 5,
    "n_actions": 3,
    "obs": "egocentric",
    "metric": "score",
    "reward_food": 1.0,
    "reward_death": -1.0,
    "eval_episodes": 1000,
    "eval_seed": 123,
    "eval_greedy": True,
    "eval_safety": False,
}

#: Orçamento oficial, em passos de ambiente. Fica fora do `CONTRATO` porque não descreve o
#: ambiente, mas é igualmente obrigatório: comparar um algoritmo que treinou 5 M passos com
#: outro que treinou 500 mil não mede algoritmo, mede paciência. Validado a partir de
#: `config["total_steps"]`.
ORCAMENTO_OFICIAL = 5_000_000

#: Sementes por configuração. Era convenção escrita no `COMPARABILITY.md` e nada mais —
#: e foi assim que a arena publicou um ACKTR de **uma** semente ao lado de um PPO de três,
#: com a amplitude entre sementes do PPO valendo 19 pontos. Convenção que ninguém confere
#: não é contrato. Ver `configuracoes_incompletas`.
SEMENTES_OFICIAIS = 3

#: Piso e teto do 10x10, medidos e documentados no README.
PISO_ALEATORIO = 1.21
SCORE_PERFEITO = 97


class ContractViolation(Exception):
    """Levantada quando um registro não obedece ao contrato de comparabilidade."""


# ------------------------------------------------------------------- estrutura
@dataclass
class RunRecord:
    """Uma execução completa de um algoritmo, com curva e resultado final.

    Campos
    ------
    algo, variant, seed
        Identidade da execução. `runs/<algo>/<variant>/seed<N>/history.json`.
    net, params
        Tronco usado e número de parâmetros treináveis — o eixo "arquitetura importa?".
    config
        Hiperparâmetros do agente, como dicionário livre. Não é validado; é documentação.
    env_spec
        O recorte do contrato que esta execução usou. **É** validado.
    curve
        Lista de pontos ao longo do treino. Cada ponto tem, no mínimo, `global_step`;
        `eval_score_mean` aparece só nos passos em que a avaliação rodou.
    final
        O `stats` de `snakeai.eval.evaluate` para o modelo do **último** passo. É este que
        entra na curva e na arena.
    melhor
        O mesmo `stats`, para o **melhor checkpoint** já visto, mais o `global_step` em que
        ele apareceu. RL profundo não melhora monotonicamente — não há garantia nenhuma
        fora do caso tabular — e uma execução pode terminar pior do que já esteve. Guardar
        os dois separa duas perguntas diferentes: *como o algoritmo terminou* (final) e
        *o melhor que ele produziu* (melhor). A primeira é a da arena; a segunda é a de
        quem vai levar o modelo para o jogo. Ver `docs/COMPARABILITY.md`.
    comparable, caveat
        `False` marca uma curva que entra no gráfico como contexto histórico, com o
        motivo em `caveat`. Toda execução nova nasce `True`.
    """

    algo: str
    variant: str = "default"
    seed: int = 0
    net: str = ""
    params: int = 0
    config: dict = field(default_factory=dict)
    env_spec: dict = field(default_factory=lambda: dict(CONTRATO))
    curve: list = field(default_factory=list)
    final: dict = field(default_factory=dict)
    melhor: dict = field(default_factory=dict)
    comparable: bool = True
    caveat: str = ""
    meta: dict = field(default_factory=dict)
    schema_version: int = SCHEMA_VERSION

    # ---------------------------------------------------------------- derivados
    @property
    def run_id(self):
        return f"{self.algo}/{self.variant}/seed{self.seed}"

    @property
    def rel_path(self):
        return os.path.join("runs", self.algo, self.variant, f"seed{self.seed}",
                            "history.json")

    def steps(self):
        return np.array([p["global_step"] for p in self.curve], dtype=np.int64)

    @property
    def oficial(self):
        """Pode competir na arena? Comparável **e** sem violação de contrato registrada.

        Separado de `comparable` de propósito: uma execução de fumaça não é uma curva
        histórica. Ela não compete, mas também não vira contexto — simplesmente não
        aparece, e o motivo fica em `meta["contract_violations"]`.
        """
        return self.comparable and not self.meta.get("contract_violations")

    def eval_curve(self):
        """`(passos, scores)` só dos pontos em que a avaliação rodou."""
        pts = self._pontos_de_eval()
        x = np.array([p["global_step"] for p in pts], dtype=np.int64)
        y = np.array([p["eval_score_mean"] for p in pts], dtype=np.float64)
        return x, y

    def _pontos_de_eval(self):
        return [p for p in self.curve if p.get("eval_score_mean") is not None]

    def eval_curve_tempo(self):
        """`(horas, scores)` — a mesma curva no eixo de **custo**, não de dados.

        O eixo oficial da arena são passos de ambiente, que igualam os *dados vistos* e
        escondem o *esforço gasto*: o AlphaZero faz busca em árvore a cada passo e custa
        ordens de grandeza mais que o DQN para chegar ao mesmo x. Este eixo mostra a outra
        metade.

        O `wall_s` inclui as avaliações periódicas, e isso é proposital: elas são custo
        real de quem roda. Mas veja `mesmo_hardware` — comparar tempo entre execuções
        feitas em GPUs diferentes não significa nada.
        """
        pts = self._pontos_de_eval()
        h = np.array([p.get("wall_s", np.nan) for p in pts], dtype=np.float64) / 3600.0
        y = np.array([p["eval_score_mean"] for p in pts], dtype=np.float64)
        return h, y

    def passos_ate(self, limiar):
        """Primeiro passo **medido** em que a avaliação atingiu `limiar`. `None` se nunca.

        Sem interpolação, de propósito: a resolução é a cadência de avaliação
        (`eval_every_steps`), e interpolar inventaria uma precisão que a amostragem não
        tem. O número devolvido é um passo em que a medição de fato aconteceu.
        """
        x, y = self.eval_curve()
        atingiu = np.nonzero(y >= limiar)[0]
        return int(x[atingiu[0]]) if atingiu.size else None

    @property
    def hardware(self):
        """Identidade do que rodou isto, para o eixo de tempo saber quando calar a boca."""
        gpus = self.meta.get("gpus") or []
        return f"{self.meta.get('plataforma', '?')}/{','.join(gpus) or 'cpu'}"


# -------------------------------------------------------------------- gravação
class Recorder:
    """Acumula a curva durante o treino e grava o `history.json` no fim.

    Uso típico, dentro do laço de treino::

        rec = Recorder("ppo", variant="resnet_small", seed=0, net="resnet_small",
                       params=model.count_params(), config=asdict(cfg))
        ...
        rec.log(global_step=n, episodes=e, train_score_mean=m)
        rec.log(global_step=n, eval_score_mean=stats["score_mean"])   # nos passos de eval
        ...
        rec.finish(stats)
        rec.save()            # valida antes de escrever; levanta se violar o contrato
    """

    def __init__(self, algo, variant="default", seed=0, net="", params=0,
                 config=None, env_spec=None, root="runs"):
        self.root = root
        self.t0 = time.perf_counter()
        self.record = RunRecord(
            algo=algo, variant=variant, seed=seed, net=net, params=int(params),
            config=dict(config or {}),
            env_spec=dict(env_spec or CONTRATO),
            meta=_ambiente(),
        )

    def log(self, global_step, **metrics):
        """Anexa um ponto à curva. `global_step` é o eixo oficial."""
        ponto = {"global_step": int(global_step),
                 "wall_s": round(time.perf_counter() - self.t0, 3)}
        for k, v in metrics.items():
            ponto[k] = _jsonable(v)
        self.record.curve.append(ponto)
        return ponto

    def finish(self, final_stats, comparable=True, caveat="", melhor_stats=None):
        self.record.final = {k: _jsonable(v) for k, v in dict(final_stats).items()}
        if melhor_stats is not None:
            self.record.melhor = {k: _jsonable(v) for k, v in dict(melhor_stats).items()}
        self.record.comparable = bool(comparable)
        self.record.caveat = str(caveat)
        self.record.meta["wall_s_total"] = round(time.perf_counter() - self.t0, 3)
        return self.record

    def save(self, path=None, skip_validation=False):
        if not skip_validation:
            problemas = validate(self.record)
            if problemas:
                raise ContractViolation(
                    f"{self.record.run_id} viola o contrato:\n  - "
                    + "\n  - ".join(problemas)
                )
        destino = path or os.path.join(self.root, self.record.algo,
                                       self.record.variant,
                                       f"seed{self.record.seed}", "history.json")
        return save(self.record, destino)


def save(record: RunRecord, path):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(asdict(record), f, ensure_ascii=False, indent=2)
    return path


def load(path) -> RunRecord:
    with open(path, encoding="utf-8") as f:
        d = json.load(f)
    d.pop("schema_version", None)
    return RunRecord(**d, schema_version=SCHEMA_VERSION)


def configuracoes_incompletas(registros, minimo=SEMENTES_OFICIAIS):
    """As configurações `(algo, variant)` com menos de `minimo` sementes distintas.

    É uma propriedade do **conjunto**, não de uma execução — por isso não cabe em
    `validate`, que olha uma por vez. A arena chama isto para não publicar uma linha de
    uma semente com a mesma tipografia de uma de três.
    """
    grupos = {}
    for r in registros:
        grupos.setdefault((r.algo, r.variant), set()).add(r.seed)
    return [{"algo": a, "variant": v, "sementes": len(s), "faltam": minimo - len(s)}
            for (a, v), s in sorted(grupos.items()) if len(s) < minimo]


def load_all(root="runs"):
    """Carrega todo `history.json` sob `root`, ordenado por algoritmo/variante/seed."""
    achados = []
    for base, _, arquivos in os.walk(root):
        for nome in arquivos:
            if nome == "history.json":
                achados.append(load(os.path.join(base, nome)))
    achados.sort(key=lambda r: (r.algo, r.variant, r.seed))
    return achados


# ------------------------------------------------------------------- validação
def validate(record: RunRecord, strict_eval=True):
    """Devolve a lista de violações do contrato. Lista vazia = pode entrar na arena.

    Curvas marcadas `comparable=False` só precisam ter identidade, curva e um `caveat`
    explicando por que não competem — o resto do contrato não se aplica a elas.
    """
    p = []

    if not record.algo:
        p.append("`algo` vazio")
    if record.schema_version != SCHEMA_VERSION:
        p.append(f"schema_version {record.schema_version} != {SCHEMA_VERSION}")
    if not record.curve:
        p.append("curva vazia")
    else:
        steps = [pt.get("global_step") for pt in record.curve]
        if any(s is None for s in steps):
            p.append("ponto da curva sem `global_step`")
        elif list(steps) != sorted(steps):
            p.append("`global_step` não é monotônico")

    if not record.comparable:
        if not record.caveat:
            p.append("`comparable=False` exige um `caveat` explicando por quê")
        return p

    # --- daqui para baixo, só para execuções que querem competir
    for chave, esperado in CONTRATO.items():
        obtido = record.env_spec.get(chave, "<ausente>")
        if obtido != esperado:
            p.append(f"env_spec['{chave}'] = {obtido!r}, contrato exige {esperado!r}")

    if not record.final:
        p.append("`final` vazio — falta o resultado do protocolo de avaliação")
    elif strict_eval:
        f = record.final
        if f.get("episodes") != CONTRATO["eval_episodes"]:
            p.append(f"avaliação final com {f.get('episodes')} episódios, "
                     f"contrato exige {CONTRATO['eval_episodes']}")
        if not f.get("completo", True):
            p.append("avaliação final incompleta (bateu `max_steps`)")
        # As chaves de causa de fim entraram junto com a correção do protocolo — o mesmo
        # commit que passou a contar a maçã final do episódio vencedor. Um registro sem
        # elas foi medido com a régua **anterior**, e `episodes`/`completo` continuam
        # iguais nos dois casos: é a única marca que distingue. Ver
        # `docs/ANTES_DO_ARTIGO.md`.
        faltando = [k for k in ("fim_fome", "fim_colisao", "fim_tabuleiro_cheio")
                    if k not in f]
        if faltando:
            p.append(f"`final` sem {', '.join(faltando)} — medido com um protocolo "
                     "anterior ao atual; remeça com o `evaluate` desta versão")
        media = f.get("score_mean")
        if media is None:
            p.append("`final.score_mean` ausente")
        elif not (0.0 <= media <= SCORE_PERFEITO):
            p.append(f"score_mean fora da faixa possível: {media}")

    orcamento = record.config.get("total_steps")
    if orcamento is None:
        p.append("`config['total_steps']` ausente — o orçamento é parte do contrato")
    elif int(orcamento) != ORCAMENTO_OFICIAL:
        p.append(f"orçamento de {int(orcamento):,} passos; o contrato exige "
                 f"{ORCAMENTO_OFICIAL:,}. Comparar treinos de tamanhos diferentes mede "
                 "paciência, não algoritmo")

    # O `config` diz o orçamento **pretendido**; a curva diz o que foi **gasto**. Conferir
    # só o primeiro deixava passar uma execução interrompida na metade com o `config`
    # intacto — `train(ate_passos=...)` faz exatamente isso, e uma sessão do Colab caindo
    # também. Ver `docs/REVISAO_ALGORITMOS.md` §1.3.
    gasto = max((int(ponto.get("global_step", 0)) for ponto in record.curve), default=0)
    if gasto < ORCAMENTO_OFICIAL:
        p.append(f"a curva vai até {gasto:,} passos, abaixo dos {ORCAMENTO_OFICIAL:,} do "
                 "contrato — o orçamento declarado em `config` não é o que foi gasto")

    if record.params <= 0:
        p.append("`params` deve ser o número de parâmetros treináveis")
    if not record.net:
        p.append("`net` vazio — a arquitetura é um eixo de comparação")

    return p


def assert_valid(record: RunRecord, **kw):
    problemas = validate(record, **kw)
    if problemas:
        raise ContractViolation(
            f"{record.run_id} viola o contrato:\n  - " + "\n  - ".join(problemas)
        )
    return record


# ---------------------------------------------------------------------- legado
def from_legacy_csv(path, algo="dqn-legacy", variant=None, caveat=None):
    """Converte um CSV de treino do `colab-rl` para o esquema do repositório.

    Os CSVs antigos têm colunas sem nome: `índice, comprimento, passos, loss, reward`.
    O comprimento vira score pela regra `score = comprimento - 3`, e o registro nasce
    `comparable=False` — foi medido em outro ambiente, com outra recompensa e outra
    unidade de tempo. Ele é contexto histórico, não competidor.
    """
    import csv

    linhas = []
    with open(path, newline="", encoding="utf-8") as f:
        leitor = csv.reader(f)
        cabecalho = next(leitor, None)
        for row in leitor:
            if len(row) < 5:
                continue
            try:
                ep = int(float(row[0]))
                comprimento = float(row[1])
                passos = float(row[2])
                perda = float(row[3])
                recompensa = float(row[4])
            except ValueError:
                continue
            linhas.append((ep, comprimento, passos, perda, recompensa))

    if not linhas:
        raise ValueError(f"nenhuma linha aproveitável em {path} (cabeçalho: {cabecalho})")

    # Nos CSVs originais o nome do arquivo é sempre `keras_training_data.csv` e quem
    # identifica a variante é a pasta; nos normalizados de `results/legacy/` é o
    # contrário. Aceita os dois.
    if variant is None:
        raiz = os.path.splitext(os.path.basename(path))[0]
        variant = (os.path.basename(os.path.dirname(path))
                   if raiz in ("keras_training_data", "training_data") else raiz)
    curva = [
        {
            "global_step": ep,               # aqui o eixo é episódio, não passo — ver caveat
            "episodes": ep,
            "train_score_mean": comprimento - 3.0,
            "train_length_mean": comprimento,
            "episode_steps": passos,
            "loss": perda,
            "reward": recompensa,
        }
        for ep, comprimento, passos, perda, recompensa in linhas
    ]

    scores = np.array([c["train_score_mean"] for c in curva], dtype=np.float64)
    rec = RunRecord(
        algo=algo,
        variant=variant,
        seed=0,
        net="cnn-legado",
        params=0,
        env_spec={"env": "snake-on-pygame (legado)"},
        curve=curva,
        final={
            "episodes": len(curva),
            "score_mean": float(scores[-100:].mean()),
            "score_max": float(scores.max()),
        },
        comparable=False,
        caveat=(
            caveat
            or "Medido no ambiente antigo (snake-on-pygame): recompensa +comprimento ao "
               "comer, estado ordinal de 1 canal com a cabeça sobrescrita, 5 ações "
               "absolutas, eixo em episódios. Convertido por score = comprimento - 3 "
               "apenas para posicionar a curva; não é comparável com as execuções novas."
        ),
        meta={"fonte": os.path.basename(path)},
    )
    return rec


# ------------------------------------------------------------------ utilidades
def _jsonable(v):
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    if isinstance(v, np.ndarray):
        return v.tolist()
    if isinstance(v, (np.bool_,)):
        return bool(v)
    return v


def _ambiente():
    """Carimbo de proveniência: sem isto, um número no gráfico não é rastreável."""
    meta = {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "created_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
    }
    try:
        meta["commit"] = subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL,
            text=True, timeout=5,
        ).strip()
    except Exception:
        meta["commit"] = "desconhecido"

    # No Colab e no Kaggle não existe clone git: o `commit` sai "desconhecido" e a curva
    # fica sem procedência — que é justamente onde quase todas as execuções deste
    # repositório nascem. O gerador de notebooks injeta `ASSINATURA_PACOTE` no bloco de
    # código gerado, e ela identifica o pacote inteiro (é o hash do fonte embutido). Como
    # o notebook é um namespace só, esta busca a encontra lá e não a encontra aqui.
    assinatura = globals().get("ASSINATURA_PACOTE")
    if assinatura:
        meta["assinatura_pacote"] = str(assinatura)
    for mod in ("tensorflow", "keras"):
        try:
            meta[mod] = __import__(mod).__version__
        except Exception:
            pass
    return meta


# --- snakeai/env/render.py ---
"""Ver a cobra jogar — GIF de um episódio, sem pygame.

O Colab não tem display, então renderizar pelo jogo original não é opção. Aqui o episódio
vira uma sequência de imagens direto da grade `occ` do `VecSnake`, e o GIF é o artefato
que se olha para entender *como* o agente joga — coisa que nenhuma curva conta.

Vale mais do que parece: um agente com score médio 20 que morre sempre se prendendo no
próprio corpo e outro que morre por fome têm a mesma linha no gráfico e problemas
completamente diferentes.
"""


import numpy as np


__all__ = ["quadros_do_episodio", "render_episode", "PALETA_JOGO"]

#: Fundo, corpo, comida, cabeça — nas cores do gráfico da arena, para o GIF e as figuras
#: parecerem do mesmo projeto.
PALETA_JOGO = np.array(
    [
        [26, 26, 25],      # fundo (a superfície escura do gráfico)
        [27, 175, 122],    # corpo (aqua da paleta)
        [235, 104, 52],    # comida (laranja da paleta)
        [252, 252, 251],   # cabeça (tinta clara)
    ],
    dtype=np.uint8,
)


def _quadro(env: VecSnake, i=0, escala=16):
    grade = np.zeros((env.b, env.b), dtype=np.int32)
    grade[env.occ[i] > 0] = 1
    grade[env.food[i, 0], env.food[i, 1]] = 2
    grade[env.head[i, 0], env.head[i, 1]] = 3
    return PALETA_JOGO[grade].repeat(escala, 0).repeat(escala, 1)


def quadros_do_episodio(politica, board_size=10, safety=False, max_steps=2000,
                        seed=7, escala=16, canal_fome=False):
    """Roda um episódio com `politica` e devolve `(quadros, score, motivo)`.

    `politica` é a mesma interface de `snakeai.eval`: `politica(obs, mask) -> logits`.
    Assim o GIF mostra exatamente a política que o benchmark mediu, sem caminho paralelo.
    """

    # `canal_fome` tem que acompanhar o ambiente de treino: uma rede de 6 canais
    # recebendo observação de 5 quebra aqui, e o GIF é gerado no fim do treino — tarde
    # demais para descobrir. Ver `snakeai.eval.evaluate`.
    env = VecSnake(1, board_size, rng=np.random.default_rng(seed),
                   canal_fome=canal_fome)
    obs, mask = env.reset()
    quadros = [_quadro(env, escala=escala)]
    score, motivo = 0, "limite de passos"
    # Políticas com memória — o `PoliticaRecorrente` do DreamerV3, o `PoliticaComOpcoes`
    # do SOAP — precisam saber qual ação de fato saiu para avançar o estado interno.
    # `snakeai.eval` já respeitava este contrato; aqui não, e o resultado era um GIF
    # gravado com o latente congelado no valor inicial: o agente do vídeo não era o
    # agente da curva, e o vídeo é justamente o artefato que se olha para entender *como*
    # ele joga. Políticas sem memória não expõem o método e nada muda para elas.
    apos_passo = getattr(politica, "apos_passo", None)

    for _ in range(max_steps):
        logits = np.asarray(politica(obs, mask), dtype=np.float32)
        logits = np.where(mask, logits, MASK_NEG)
        if safety:
            logits = apply_safety_filter(env, logits)
        a = logits.argmax(axis=1).astype(np.int32)

        score_antes = int(env.score[0])
        comprimento_antes = int(env.length[0])
        fome_antes = int(env.hunger[0])
        obs, mask, r, d, info = env.step(a)
        if apos_passo is not None:
            apos_passo(a, d)
        quadros.append(_quadro(env, escala=escala))

        if d[0]:
            # `info["scores"]` é o score **final**, já com a maçã do último passo. Ler
            # `env.score` antes do passo perde exatamente um ponto nos episódios que
            # terminam comendo — que são precisamente as vitórias, e o GIF de uma vitória
            # saía rotulado com 96 num tabuleiro cujo perfeito é 97. Mesmo defeito que o
            # `eval.py` corrige e trava com teste.
            finais = info.get("scores")
            score = int(finais[0]) if finais is not None and len(finais) else score_antes
            if comprimento_antes >= board_size * board_size - 1:
                motivo = "tabuleiro cheio"
            elif fome_antes + 1 >= env.starve_base + 2 * comprimento_antes:
                motivo = "fome"
            else:
                motivo = "colisão"
            break
    else:
        score = int(env.score[0])

    return quadros, score, motivo


def render_episode(politica, caminho="episodio.gif", fps=15, **kw):
    """Grava o GIF e devolve `(caminho, score, motivo)`."""
    import imageio.v2 as imageio

    quadros, score, motivo = quadros_do_episodio(politica, **kw)
    imageio.mimsave(caminho, quadros, fps=fps, loop=0)
    return caminho, score, motivo


# --- snakeai/export.py ---
"""Exportar o modelo — `.keras` para retomar treino, TFLite para embarcar.

Uma armadilha silenciosa do Keras 3, registrada aqui para ninguém repetir
--------------------------------------------------------------------------
Converter para TFLite **precisa passar por um SavedModel**.
`TFLiteConverter.from_concrete_functions(...)` compila sem erro, gera um arquivo
minúsculo — e **não captura os pesos**. A inferência devolve NaN, sem nenhum aviso. O
sintoma é um `.tflite` de poucos KB quando deveria ter centenas.

Por isso `export_model` sempre passa por `model.export(dir, format="tf_saved_model")`, e
sempre valida a paridade contra o modelo original antes de declarar sucesso.

A segunda armadilha: "a saída da rede" não tem uma forma só
-----------------------------------------------------------
A paridade compara a **ação escolhida**, e por muito tempo esse cálculo assumiu que a
saída da política é `(lote, ações)`. Neste repositório ela é isso em três dos formatos e
outra coisa nos demais:

======================================  ==============================  ================
construtor                              saída de política                quem usa
======================================  ==============================  ================
``build_actor_critic``                  ``(lote, ações)``                PPO, A2C, ACKTR…
``build_q_network`` (sem C51)           ``(lote, ações)``                DQN
``build_q_network`` (``n_atoms > 0``)   ``(lote, ações, átomos)``        Rainbow, C51
``build_actor_critic_populacao``        ``(lote, políticas, ações)``     LBC
``build_policy_q``                      ``(lote, ações)`` **duas vezes** ACER
======================================  ==============================  ================

O eixo das ações muda de lugar, e no ACER duas saídas diferentes têm exatamente a mesma
forma. Por isso a redução para "um escore por ação" mora em `_escores_por_acao`, e a
escolha de qual tensor do `.tflite` corresponde à política mora em `_indice_da_politica`
— as duas aplicadas **do mesmo jeito nos dois lados** da comparação. Reduzir só o lado
Keras é o defeito que quebrava o Rainbow no fim de um treino inteiro:

    ValueError: operands could not be broadcast together with shapes (200,) (200,121)

É o mesmo erro de `DQN.politica_do_modelo` (§2.17) um passo adiante — lá o C51 quebrava a
avaliação do checkpoint, aqui quebrava a exportação. Ver `docs/REVISAO_ALGORITMOS.md`.
"""


import os
import shutil
import time

import numpy as np


__all__ = ["export_model", "medir_latencia", "conferir_paridade", "canais_do_modelo"]


def canais_do_modelo(modelo, padrao=N_CHANNELS):
    """Quantos canais a rede espera na entrada — **perguntando à rede**, não à constante.

    O contrato são 5 canais, mas uma execução com `canal_fome=True` treina uma rede de 6.
    Exportar essa rede alimentando-a com a constante quebra na primeira inferência, com
    uma mensagem sobre formas — e isso acontece **depois** do treino inteiro, na última
    célula do notebook. Ver `snakeai.eval.evaluate`, que tem o mesmo cuidado.
    """
    try:
        forma = modelo.input_shape
        if isinstance(forma, (list, tuple)) and forma and isinstance(forma[0], (list, tuple)):
            forma = forma[0]                      # modelos de múltiplas entradas
        canais = forma[-1]
        return int(canais) if canais else padrao
    except Exception:                             # rede sem `input_shape` conhecido
        return padrao


def medir_latencia(fn, board_size=10, repeticoes=200, aquecimento=20, canais=N_CHANNELS):
    """Latência de inferência com lote 1 — o que importa se o modelo for para o jogo."""
    x = np.zeros((1, board_size, board_size, canais), dtype=np.float32)
    for _ in range(aquecimento):
        fn(x)
    t0 = time.perf_counter()
    for _ in range(repeticoes):
        fn(x)
    return (time.perf_counter() - t0) / repeticoes * 1000.0


def _q_de_logits_c51(logits):
    """`(…, ações, átomos)` de logits → `(…, ações)`, para **escolher a ação**.

    O `Q` do C51 é `Σ_z p(z)·z` com `z` no suporte `linspace(v_min, v_max, n_atoms)`. O
    exportador não conhece `v_min`/`v_max` — e não precisa: como o suporte é afim e
    crescente no índice do átomo (`z_i = v_min + i·Δz`, com `Δz > 0`), vale

        argmax_a Σ_i p(a,i)·z_i  =  argmax_a Σ_i p(a,i)·i

    ou seja, a **ação escolhida** não depende do suporte, só da esperança do índice. É por
    isso que esta função devolve o índice esperado em vez do `Q` de verdade: o número não
    é o `Q`, mas o `argmax` é o mesmo, e é só o `argmax` que esta comparação usa.

    A média simples dos logits, que estava aqui antes, **não** tem essa propriedade — ela
    ignora a softmax e pode trocar a ação escolhida.
    """
    z = logits - logits.max(axis=-1, keepdims=True)
    p = np.exp(z)
    p /= p.sum(axis=-1, keepdims=True)
    indices = np.arange(p.shape[-1], dtype=p.dtype)
    return (p * indices).sum(axis=-1)


def _escores_por_acao(t, n_actions=N_ACTIONS):
    """Reduz a saída da política a **um escore por ação**, com as ações no último eixo.

    Cobre os três formatos que os construtores de `snakeai.nets.registry` produzem:

    * `(lote, ações)` — devolvido como está;
    * `(lote, políticas, ações)` (LBC) — devolvido como está, e o `argmax` do chamador
      passa a comparar a escolha de **cada** cabeça da população. Conferir todas é mais
      forte que conferir só a `indice_alvo`, que o exportador não conhece;
    * `(lote, ações, átomos)` (C51) — colapsado por `_q_de_logits_c51`.

    A desambiguação é por posição do eixo com `N_ACTIONS`, com o **último** ganhando o
    desempate: `(lote, 3, 3)` é a população de três políticas do LBC, não um C51 de três
    átomos — que seria uma configuração sem sentido (o C51 existe para ter resolução).
    """
    t = np.asarray(t, dtype=np.float32)
    if t.ndim == 2:
        return t
    if t.ndim >= 3 and t.shape[-1] == n_actions:
        return t
    if t.ndim == 3 and t.shape[1] == n_actions:
        return _q_de_logits_c51(t)
    raise ValueError(
        f"saída de forma {t.shape} sem um eixo de {n_actions} ações reconhecível — "
        "se for um formato novo de política, ensine-o a `_escores_por_acao`"
    )


def _indice_da_politica(candidatos, referencia):
    """Qual das saídas do `.tflite` é a política — casando a **forma** com a do Keras.

    A regra antiga era "a que tem `N_ACTIONS` colunas", e ela erra dos dois jeitos:

    * **não acha** a saída certa quando a política é `(lote, ações, átomos)` (C51: a
      última dimensão são os átomos), e caía no `cand[0]` sem avisar;
    * **acha duas** no ACER, cujas saídas `logits` e `Q(s,·)` têm a mesma forma, e no LBC,
      onde `(lote, 3, 3)` e `(lote, 3)` casam as duas. A ordem das saídas do
      `Interpreter` não é a ordem das saídas do `keras.Model` — o SavedModel as nomeia
      `output_0`, `output_1`… e o conversor pode reordená-las. Pegar a primeira é sortear.

    Aqui a forma decide, e quando ela empata o **valor** desempata: a saída correta é a
    que se parece com a do Keras. Isso é exatamente o que a paridade afirma, então usar o
    critério para escolher não enfraquece nada — se nenhuma das candidatas se parecer com
    a referência, todas reprovam igual.
    """
    forma = tuple(referencia.shape[1:])
    iguais = [i for i, c in enumerate(candidatos) if tuple(c.shape[1:]) == forma]
    if not iguais:
        formas = ", ".join(str(tuple(c.shape[1:])) for c in candidatos)
        raise ValueError(
            f"nenhuma saída do .tflite tem a forma da política do Keras {forma} — "
            f"saídas disponíveis: {formas}"
        )
    if len(iguais) == 1:
        return iguais[0]
    return min(iguais, key=lambda i: float(np.abs(candidatos[i] - referencia).max()))


def conferir_paridade(modelo, blob_tflite, board_size=10, n=200, seed=0, canais=None):
    """O `.tflite` escolhe a mesma ação que o `.keras`, em `n` estados aleatórios?

    Não basta comparar os logits: o que importa para o jogo é a **ação escolhida**. Uma
    diferença numérica de quantização é aceitável; uma ação diferente não é.

    Os dois lados passam pela **mesma** redução (`_escores_por_acao`) — reduzir só um deles
    é comparar coisas de formas diferentes, que é como isto quebrava no Rainbow.
    """
    import tensorflow as tf

    rng = np.random.default_rng(seed)
    canais = canais or canais_do_modelo(modelo)
    x = rng.normal(size=(n, board_size, board_size, canais)).astype(np.float32)

    saida = modelo(x, training=False)
    logits_keras = np.asarray(saida[0] if isinstance(saida, (list, tuple)) else saida,
                              dtype=np.float32)

    itp = tf.lite.Interpreter(model_content=blob_tflite)
    itp.allocate_tensors()
    entrada = itp.get_input_details()[0]
    saidas = itp.get_output_details()

    indice, logits_lite = None, []
    for i in range(n):
        itp.set_tensor(entrada["index"], x[i: i + 1])
        itp.invoke()
        cand = [itp.get_tensor(o["index"]) for o in saidas]
        if indice is None:
            indice = _indice_da_politica(cand, logits_keras[i: i + 1])
        logits_lite.append(cand[indice][0])
    logits_lite = np.asarray(logits_lite, dtype=np.float32)

    escolha_keras = _escores_por_acao(logits_keras).argmax(-1)
    escolha_lite = _escores_por_acao(logits_lite).argmax(-1)
    return {
        "acoes_iguais": float((escolha_keras == escolha_lite).mean()),
        "erro_max_logits": float(np.abs(logits_keras - logits_lite).max()),
    }


def export_model(modelo, out_dir="export", board_size=10, formatos=("fp16", "int8"),
                 validar=True):
    """Exporta e **mede**: tamanho, latência e paridade de ação.

    Devolve um dicionário pronto para virar linha do `MODELS.md`.

    Uma falha da **conferência** não derruba a exportação: ela vira
    `{"erro": ...}` no relatório, no lugar de `acoes_iguais`. Os arquivos já estão em
    disco quando ela roda, e esta função é a penúltima célula de um notebook que gastou
    horas de GPU — deixar a validação levar o treino junto foi exatamente o que aconteceu
    com o Rainbow. O relatório é impresso, então a falha continua visível; o que ela não
    faz mais é apagar o resto.
    """
    import tensorflow as tf

    os.makedirs(out_dir, exist_ok=True)
    canais = canais_do_modelo(modelo)
    caminho_keras = os.path.join(out_dir, "modelo.keras")
    modelo.save(caminho_keras)

    resultado = {
        "params": int(modelo.count_params()),
        "keras_kb": round(os.path.getsize(caminho_keras) / 1024, 1),
        "canais": canais,
        "tf_ms": round(medir_latencia(lambda x: modelo(x, training=False), board_size,
                                      canais=canais), 4),
    }

    sm_dir = os.path.join(out_dir, "saved_model")
    if os.path.isdir(sm_dir):
        shutil.rmtree(sm_dir)
    modelo.export(sm_dir, format="tf_saved_model")

    for nome in formatos:
        conv = tf.lite.TFLiteConverter.from_saved_model(sm_dir)
        if nome != "fp32":
            conv.optimizations = [tf.lite.Optimize.DEFAULT]
        if nome == "fp16":
            conv.target_spec.supported_types = [tf.float16]
        blob = conv.convert()

        caminho = os.path.join(out_dir, f"modelo_{nome}.tflite")
        with open(caminho, "wb") as f:
            f.write(blob)
        resultado[f"{nome}_kb"] = round(len(blob) / 1024, 1)

        itp = tf.lite.Interpreter(model_content=blob)
        itp.allocate_tensors()
        ent = itp.get_input_details()[0]
        xi = np.zeros(ent["shape"], dtype=np.float32)

        def roda(_x, _itp=itp, _ent=ent):
            _itp.set_tensor(_ent["index"], xi)
            _itp.invoke()

        resultado[f"{nome}_ms"] = round(medir_latencia(roda, board_size,
                                                       canais=canais), 4)

        if validar:
            try:
                resultado[f"{nome}_paridade"] = conferir_paridade(modelo, blob, board_size,
                                                                  canais=canais)
            except Exception as e:                # noqa: BLE001 — ver docstring
                resultado[f"{nome}_paridade"] = {"erro": f"{type(e).__name__}: {e}"}
            if resultado[f"{nome}_kb"] < resultado["params"] / 4096:
                resultado[f"{nome}_alerta"] = (
                    "arquivo pequeno demais para esse número de parâmetros — "
                    "provável perda de pesos na conversão"
                )

    return resultado


# --- snakeai/plot.py ---
"""O gráfico da arena — onde os algoritmos finalmente ficam lado a lado.

Regras de leitura que este módulo impõe, e o porquê de cada uma:

* **Um eixo só.** Score de avaliação contra passos de ambiente. Nada de segundo eixo y:
  duas escalas empilhadas inventam correlação que não existe nos dados.
* **Cor é identidade, não posição.** Cada algoritmo recebe um slot fixo da paleta, sempre
  o mesmo. Filtrar a arena não repinta os sobreviventes — quem aprendeu que "PPO é azul"
  continua certo no gráfico seguinte.
* **Mediana com faixa interquartil**, nunca uma semente só. Uma curva de RL de execução
  única não é resultado, é anedota.
* **Curvas legadas em painel próprio.** Elas vêm de `comparable=False` e são medidas em
  *episódios*, não em passos de ambiente. Plotá-las no mesmo eixo x seria fabricar um eixo
  comum que não existe — o mesmo pecado do gráfico de dois eixos y, com outra roupa. Elas
  ganham um painel ao lado, com o próprio eixo rotulado, em cinza tracejado.
* **Piso e teto sempre visíveis.** Sem o piso aleatório de 1,21 desenhado, qualquer curva
  parece aprendizado; com ele, dá para ver quem só está tendo sorte.
* **Rótulo direto no fim de cada curva**, além da legenda. Três das cores da paleta clara
  ficam abaixo de 3:1 de contraste com o fundo, e a regra é que nesse caso a identidade
  não pode depender só da cor.

A paleta é a de referência do sistema de dataviz, validada para daltonismo nos dois modos
(pior par adjacente ΔE 9,1 no claro e 8,4 no escuro).
"""


import numpy as np

__all__ = ["PALETA", "VARIANTE_PRINCIPAL", "cores_por_algoritmo", "e_principal",
           "separa_principais", "arena_figure", "arena_familias",
           "arena_tempo", "arena_vitorias", "arena_table", "plot_run",
           "mesmo_hardware"]

# ---------------------------------------------------------------------- paleta
PALETA = {
    "light": {
        "surface": "#fcfcfb",
        "plane": "#f9f9f7",
        "ink": "#0b0b0b",
        "ink2": "#52514e",
        "muted": "#898781",
        "grid": "#e1e0d9",
        "axis": "#c3c2b7",
        "series": ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4",
                   "#008300", "#4a3aa7", "#e34948"],
        "legado": "#898781",
    },
    "dark": {
        "surface": "#1a1a19",
        "plane": "#0d0d0d",
        "ink": "#ffffff",
        "ink2": "#c3c2b7",
        "muted": "#898781",
        "grid": "#2c2c2a",
        "axis": "#383835",
        "series": ["#3987e5", "#d95926", "#199e70", "#c98500", "#d55181",
                   "#008300", "#9085e9", "#e66767"],
        "legado": "#898781",
    },
}

#: Ordem fixa dos slots. Um algoritmo novo entra no fim; ninguém troca de cor por isso.
ORDEM_ALGORITMOS = ["ppo", "dqn", "rainbow", "a2c", "acer", "alphazero",
                    "muzero", "acktr", "acektr", "dreamerv3", "lbc", "soap",
                    "dqn-legacy"]

#: A variante que **o notebook principal de cada algoritmo** produz na configuração padrão.
#: É a definição de "resultado principal" deste repositório: um braço por algoritmo, o que
#: `NN_algo.ipynb` roda sem tocar em nada.
#:
#: Existe porque o gráfico e a tabela respondem perguntas diferentes. A tabela é o registro
#: e leva tudo; o gráfico é a **comparação entre algoritmos**, e uma ablação ali dentro
#: compete visualmente com o braço que ela deveria explicar — o `ppo · esparso` desenha uma
#: segunda curva azul 17 pontos abaixo do PPO, e quem olha de longe lê "o PPO é instável"
#: em vez de "este é o controle de orçamento de gradiente". Ablação tem par próprio, e o
#: par está na tabela.
#:
#: Os nomes seguem o padrão dos agentes: a base é `cfg.net` na maioria, `base` no DQN,
#: `completo` no Rainbow, o preset no DreamerV3 e o parâmetro de busca no AlphaZero e no
#: MuZero. Qualquer desvio vira `+marca` ou `_sufixo` e, por construção, **deixa** de casar
#: com este mapa. Uma execução que não casa não some em silêncio: `arena --all` lista as que
#: ficaram fora do gráfico, pelo mesmo motivo do `COMPARABILITY.md`.
VARIANTE_PRINCIPAL = {
    "ppo": "resnet_small",
    "dqn": "base",
    "rainbow": "completo",
    "a2c": "resnet_small",
    "acer": "resnet_small",
    "alphazero": "sims32",
    "muzero": "unroll5",
    "acktr": "resnet_small",
    "acektr": "resnet_small",
    "dreamerv3": "dreamer_small",
    "lbc": "resnet_small",
    "soap": "resnet_small",
}


def e_principal(r):
    """A execução é o braço principal do algoritmo dela?

    `"default"` e `""` contam: é o que o `AgentBase` dá a quem não declara variante
    nenhuma, e não declarar é a definição de padrão. O mapa cobre os agentes que
    **derivam** o nome da configuração — todos os doze hoje.
    """
    return r.variant in ("default", "") or VARIANTE_PRINCIPAL.get(r.algo) == r.variant


def separa_principais(registros):
    """`(principais, ablacoes)` — a divisão que o gráfico usa e a tabela não."""
    principais = [r for r in registros if e_principal(r)]
    ablacoes = [r for r in registros if not e_principal(r)]
    return principais, ablacoes


#: Famílias, na ordem em que os painéis aparecem. Existem porque a arena passou de oito
#: algoritmos e **oito é o limite honesto de uma paleta categórica**: a nona cor seria
#: indistinguível de alguma das oito sob daltonismo. A saída não é gerar mais uma cor, é
#: mudar a forma do gráfico — *small multiples*, um painel por família.
#:
#: O agrupamento é o de sempre em RL, e não uma conveniência visual: o que o algoritmo
#: aprende (política, valor, ou um modelo do mundo) é a divisão que explica por que as
#: curvas têm formatos diferentes.
FAMILIAS = [
    ("política", "gradiente de política",
     ["ppo", "a2c", "acktr", "acektr", "acer", "lbc", "soap"]),
    ("valor", "função de valor", ["dqn", "rainbow"]),
    ("modelo", "modelo do mundo e busca", ["alphazero", "muzero", "dreamerv3"]),
]


def familia_de(algo):
    for chave, _, membros in FAMILIAS:
        if algo in membros:
            return chave
    return "outros"

PISO_ALEATORIO = 1.21
SCORE_PERFEITO = 97

#: Limiar padrão da coluna "passos até". 40 é bem acima do piso (1,21) e bem abaixo do teto
#: (97): alto o bastante para exigir que o agente jogue de verdade, baixo o bastante para
#: a maioria alcançar dentro do orçamento — um limiar que quase ninguém atinge não ordena
#: nada.
LIMIAR_PADRAO = 40.0


def cores_por_algoritmo(algoritmos, mode="light"):
    """Mapeia algoritmo -> cor, em ordem fixa. Nunca cicla nem gera hue nova.

    Passar do oitavo algoritmo é um erro deliberado: a nona cor seria
    indistinguível de alguma das oito sob daltonismo. Nesse ponto o gráfico
    precisa virar *small multiples*, não ganhar mais uma cor.
    """
    p = PALETA[mode]["series"]
    conhecidos = [a for a in ORDEM_ALGORITMOS if a in algoritmos]
    novos = sorted(a for a in algoritmos if a not in ORDEM_ALGORITMOS)
    ordenados = conhecidos + novos
    if len(ordenados) > len(p):
        raise ValueError(
            f"{len(ordenados)} algoritmos para {len(p)} slots de cor. "
            "Use `arena_figure(..., familias=True)` — small multiples por família — "
            "ou agrupe a cauda em 'outros'. Não gere cor nova."
        )
    return {a: p[i] for i, a in enumerate(ordenados)}


def cores_por_familia(mode="light"):
    """Cor de cada algoritmo **dentro do painel da sua família**.

    Nos *small multiples*, cada painel é uma unidade de leitura com no máximo quatro
    séries coloridas; as outras famílias aparecem em cinza, só para dar contexto. Duas
    famílias podem repetir um matiz — o que é seguro porque elas nunca aparecem coloridas
    no mesmo painel, e cada curva colorida ganha rótulo direto.

    A cor é presa ao algoritmo pela posição dele dentro da família, que é fixa. Filtrar
    execuções não repinta ninguém.
    """
    p = PALETA[mode]["series"]
    return {a: p[i] for _, _, membros in FAMILIAS for i, a in enumerate(membros)}


# ------------------------------------------------------------------ agregação
def agrega_sementes(registros, pontos=60):
    """Junta as sementes de uma mesma `(algo, variante)` numa mediana com faixa IQR.

    As sementes raramente avaliam nos mesmos passos, então interpolamos todas numa
    grade log-espaçada comum antes de tirar os quantis. A grade para no menor
    `max(step)` entre as sementes — extrapolar seria inventar dado.
    """
    curvas = []
    for r in registros:
        x, y = r.eval_curve()
        if x.size >= 2:
            curvas.append((x, y))
    if not curvas:
        return None

    x_min = max(1, max(c[0][0] for c in curvas))
    x_max = min(c[0][-1] for c in curvas)
    if x_max <= x_min:
        return None

    grade = np.unique(np.geomspace(x_min, x_max, pontos).astype(np.int64))
    empilhado = np.stack([np.interp(grade, x, y) for x, y in curvas])
    return {
        "x": grade,
        "mediana": np.median(empilhado, axis=0),
        "q1": np.percentile(empilhado, 25, axis=0),
        "q3": np.percentile(empilhado, 75, axis=0),
        "n_sementes": len(curvas),
    }


def mesmo_hardware(registros):
    """Todas as execuções vieram da mesma máquina? Devolve `(bool, conjunto)`.

    O eixo de tempo só significa alguma coisa dentro de um mesmo hardware. Uma curva feita
    numa P100 do Kaggle e outra numa T4 do Colab colocadas lado a lado em horas comparam os
    aceleradores, não os algoritmos — e o gráfico não avisaria.
    """
    hw = {r.hardware for r in registros}
    return len(hw) <= 1, hw


def agrega_tempo(registros, pontos=60):
    """Como `agrega_sementes`, no eixo de horas de GPU."""
    curvas = []
    for r in registros:
        h, y = r.eval_curve_tempo()
        ok = np.isfinite(h) & (h > 0)
        if ok.sum() >= 2:
            curvas.append((h[ok], y[ok]))
    if not curvas:
        return None

    x_min = max(1e-4, max(c[0][0] for c in curvas))
    x_max = min(c[0][-1] for c in curvas)
    if x_max <= x_min:
        return None
    grade = np.geomspace(x_min, x_max, pontos)
    emp = np.stack([np.interp(grade, h, y) for h, y in curvas])
    return {"x": grade, "mediana": np.median(emp, axis=0),
            "q1": np.percentile(emp, 25, axis=0), "q3": np.percentile(emp, 75, axis=0),
            "n_sementes": len(curvas)}


def _agrupa(registros):
    grupos = {}
    for r in registros:
        grupos.setdefault((r.algo, r.variant), []).append(r)
    return grupos


# -------------------------------------------------------------------- figuras
def arena_familias(registros, mode="light", figsize=(14.5, 4.8), titulo=None,
                   x_log=True, mostrar_legado=True, so_principais=True):
    """*Small multiples*: um painel por família, com as demais em cinza ao fundo.

    Esta é a forma que a arena assume quando passa de oito algoritmos. Ela não é um
    consolo por não caber tudo num painel — é melhor para a pergunta que a arena de fato
    responde. Sobrepor nove curvas com faixa interquartil produz um emaranhado onde a
    comparação relevante ("o Rainbow supera o DQN?") fica *mais* difícil, não menos.

    Cada painel mostra a família em cor e **todas as outras curvas em cinza claro**, na
    mesma escala. Sem esse fundo, três painéis lado a lado seriam três gráficos
    independentes e a comparação entre famílias se perderia — que é justamente o que a
    arena existe para permitir.
    """
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter

    p = PALETA[mode]
    cores = cores_por_familia(mode)
    comparaveis = [r for r in registros if r.oficial]
    if so_principais:
        comparaveis, _ = separa_principais(comparaveis)

    agregados = {}
    for (algo, variante), rs in sorted(_agrupa(comparaveis).items()):
        ag = agrega_sementes(rs)
        if ag is not None:
            agregados[(algo, variante)] = ag

    presentes = [f for f in FAMILIAS
                 if any(a in f[2] for a, _ in agregados)] or FAMILIAS
    legado = [r for r in registros if not r.comparable] if mostrar_legado else []

    # O painel legado entra com largura menor e **eixo x próprio**: ele mede episódios, e
    # pendurá-lo no eixo de passos seria fabricar um eixo comum que não existe.
    larguras = [1.0] * len(presentes) + ([0.62] if legado else [])
    fig = plt.figure(figsize=figsize, facecolor=p["plane"])
    gs = fig.add_gridspec(1, len(larguras), width_ratios=larguras, wspace=.08)
    axes = [fig.add_subplot(gs[0])]
    axes += [fig.add_subplot(gs[i], sharex=axes[0], sharey=axes[0])
             for i in range(1, len(presentes))]
    ax_leg = fig.add_subplot(gs[-1], sharey=axes[0]) if legado else None

    topo = max((max(ag["mediana"]) for ag in agregados.values()), default=0.0)
    topo = max(topo * 1.3, PISO_ALEATORIO * 4)

    for i, (ax, (chave, rotulo, membros)) in enumerate(zip(axes, presentes)):
        ax.set_facecolor(p["surface"])
        ax.axhline(PISO_ALEATORIO, color=p["muted"], lw=1.0, zorder=1)
        if i == len(presentes) - 1:
            # rotulada uma vez, e no painel mais vazio: repetir a referência nos três
            # seria ruído, e à esquerda ela cai em cima do início das curvas
            ax.annotate(f"piso aleatório · {PISO_ALEATORIO:.2f}".replace(".", ","),
                        xy=(0.98, PISO_ALEATORIO), xycoords=("axes fraction", "data"),
                        xytext=(0, 5), textcoords="offset points",
                        color=p["muted"], fontsize=8, va="bottom", ha="right")

        # contexto: todo o resto, em cinza, atrás
        for (algo, _), ag in agregados.items():
            if algo not in membros:
                ax.plot(ag["x"], ag["mediana"], color=p["legado"], lw=1.2,
                        alpha=.45, zorder=2, solid_capstyle="round")

        rotulos = []
        for (algo, variante), ag in agregados.items():
            if algo not in membros:
                continue
            cor = cores[algo]
            nome = algo if variante in ("default", "") else f"{algo} · {variante}"
            ax.fill_between(ag["x"], ag["q1"], ag["q3"], color=cor, alpha=.16,
                            linewidth=0, zorder=3)
            ax.plot(ag["x"], ag["mediana"], color=cor, lw=2.0, zorder=4,
                    label=f"{nome}  (n={ag['n_sementes']})", solid_capstyle="round")
            rotulos.append((ag["x"][-1], ag["mediana"][-1], nome, cor))

        for x, y, nome, _ in _sem_colisao(rotulos):
            ax.annotate(nome, xy=(x, y), xytext=(5, 0), textcoords="offset points",
                        color=p["ink2"], fontsize=8.5, va="center", ha="left", zorder=5)

        ax.set_title(rotulo, color=p["ink2"], fontsize=10.5, loc="left", pad=10)
        if x_log:
            ax.set_xscale("log")
        ax.set_ylim(0, topo)
        if not agregados:
            ax.set_xlim(1e4, 1e7)
        else:
            # espaço à direita para o rótulo direto de cada curva não sair do painel
            ax.margins(x=.30)
        ax.grid(True, which="major", color=p["grid"], lw=0.8, zorder=0)
        ax.set_axisbelow(True)
        for lado in ("top", "right"):
            ax.spines[lado].set_visible(False)
        for lado in ("left", "bottom"):
            ax.spines[lado].set_color(p["axis"])
        ax.tick_params(colors=p["muted"], labelsize=9, length=0)
        ax.xaxis.set_major_formatter(FuncFormatter(_formata_passos))
        ax.set_xlabel("passos de ambiente", color=p["ink2"], fontsize=9.5)
        if i:
            ax.tick_params(labelleft=False)
        if rotulos:
            leg = ax.legend(loc="upper left", frameon=False, fontsize=8.5,
                            labelcolor=p["ink2"], handlelength=1.6)
            for t in leg.get_texts():
                t.set_color(p["ink2"])

    if ax_leg is not None:
        _painel_legado(ax_leg, legado, p, ylim=(0, topo))
        ax_leg.tick_params(labelleft=False)

    axes[0].set_ylabel("score na avaliação (1.000 episódios, greedy)",
                       color=p["ink2"], fontsize=10)
    fig.suptitle(titulo or "snake-arena · por família de algoritmo",
                 color=p["ink"], fontsize=13, x=.006, ha="left", y=.985)
    fig.text(.006, .015,
             "cada painel colore uma família e mantém as demais curvas em cinza, na mesma "
             "escala; nove algoritmos não cabem numa paleta categórica. O painel do legado "
             "tem eixo x próprio, em episódios — ver docs/COMPARABILITY.md.",
             color=p["muted"], fontsize=8)
    fig.subplots_adjust(left=.058, right=.985, top=.83, bottom=.16)
    return fig, tuple(axes) + ((ax_leg,) if ax_leg is not None else ())


def arena_figure(registros, mode="light", figsize=(12.5, 6.2), titulo=None,
                 mostrar_legado=True, x_log=True, familias="auto", so_principais=True):
    """A figura principal do benchmark. Devolve `(fig, (ax, ax_legado))`.

    `familias="auto"` (o padrão) troca para *small multiples* assim que o número de
    algoritmos passa dos oito slots de cor. É automático de propósito: a alternativa
    seria a arena quebrar — ou, pior, ganhar uma nona cor — no dia em que o nono
    algoritmo termina de treinar.

    `registros` é uma lista de `snakeai.record.RunRecord` — tipicamente
    `record.load_all("runs")` mais as curvas legadas convertidas.

    `so_principais=True` (o padrão) desenha **um braço por algoritmo** — o que o notebook
    principal produz na configuração padrão, conforme `VARIANTE_PRINCIPAL`. As ablações
    saem do gráfico e continuam na tabela: aqui a pergunta é *quem vai mais longe com os
    mesmos dados*, e uma ablação desenhada ao lado do seu controle, na mesma cor, responde
    outra. Elas não somem em silêncio — a contagem vai no rodapé da figura e a lista, no
    `arena --all`.

    O painel grande tem só as execuções `comparable=True`, no eixo oficial de passos de
    ambiente. As legadas, quando existem, vão para um painel estreito à direita com o
    **próprio eixo em episódios** — porque é isso que elas medem, e fingir o contrário
    seria exatamente o erro que este repositório foi criado para consertar.
    """
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter

    p = PALETA[mode]
    comparaveis = [r for r in registros if r.oficial]
    ablacoes = []
    if so_principais:
        comparaveis, ablacoes = separa_principais(comparaveis)
    legado = [r for r in registros if not r.comparable] if mostrar_legado else []

    algos = {r.algo for r in comparaveis}
    if familias is True or (familias == "auto" and len(algos) > len(p["series"])):
        return arena_familias(registros, mode=mode, titulo=titulo, x_log=x_log,
                              mostrar_legado=mostrar_legado, so_principais=so_principais)

    cores = cores_por_algoritmo(algos, mode)

    fig = plt.figure(figsize=figsize, facecolor=p["plane"])
    if legado:
        gs = fig.add_gridspec(1, 2, width_ratios=(3.4, 1), wspace=.22)
        ax = fig.add_subplot(gs[0])
        ax_leg = fig.add_subplot(gs[1])
    else:
        ax = fig.add_subplot(1, 1, 1)
        ax_leg = None
    ax.set_facecolor(p["surface"])

    # --- referências primeiro, para ficarem atrás dos dados
    ax.axhline(PISO_ALEATORIO, color=p["muted"], lw=1.0, zorder=1)
    ax.annotate(f"piso aleatório com máscara · {PISO_ALEATORIO:.2f}".replace(".", ","),
                xy=(0.995, PISO_ALEATORIO), xycoords=("axes fraction", "data"),
                xytext=(0, 5), textcoords="offset points",
                color=p["muted"], fontsize=8.5, va="bottom", ha="right")

    # --- as curvas que competem
    rotulos = []
    for (algo, variante), rs in sorted(_agrupa(comparaveis).items()):
        ag = agrega_sementes(rs)
        if ag is None:
            continue
        cor = cores[algo]
        nome = algo if variante in ("default", "") else f"{algo} · {variante}"
        ax.fill_between(ag["x"], ag["q1"], ag["q3"], color=cor, alpha=.16,
                        linewidth=0, zorder=3)
        ax.plot(ag["x"], ag["mediana"], color=cor, lw=2.0, zorder=4,
                label=f"{nome}  (n={ag['n_sementes']})", solid_capstyle="round")
        rotulos.append((ag["x"][-1], ag["mediana"][-1], nome, cor))

    # o teto do eixo y vem de TODOS os dados, inclusive os legados: os dois painéis
    # compartilham a escala de score, e calcular só a partir das curvas oficiais faz o
    # painel da direita ser cortado quando a arena ainda está vazia. Ele é calculado
    # **antes** dos rótulos porque é a escala deles: ver `_sem_colisao`.
    topo_oficial = max((y for _, y, _, _ in rotulos), default=0.0)
    topo_legado = max(
        (max(c["train_score_mean"] for c in r.curve) for r in legado), default=0.0
    ) if legado else 0.0
    topo = max(topo_oficial * 1.3, topo_legado * 1.15, PISO_ALEATORIO * 4)

    # --- rótulo direto no fim de cada curva (a "relief rule" do contraste)
    for x, y, nome, cor in _sem_colisao(rotulos, minimo=0.033, escala=topo):
        ax.annotate(nome, xy=(x, y), xytext=(6, 0), textcoords="offset points",
                    color=p["ink2"], fontsize=9, va="center", ha="left", zorder=5)

    # --- eixos e cromo
    if x_log:
        ax.set_xscale("log")
    ax.set_xlabel("passos de ambiente", color=p["ink2"], fontsize=10)
    ax.set_ylabel("score na avaliação (1.000 episódios, greedy)",
                  color=p["ink2"], fontsize=10)
    ax.set_title(titulo or "snake-arena · mesmo ambiente, mesmo orçamento, mesma régua",
                 color=p["ink"], fontsize=13, pad=14, loc="left")

    ax.grid(True, which="major", color=p["grid"], lw=0.8, ls="-", zorder=0)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    for lado in ("left", "bottom"):
        ax.spines[lado].set_color(p["axis"])
        ax.spines[lado].set_linewidth(1.0)
    ax.tick_params(colors=p["muted"], labelsize=9, length=0)
    ax.xaxis.set_major_formatter(FuncFormatter(_formata_passos))

    ax.set_ylim(0, topo)
    if rotulos:
        ax.margins(x=.18)
    else:
        # arena vazia: um eixo x de 1 a 10 e um retângulo em branco não comunicam nada
        ax.set_xlim(1e4, 1e7)
        ax.annotate(
            "nenhuma execução oficial ainda\n\n"
            "as curvas entram aqui quando forem treinadas no orçamento do contrato",
            xy=(0.5, 0.55), xycoords="axes fraction", ha="center", va="center",
            color=p["muted"], fontsize=11, linespacing=1.6)

    if len(rotulos) >= 2:
        leg = ax.legend(loc="upper left", frameon=False, fontsize=9,
                        labelcolor=p["ink2"], handlelength=1.6)
        for t in leg.get_texts():
            t.set_color(p["ink2"])

    # --- painel legado: eixo próprio, unidade própria, mesma escala de score
    # o rodapé é uma linha por assunto: emendar os dois numa só passa da largura da
    # figura e o texto sai cortado na borda, que foi como ele nasceu
    linhas_rodape = [_nota_de_ablacoes(ablacoes)]
    if ax_leg is not None:
        _painel_legado(ax_leg, legado, p, ylim=ax.get_ylim())
        linhas_rodape.append(
            "Os dois painéis não compartilham eixo x — e não podem. À esquerda, "
            "passos de ambiente no jogo novo; à direita, episódios no jogo de 2019, "
            "com outra recompensa e score de treino em vez de avaliação.")
        fig.subplots_adjust(left=.075, right=.985, top=.88, bottom=.155)
    else:
        fig.tight_layout()
        if any(linhas_rodape):
            fig.subplots_adjust(bottom=.16)
    for i, linha in enumerate([t for t in linhas_rodape if t]):
        fig.text(0.012, 0.030 - i * 0.020, linha, color=p["muted"], fontsize=8)
    return fig, (ax, ax_leg)


def _nota_de_ablacoes(ablacoes):
    """O que ficou fora do gráfico, dito no próprio gráfico.

    Uma figura que esconde execuções sem avisar afirma que aquilo é tudo o que existe — o
    mesmo defeito que o `COMPARABILITY.md` chama de pior que incluir. O rodapé dá a
    contagem; a tabela dá os nomes, os números e o par de comparação de cada uma.
    """
    quantas = len({(r.algo, r.variant) for r in ablacoes})
    if not quantas:
        return ""
    if quantas == 1:
        fora = "1 configuração de ablação fica de fora dele e está na tabela"
    else:
        fora = (f"{quantas} configurações de ablação ficam de fora dele e estão na tabela")
    return f"O gráfico mostra o braço principal de cada algoritmo: {fora}, com o controle de cada uma."


def _painel_legado(ax, legado, p, ylim=None):
    """As curvas históricas, no eixo delas: episódios de treino.

    Compartilham a escala y com o painel principal — score é score, essa parte é
    conversível. O eixo x é que não é, e por isso está separado.
    """
    ax.set_facecolor(p["surface"])
    ax.axhline(PISO_ALEATORIO, color=p["muted"], lw=1.0, zorder=1)

    melhor = (None, -1.0)
    for r in legado:
        x = np.array([c["episodes"] for c in r.curve], dtype=np.float64)
        y = np.array([c["train_score_mean"] for c in r.curve], dtype=np.float64)
        if y.size > 400:
            # 10 mil episódios num painel estreito viram um borrão cinza; a janela
            # larga mostra a tendência, que é o que o painel de contexto precisa dizer.
            k = max(1, y.size // 40)
            nucleo = np.ones(k) / k
            y = np.convolve(y, nucleo, mode="valid")
            x = x[k - 1:]
        ax.plot(x, y, color=p["legado"], lw=1.2, ls=(0, (4, 3)), alpha=.6, zorder=2)
        if y.max() > melhor[1]:
            melhor = (r.variant, float(y.max()), float(x[int(y.argmax())]))

    if melhor[0]:
        # rótulo ancorado no canto, não no ponto: no painel estreito um rótulo junto
        # ao máximo sai pela borda direita
        ax.annotate(f"melhor: {melhor[0]}\nmédia móvel {melhor[1]:.1f}".replace(".", ","),
                    xy=(0.04, 0.97), xycoords="axes fraction",
                    color=p["ink2"], fontsize=8.5, ha="left", va="top",
                    linespacing=1.5)

    ax.set_title("legado · 2019", color=p["ink2"], fontsize=10, loc="left", pad=14)
    ax.set_xlabel("episódios de treino", color=p["muted"], fontsize=9)
    if ylim:
        ax.set_ylim(ylim)
    ax.grid(True, color=p["grid"], lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    for lado in ("left", "bottom"):
        ax.spines[lado].set_color(p["axis"])
    ax.tick_params(colors=p["muted"], labelsize=8.5, length=0)
    ax.xaxis.set_major_formatter(__import__("matplotlib").ticker.FuncFormatter(_formata_passos))


def _ate_o_limiar(registros, limiar):
    """Mediana dos passos até `limiar`, e quantas sementes chegaram lá.

    Quem não chegou **não** entra na mediana como um número grande inventado: fica de fora
    e o `n` denuncia. Uma mediana calculada sobre metade das sementes que chegaram, sem
    dizer que foi metade, seria a pior das duas opções.
    """
    passos = [r.passos_ate(limiar) for r in registros]
    chegaram = [p for p in passos if p is not None]
    return {"passos_ate": int(np.median(chegaram)) if chegaram else None,
            "sementes_ate": len(chegaram), "limiar": limiar}


def _sem_colisao(rotulos, minimo=0.045, escala=None):
    """Empurra rótulos que ficariam sobrepostos, preservando a ordem vertical.

    `escala` é o **teto do eixo**, e passá-la é o que faz a separação valer em pixels em
    vez de em pontos de score. Sem ela a referência é a faixa dos próprios rótulos — que
    encolhe justamente quando as curvas convergem, ou seja, o mínimo fica menor no único
    caso em que ele importa. Com os seis braços principais terminando entre 47 e 82, PPO,
    ACKTR e ACER escreviam um por cima do outro.
    """
    if not rotulos:
        return []
    ordenado = sorted(rotulos, key=lambda t: t[1])
    ys = [t[1] for t in ordenado]
    faixa = max(ys[-1] - ys[0], 1e-9)
    minimo = minimo * (escala if escala else faixa)
    for i in range(1, len(ys)):
        if ys[i] - ys[i - 1] < minimo:
            ys[i] = ys[i - 1] + minimo
    return [(x, ys[i], nome, cor) for i, (x, _, nome, cor) in enumerate(ordenado)]


def _formata_passos(v, _pos=None):
    if v >= 1e6:
        return f"{v / 1e6:g} M"
    if v >= 1e3:
        return f"{v / 1e3:g} mil"
    return f"{v:g}"


def plot_run(record, mode="light", figsize=(11, 3.4)):
    """Diagnóstico de uma execução: treino (com exploração) contra avaliação (honesta).

    As duas subindo juntas = aprendeu. A de treino subindo sozinha = está explorando com
    sorte, e o número honesto não acompanha.
    """
    import matplotlib.pyplot as plt

    p = PALETA[mode]
    fig, ax = plt.subplots(figsize=figsize, facecolor=p["plane"])
    ax.set_facecolor(p["surface"])

    treino = [(c["global_step"], c["train_score_mean"]) for c in record.curve
              if c.get("train_score_mean") is not None]
    if treino:
        x, y = zip(*treino)
        ax.plot(x, y, color=p["muted"], lw=1.4, label="treino (com exploração)")

    x, y = record.eval_curve()
    if x.size:
        ax.plot(x, y, color=p["series"][0], lw=2.0, label="avaliação (greedy)")

    ax.axhline(PISO_ALEATORIO, color=p["muted"], lw=1.0)
    ax.set_xlabel("passos de ambiente", color=p["ink2"], fontsize=10)
    ax.set_ylabel("score", color=p["ink2"], fontsize=10)
    ax.set_title(record.run_id, color=p["ink"], fontsize=12, loc="left", pad=10)
    ax.grid(True, color=p["grid"], lw=0.8)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    for lado in ("left", "bottom"):
        ax.spines[lado].set_color(p["axis"])
    ax.tick_params(colors=p["muted"], labelsize=9, length=0)
    leg = ax.legend(frameon=False, fontsize=9)
    for t in leg.get_texts():
        t.set_color(p["ink2"])
    fig.tight_layout()
    return fig, ax


# --------------------------------------------------------------------- tabela
def arena_vitorias(registros, mode="light", figsize=(8.6, 4.4), titulo=None,
                   so_principais=True):
    """Como cada episódio **termina** — e a taxa de vitória dentro disso.

    Por que este painel existe
    --------------------------
    "Melhor" não é uma coisa só. A média e a taxa de vitória são dois funcionais da mesma
    distribuição — `E[X]` e `P(X = perfeito)` — e **elas discordam nestes dados**: o
    Rainbow é o penúltimo em média e ainda assim fecha o tabuleiro **nove vezes mais** que
    o A2C, que tem 15 pontos a mais de score (19,9% contra 2,2%). Publicar só a média
    deixaria isso invisível.

    O que cada uma joga fora explica a discordância. A taxa de vitória é um limiar no
    extremo: um episódio de 96 conta igual a um de 3, e é por isso que o A2C — que joga
    bem e morre na casa dos 90 — tem 69,61 de média e 2,2% de vitória. A média usa o
    episódio inteiro, mas não distingue "sempre 78" de "metade perfeito, metade zero".

    Por isso a barra não é a taxa de vitória sozinha: é a **repartição inteira** das causas
    de fim, com a vitória como primeiro segmento. É a mesma leitura que o log de treino já
    faz por iteração — score sozinho é ambíguo, 1,2 ponto pode ser "bate em tudo" ou "anda
    em círculo até morrer de fome" — trazida para o fim da execução.

    Ele **não substitui** o painel oficial, pela mesma razão que o `arena_tempo` não
    substitui: é outra pergunta. Esta aqui é *qual eu levaria para o jogo*; a oficial é
    *quem aprende mais com os mesmos dados*.
    """
    import matplotlib.pyplot as plt

    p = PALETA[mode]
    comparaveis = [r for r in registros if r.oficial]
    if so_principais:
        comparaveis, _ = separa_principais(comparaveis)

    linhas = []
    for (algo, variante), rs in _agrupa(comparaveis).items():
        partes = [_mediana_do_final(rs, k) for k in
                  ("fim_tabuleiro_cheio", "fim_colisao", "fim_fome")]
        if any(v is None for v in partes):
            continue                      # protocolo antigo, sem as chaves de causa
        # as três medianas são tomadas por semente e independentes, então a soma foge de
        # 1 por frações de ponto. Normalizar é honesto porque o que a barra afirma é a
        # **proporção** entre as causas; o número impresso é a mediana, não o normalizado.
        total = sum(partes) or 1.0
        linhas.append({
            "nome": algo if variante in ("default", "") else f"{algo} · {variante}",
            "vitoria": partes[0],
            "fatias": [v / total for v in partes],
            "media": _mediana_do_final(rs, "score_mean") or 0.0,
            "n": len(rs),
        })

    linhas.sort(key=lambda d: d["vitoria"])
    fig, ax = plt.subplots(figsize=figsize, facecolor=p["plane"])
    ax.set_facecolor(p["surface"])

    cores = (p["series"][2], p["series"][1], p["muted"])
    nomes = ("tabuleiro cheio", "colisão", "fome")
    y = list(range(len(linhas)))
    for i, linha in enumerate(linhas):
        esquerda = 0.0
        for fatia, cor, nome in zip(linha["fatias"], cores, nomes):
            ax.barh(i, fatia * 100, left=esquerda * 100, color=cor, height=.62,
                    zorder=3, label=nome if i == 0 else None,
                    edgecolor=p["surface"], linewidth=1.2)
            esquerda += fatia
        # o número fica **fora** da barra: dentro ele some quando a fatia é 2%, que é
        # justamente o caso que esta figura existe para mostrar
        ax.annotate(f"{linha['vitoria'] * 100:.1f}%".replace(".", ","),
                    xy=(101, i), xytext=(0, 0), textcoords="offset points",
                    color=p["ink"], fontsize=9.5, va="center", ha="left", zorder=5)
        ax.annotate(f"média {linha['media']:.2f}".replace(".", ","),
                    xy=(119, i), color=p["muted"], fontsize=9, va="center", ha="left",
                    zorder=5)

    ax.set_yticks(y, [linha["nome"] for linha in linhas],
                  color=p["ink2"], fontsize=9.5)
    ax.set_xlim(0, 145)
    ax.set_xticks([0, 25, 50, 75, 100], ["0", "25%", "50%", "75%", "100%"])
    ax.set_xlabel("como os 1.000 episódios de avaliação terminam",
                  color=p["ink2"], fontsize=10, loc="left")
    ax.set_title(titulo or "snake-arena · quem fecha o tabuleiro",
                 color=p["ink"], fontsize=13, pad=14, loc="left")
    ax.grid(True, axis="x", color=p["grid"], lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    for lado in ("top", "right", "left"):
        ax.spines[lado].set_visible(False)
    ax.spines["bottom"].set_color(p["axis"])
    # a linha do eixo para nos 100%: ela não tem o que dizer embaixo dos rótulos
    ax.spines["bottom"].set_bounds(0, 100)
    ax.tick_params(colors=p["muted"], labelsize=9, length=0)

    if linhas:
        leg = ax.legend(loc="lower right", bbox_to_anchor=(1.0, 1.0), ncols=3,
                        frameon=False, fontsize=9, handlelength=1.2)
        for t in leg.get_texts():
            t.set_color(p["ink2"])

    for i, linha in enumerate((
            "A ordem aqui não é a ordem do painel oficial — e é isso que a figura tem a "
            "dizer.",
            "Média e taxa de vitória medem coisas diferentes da mesma distribuição; "
            "nenhuma das duas é 'a qualidade do modelo'.")):
        fig.text(.012, .042 - i * .024, linha, color=p["muted"], fontsize=8)
    fig.subplots_adjust(left=.20, right=.985, top=.80, bottom=.24)
    return fig, ax


def _mediana_do_final(registros, chave):
    """Mediana entre sementes de um campo de `final` — a estatística oficial da arena."""
    valores = [r.final.get(chave) for r in registros if r.final.get(chave) is not None]
    return float(np.median(valores)) if valores else None


def arena_tempo(registros, mode="light", figsize=(7.4, 5.0), titulo=None,
                limiar=LIMIAR_PADRAO, so_principais=True):
    """A arena no eixo de **custo**: score contra horas de GPU. Devolve `(fig, ax)`.

    O eixo oficial — passos de ambiente — iguala os *dados vistos*. É o padrão da
    literatura e é o certo para "quem aprende mais com a mesma experiência". Mas ele
    esconde uma diferença enorme: o AlphaZero roda uma busca em árvore a cada passo e custa
    ordens de grandeza mais que o DQN para chegar ao mesmo ponto no eixo x. Comparar ali dá
    a ele computação de graça.

    Este painel mostra a outra metade da verdade, e **não substitui** o oficial: são duas
    perguntas diferentes, e a resposta de uma não vale para a outra.

    Quando as execuções vêm de hardwares diferentes o gráfico **diz isso na cara**, porque
    aí ele compara aceleradores e não algoritmos — e essa é a forma mais fácil de ler um
    número errado com confiança.
    """
    import matplotlib.pyplot as plt

    p = PALETA[mode]
    comparaveis = [r for r in registros if r.oficial]
    if so_principais:
        comparaveis, _ = separa_principais(comparaveis)

    algos = {r.algo for r in comparaveis}
    if len(algos) > len(p["series"]):
        # A mesma regra do painel principal: acima de oito, a saída é mudar a forma do
        # gráfico, não gerar cor nova. Aqui isso vira um painel de tempo por família.
        return arena_tempo_familias(registros, mode=mode, titulo=titulo,
                                    so_principais=so_principais)
    # `cores_por_algoritmo`, e **não** `cores_por_familia`: num painel único a cor por
    # posição-dentro-da-família repete matiz entre famílias, e três curvas azuis no mesmo
    # eixo é exatamente a ambiguidade que a paleta existe para evitar.
    cores = cores_por_algoritmo(algos, mode)

    fig, ax = plt.subplots(figsize=figsize, facecolor=p["plane"])
    ax.set_facecolor(p["surface"])
    ax.axhline(PISO_ALEATORIO, color=p["muted"], lw=1.0, zorder=1)

    rotulos, topo = [], 0.0
    for (algo, variante), rs in sorted(_agrupa(comparaveis).items()):
        ag = agrega_tempo(rs)
        if ag is None:
            continue
        cor = cores[algo]
        nome = algo if variante in ("default", "") else f"{algo} · {variante}"
        ax.fill_between(ag["x"], ag["q1"], ag["q3"], color=cor, alpha=.16, linewidth=0,
                        zorder=3)
        ax.plot(ag["x"], ag["mediana"], color=cor, lw=2.0, zorder=4,
                label=f"{nome}  (n={ag['n_sementes']})", solid_capstyle="round")
        rotulos.append((ag["x"][-1], ag["mediana"][-1], nome, cor))
        topo = max(topo, float(ag["mediana"].max()))

    for x, y, nome, _ in _sem_colisao(rotulos):
        ax.annotate(nome, xy=(x, y), xytext=(6, 0), textcoords="offset points",
                    color=p["ink2"], fontsize=9, va="center", ha="left", zorder=5)

    ax.set_xscale("log")
    ax.set_xlabel("horas de GPU (inclui as avaliações periódicas)",
                  color=p["ink2"], fontsize=10)
    ax.set_ylabel("score na avaliação (1.000 episódios, greedy)",
                  color=p["ink2"], fontsize=10)
    ax.set_title(titulo or "snake-arena · o mesmo resultado, no eixo do custo",
                 color=p["ink"], fontsize=13, pad=14, loc="left")
    ax.set_ylim(0, max(topo * 1.3, PISO_ALEATORIO * 4))
    ax.margins(x=.22)
    ax.grid(True, which="major", color=p["grid"], lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    for lado in ("left", "bottom"):
        ax.spines[lado].set_color(p["axis"])
    ax.tick_params(colors=p["muted"], labelsize=9, length=0)
    if rotulos:
        leg = ax.legend(loc="upper left", frameon=False, fontsize=9)
        for t in leg.get_texts():
            t.set_color(p["ink2"])

    igual, hw = mesmo_hardware(comparaveis)
    aviso = ("mesmo hardware em todas as execuções: " + (next(iter(hw)) if hw else "—")
             if igual else
             "⚠ HARDWARES DIFERENTES (" + " · ".join(sorted(hw)) +
             "): este eixo está comparando aceleradores, não algoritmos")
    fig.text(.012, .015, aviso, color=p["muted"] if igual else p["ink"], fontsize=8.5)
    fig.subplots_adjust(left=.115, right=.97, top=.9, bottom=.145)
    return fig, ax


def arena_tempo_familias(registros, mode="light", figsize=(14.5, 4.6), titulo=None,
                         so_principais=True):
    """`arena_tempo` acima de oito algoritmos: um painel por família, o resto em cinza."""
    import matplotlib.pyplot as plt

    p = PALETA[mode]
    cores = cores_por_familia(mode)
    comparaveis = [r for r in registros if r.oficial]
    if so_principais:
        comparaveis, _ = separa_principais(comparaveis)

    agregados = {}
    for chave, rs in sorted(_agrupa(comparaveis).items()):
        ag = agrega_tempo(rs)
        if ag is not None:
            agregados[chave] = ag

    presentes = [f for f in FAMILIAS if any(a in f[2] for a, _ in agregados)] or FAMILIAS
    fig, axes = plt.subplots(1, len(presentes), figsize=figsize, sharex=True, sharey=True,
                             facecolor=p["plane"])
    axes = np.atleast_1d(axes)
    topo = max((float(a["mediana"].max()) for a in agregados.values()), default=0.0)

    for i, (ax, (_, rotulo, membros)) in enumerate(zip(axes, presentes)):
        ax.set_facecolor(p["surface"])
        ax.axhline(PISO_ALEATORIO, color=p["muted"], lw=1.0, zorder=1)
        for (algo, _), ag in agregados.items():
            if algo not in membros:
                ax.plot(ag["x"], ag["mediana"], color=p["legado"], lw=1.2, alpha=.45,
                        zorder=2)
        for (algo, variante), ag in agregados.items():
            if algo not in membros:
                continue
            nome = algo if variante in ("default", "") else f"{algo} · {variante}"
            ax.plot(ag["x"], ag["mediana"], color=cores[algo], lw=2.0, zorder=4,
                    label=f"{nome}  (n={ag['n_sementes']})", solid_capstyle="round")
        ax.set_xscale("log")
        ax.set_ylim(0, max(topo * 1.3, PISO_ALEATORIO * 4))
        ax.set_title(rotulo, color=p["ink2"], fontsize=10.5, loc="left", pad=10)
        ax.set_xlabel("horas de GPU", color=p["ink2"], fontsize=9.5)
        ax.grid(True, color=p["grid"], lw=0.8, zorder=0)
        ax.set_axisbelow(True)
        for lado in ("top", "right"):
            ax.spines[lado].set_visible(False)
        ax.tick_params(colors=p["muted"], labelsize=9, length=0, labelleft=(i == 0))
        if any(a in membros for a, _ in agregados):
            leg = ax.legend(loc="upper left", frameon=False, fontsize=8.5)
            for t in leg.get_texts():
                t.set_color(p["ink2"])

    axes[0].set_ylabel("score na avaliação", color=p["ink2"], fontsize=10)
    fig.suptitle(titulo or "snake-arena · o mesmo resultado, no eixo do custo",
                 color=p["ink"], fontsize=13, x=.006, ha="left", y=.985)
    igual, hw = mesmo_hardware(comparaveis)
    fig.text(.006, .015,
             ("mesmo hardware: " + (next(iter(hw)) if hw else "—")) if igual else
             "⚠ HARDWARES DIFERENTES (" + " · ".join(sorted(hw)) +
             "): este eixo compara aceleradores, não algoritmos",
             color=p["muted"] if igual else p["ink"], fontsize=8.5)
    fig.subplots_adjust(left=.058, right=.99, top=.83, bottom=.17)
    return fig, tuple(axes)


def arena_table(registros, markdown=True, limiar=LIMIAR_PADRAO):
    """A tabela de resultados — a visão que o gráfico não dá.

    Existe também porque três cores da paleta clara ficam abaixo de 3:1 de contraste:
    a regra manda oferecer rótulos visíveis **ou** a visão em tabela. Aqui temos as duas.
    """
    linhas = []
    for (algo, variante), rs in sorted(_agrupa([r for r in registros if r.oficial]).items()):
        finais = [r.final for r in rs if r.final]
        if not finais:
            continue
        medias = np.array([f["score_mean"] for f in finais], dtype=np.float64)
        passos = max((r.curve[-1]["global_step"] for r in rs if r.curve), default=0)
        linhas.append({
            "algo": algo,
            "variante": variante,
            "rede": rs[0].net,
            "params": rs[0].params,
            "sementes": len(rs),
            "passos": int(passos),
            "score_mean": float(np.median(medias)),
            "score_spread": float(medias.max() - medias.min()) if len(medias) > 1 else 0.0,
            "score_median": float(np.median([f.get("score_median", np.nan) for f in finais])),
            "score_max": int(max(f.get("score_max", 0) for f in finais)),
            "win_rate": float(np.median([f.get("win_rate", 0.0) for f in finais])),
            # coluna à parte, como o filtro de flood-fill e a busca do AlphaZero: o
            # melhor checkpoint responde "o melhor que este algoritmo produziu", que não
            # é a mesma pergunta que "como ele terminou"
            "melhor_mean": float(np.median(
                [r.melhor["score_mean"] for r in rs if r.melhor])) if any(
                    r.melhor for r in rs) else None,
            # A leitura HORIZONTAL da curva: em vez de "quanto marcou no fim", "quantos
            # passos precisou para chegar a `limiar`". Sai dos mesmos dados e responde à
            # outra pergunta — eficiência amostral no sentido estrito.
            **_ate_o_limiar(rs, limiar),
            "horas": float(np.median([r.meta.get("wall_s_total", np.nan) / 3600
                                      for r in rs])),
        })
    linhas.sort(key=lambda d: -d["score_mean"])

    if not markdown:
        return linhas

    out = [
        "| algoritmo | rede | params | sementes | passos | score (last) | melhor ckpt | "
        + f"passos até {limiar:.0f} | horas | amplitude | mediana/ep | máx | cheio |",
        "|---|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|",
        f"| _piso aleatório_ | — | — | — | 0 | **{PISO_ALEATORIO:.2f}** | — | — | — | — | 1 | — | 0% |".replace(".", ","),
    ]
    for d in linhas:
        nome = d["algo"] if d["variante"] in ("default", "") else f"{d['algo']} · {d['variante']}"
        melhor = f"{d['melhor_mean']:.2f}" if d["melhor_mean"] is not None else "—"
        if d["passos_ate"] is None:
            ate = "não chegou"
        else:
            ate = f"{d['passos_ate']:,}"
            if d["sementes_ate"] < d["sementes"]:
                ate += f" ({d['sementes_ate']}/{d['sementes']})"
        horas = f"{d['horas']:.1f}" if np.isfinite(d["horas"]) else "—"
        out.append(
            f"| {nome} | `{d['rede']}` | {d['params']:,} | {d['sementes']} | "
            f"{d['passos']:,} | **{d['score_mean']:.2f}** | {melhor} | {ate} | {horas} | "
            f"±{d['score_spread']:.2f} | "
            f"{d['score_median']:.0f} | {d['score_max']} | {d['win_rate']:.1%} |"
        )
    out.append(f"\nScore perfeito no 10×10: **{SCORE_PERFEITO}**.")
    out.append(
        f"\n**passos até {limiar:.0f}** é a curva lida na horizontal em vez da vertical: "
        "em vez de *quanto marcou no fim*, *quantos passos precisou para chegar lá*. Sai "
        "dos mesmos dados e responde à outra pergunta — menor é melhor. A resolução é a "
        "cadência de avaliação, e não há interpolação: o passo mostrado é um em que a "
        "medição de fato aconteceu. `(k/n)` significa que só `k` das `n` sementes "
        "chegaram, e as que não chegaram ficam **fora** da mediana em vez de entrar como "
        "um número inventado."
    )
    out.append(
        "\n**horas** é tempo de parede da execução inteira, útil só entre execuções do "
        "mesmo hardware. O eixo de passos iguala os *dados vistos*; ele não iguala o "
        "*esforço*, e a diferença entre os dois é enorme para quem faz busca em árvore."
    )
    out.append(
        "\nA coluna **score (last)** é o número oficial: o modelo do último passo, que é o "
        "estado final do algoritmo. O valor é a **mediana entre as sementes** do score "
        "médio de cada uma — não a média entre elas. É a mesma estatística que o gráfico "
        "desenha como linha, com o intervalo entre sementes como faixa, e com três "
        "sementes ela é o que uma semente divergente não consegue arrastar. Os documentos "
        "de ablação (`ORCAMENTO_DE_GRADIENTE.md`, `CANAL_DE_FOME.md`) reportam **média e "
        "desvio**, porque lá a pergunta é o tamanho de um efeito, não a ordem de um "
        "ranking: os dois números convivem, e cada um diz qual é. **mediana/ep** é outra "
        "coisa ainda — a mediana entre *episódios*, não entre sementes. **melhor ckpt** é "
        "o melhor que aquela execução produziu em algum momento — fica à parte porque "
        "premia quem foi medido mais vezes, pela mesma razão que a busca do AlphaZero e o "
        "filtro de flood-fill ficam fora da curva."
    )
    return "\n".join(out)


# --- snakeai/nets/resnet.py ---
"""Tronco residual totalmente convolucional — a rede do PPO.

No espírito do AlphaZero, mas minúsculo. Convoluções 3×3 com `padding="same"` num
tabuleiro 10×10 dão campo receptivo global depois de ~5 camadas, então 3 blocos residuais
já enxergam o tabuleiro inteiro — **sem jogar fora a posição**, que é onde as redes com
pooling do repositório antigo se perdiam.

Sobre normalização: PPO e BatchNorm se dão mal. As estatísticas do rollout não batem com
as do minibatch de update, e o valor aprendido fica dependente do tamanho do lote. Usamos
**GroupNorm**, que normaliza por amostra e não tem esse problema — e que funciona igual
para DQN, o que mantém a comparação limpa.
"""


import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

from keras import layers

__all__ = ["PRESETS", "residual_block", "resnet", "TRONCOS_RESIDUAIS"]

#: nome -> (largura, número de blocos residuais)
PRESETS = {
    "resnet_tiny": (32, 2),     # ~40k params com a cabeça
    "resnet_small": (48, 3),    # ~135k — o ponto doce
    "resnet_base": (64, 4),     # ~320k
}


def residual_block(x, largura, nome):
    y = layers.Conv2D(largura, 3, padding="same", use_bias=False,
                      kernel_initializer="he_normal", name=f"{nome}_c1")(x)
    y = layers.GroupNormalization(groups=8, name=f"{nome}_n1")(y)
    y = layers.Activation("relu", name=f"{nome}_a1")(y)
    y = layers.Conv2D(largura, 3, padding="same", use_bias=False,
                      kernel_initializer="he_normal", name=f"{nome}_c2")(y)
    y = layers.GroupNormalization(groups=8, name=f"{nome}_n2")(y)
    out = layers.Add(name=f"{nome}_add")([x, y])
    return layers.Activation("relu", name=f"{nome}_a2")(out)


def resnet(x, preset="resnet_small", nome=None):
    """Tronco residual. Devolve o mapa de features `(B, B, largura)`, **sem achatar**.

    Não achatar é de propósito: as cabeças convolucionais 1×1 de `heads.py` aproveitam a
    estrutura espacial, e achatar cedo seria desperdiçá-la.
    """
    if preset not in PRESETS:
        raise ValueError(f"preset desconhecido: {preset!r}. Use um de {list(PRESETS)}")
    largura, blocos = PRESETS[preset]
    nome = nome or preset

    x = layers.Conv2D(largura, 3, padding="same", use_bias=False,
                      kernel_initializer="he_normal", name=f"{nome}_stem_c")(x)
    x = layers.GroupNormalization(groups=8, name=f"{nome}_stem_n")(x)
    x = layers.Activation("relu", name=f"{nome}_stem_a")(x)
    for i in range(blocos):
        x = residual_block(x, largura, f"{nome}_res{i}")
    return x


TRONCOS_RESIDUAIS = {
    nome: (lambda x, _p=nome: resnet(x, preset=_p)) for nome in PRESETS
}


# --- snakeai/nets/classic.py ---
"""Os troncos convolucionais do `colab-rl`, portados para Keras 3 e corrigidos.

Estes são os corpos de rede que produziram as curvas históricas. Estão aqui para que a
pergunta "quanto do ganho é o algoritmo e quanto é a arquitetura?" tenha resposta medida
em vez de opinião.

Duas coisas que a portabilidade revelou, e que valem mais que o código
------------------------------------------------------------------------

**1. "CNN2" significava duas coisas diferentes no mesmo repositório.**

O `colab-rl` tinha as CNNs definidas em dois lugares, com a mesma numeração e conteúdo
diferente:

===========  ==============================  ==================================
nome         em `models/utilities/networks.py`  nos notebooks
===========  ==============================  ==================================
``CNN1``     16→32, **quebrada** (`return model`)  32→64→64 (Rainbow)
``CNN2``     16→32→32, **quebrada**             32→64→64 com regularização L2
``CNN3``     32→64→64 (Rainbow)                 3 blocos VGG com max-pooling
``CNN4``     não existia                        idem CNN3, com dropout
===========  ==============================  ==================================

Ou seja: o notebook chamado *"DQN (RMSprop - CNN2 - KL-Divergence)"* usava um tronco que
**não é** a `CNN2` do pacote. Um leitor que fosse ao `networks.py` entender o experimento
leria a rede errada. Aqui as redes têm nome descritivo (`cnn_rainbow`, `cnn_alphazero`,
`cnn_vgg`, `cnn_vgg_dropout`) e os apelidos numéricos apontam para as definições **dos
notebooks**, que são as que de fato rodaram.

**2. As redes com pooling destroem o tabuleiro.**

`cnn_vgg` e `cnn_vgg_dropout` aplicam três `MaxPooling2D(2, 2)` seguidos. Num tabuleiro
10×10 isso é ``10 → 5 → 2 → 1``: a saída do tronco tem **uma única célula**. Toda a
informação de *onde* as coisas estão no tabuleiro é jogada fora antes da cabeça densa —
sobra só "existe corpo em algum lugar", "existe comida em algum lugar".

Essas arquiteturas foram desenhadas para imagens 224×224, onde três poolings deixam 28×28.
Copiadas para 10×10, elas colapsam. É uma explicação forte para o platô dos notebooks que
as usavam, e por isso `cnn_vgg_sem_pool` existe: mesma rede, sem os poolings, para medir
exatamente quanto custou.

Os troncos são fiéis ao original de propósito. A correção fica na cabeça (`heads.py`) e
nas variantes explicitamente marcadas.
"""


import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, regularizers

__all__ = [
    "cnn_rainbow",
    "cnn_alphazero",
    "cnn_vgg",
    "cnn_vgg_dropout",
    "cnn_vgg_sem_pool",
    "TRONCOS_CLASSICOS",
    "APELIDOS_LEGADOS",
]


def cnn_rainbow(x, nome="cnn_rainbow"):
    """32→64→64 com kernels 3×3, 2×2, 1×1, sem padding.

    Da implementação do Rainbow do @Kaixhin. É a `CNN1` dos notebooks e a `CNN3` do
    `networks.py` — a mesma rede com dois nomes.
    """
    x = layers.Conv2D(32, 3, activation="relu", name=f"{nome}_c1")(x)
    x = layers.Conv2D(64, 2, activation="relu", name=f"{nome}_c2")(x)
    x = layers.Conv2D(64, 1, activation="relu", name=f"{nome}_c3")(x)
    return layers.Flatten(name=f"{nome}_flat")(x)


def cnn_alphazero(x, l2const=1e-4, nome="cnn_alphazero"):
    """A mesma pilha da `cnn_rainbow`, com regularização L2 e ativação separada.

    É a `CNN2` **dos notebooks** — a que rodou no experimento "CNN2 - KL-Divergence".
    """
    reg = regularizers.l2(l2const)
    for i, (filtros, k) in enumerate(((32, 3), (64, 2), (64, 1)), start=1):
        x = layers.Conv2D(filtros, k, kernel_regularizer=reg, name=f"{nome}_c{i}")(x)
        x = layers.Activation("relu", name=f"{nome}_a{i}")(x)
    return layers.Flatten(name=f"{nome}_flat")(x)


def _blocos_vgg(x, nome, dropout=0.0, pooling=True):
    plano = ((16, 2), (32, 2), (64, 3))
    for b, (filtros, convs) in enumerate(plano, start=1):
        for c in range(1, convs + 1):
            x = layers.Conv2D(filtros, 3, activation="relu", padding="same",
                              name=f"{nome}_b{b}_c{c}")(x)
            if dropout:
                x = layers.Dropout(dropout, name=f"{nome}_b{b}_d{c}")(x)
        if pooling:
            x = layers.MaxPooling2D(2, strides=2, name=f"{nome}_b{b}_pool")(x)
    return layers.Flatten(name=f"{nome}_flat")(x)


def cnn_vgg(x, nome="cnn_vgg"):
    """Três blocos no estilo VGG com max-pooling. É a `CNN3` dos notebooks.

    **Atenção:** os três poolings reduzem um tabuleiro 10×10 a 1×1. Ver o cabeçalho do
    módulo. Mantida fiel ao original porque é o que produziu as curvas históricas.
    """
    return _blocos_vgg(x, nome, dropout=0.0, pooling=True)


def cnn_vgg_dropout(x, nome="cnn_vgg_dropout", taxa=0.1):
    """`cnn_vgg` com dropout de 0,1 após cada convolução. É a `CNN4` dos notebooks."""
    return _blocos_vgg(x, nome, dropout=taxa, pooling=True)


def cnn_vgg_sem_pool(x, nome="cnn_vgg_sem_pool"):
    """`cnn_vgg` sem os max-poolings — a variante de ablação.

    Não existia no repositório antigo. Existe aqui para responder, com número, quanto do
    platô daquelas execuções veio de colapsar o tabuleiro a uma célula.
    """
    return _blocos_vgg(x, nome, dropout=0.0, pooling=False)


#: Nome descritivo -> função de tronco.
TRONCOS_CLASSICOS = {
    "cnn_rainbow": cnn_rainbow,
    "cnn_alphazero": cnn_alphazero,
    "cnn_vgg": cnn_vgg,
    "cnn_vgg_dropout": cnn_vgg_dropout,
    "cnn_vgg_sem_pool": cnn_vgg_sem_pool,
}

#: Apelidos numéricos do repositório antigo. Apontam para as definições **dos
#: notebooks**, que são as que realmente rodaram (ver o cabeçalho do módulo).
APELIDOS_LEGADOS = {
    "cnn1": "cnn_rainbow",
    "cnn2": "cnn_alphazero",
    "cnn3": "cnn_vgg",
    "cnn4": "cnn_vgg_dropout",
}


# --- snakeai/nets/heads.py ---
"""Cabeças de rede — dueling, noisy e distribucional (C51).

São os componentes que separam um DQN simples de um Rainbow. Ficam separados dos troncos
de propósito: qualquer cabeça encaixa em qualquer tronco, e é isso que permite perguntar
"quanto o dueling vale?" com o resto do experimento congelado.

Todas foram reescritas para Keras 3. A `NoisyDense` do repositório antigo herdava de
`Dense` e mexia nos internals dela (`self.kernel`, `build` reimplementado), o que quebra
em qualquer versão moderna; esta é uma `Layer` própria, com `add_weight` e `keras.random`.
"""


import contextlib
import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

__all__ = ["NoisyDense", "CentraNaMedia", "dueling_head", "distributional_head", "q_de_distribuicao",
           "ruido_ligado"]


@keras.saving.register_keras_serializable(package="snakeai")
class NoisyDense(layers.Layer):
    """Camada densa com ruído fatorado nos pesos (Fortunato et al., 2017).

    Substitui a exploração ε-greedy por ruído aprendido: a rede começa barulhenta e vai
    reduzindo o próprio σ conforme fica confiante. A vantagem sobre o ε-greedy é que a
    exploração passa a ser **dependente do estado** — o agente explora onde ainda não sabe,
    não uniformemente.

    Uma decisão importante: **o ruído é desligado quando `training=False`**. O protocolo de
    avaliação do contrato é greedy e determinístico; se a rede sorteasse ruído durante o
    benchmark, o mesmo modelo daria números diferentes a cada execução e a comparação entre
    algoritmos perderia o sentido. Alguns trabalhos mantêm o ruído na avaliação — aqui não,
    e a escolha está registrada porque muda o número publicado.

    Só que **a coleta não é a avaliação**, e amarrar o ruído a `training` juntava as duas:
    a política de comportamento saía determinística, e um Rainbow com `eps=0` (porque "a
    exploração é responsabilidade da rede") passava o treino inteiro sem explorar nada. O
    atributo `ruido` desempata: `None` segue o `training`, `True` força ruído, `False`
    força determinismo. Use o gerenciador `ruido_ligado` — ele é para uso **eager**, na
    coleta; dentro de uma `tf.function` o valor vira constante no traçado.

    Parâmetros
    ----------
    units : int
        Dimensão de saída.
    sigma0 : float
        Escala inicial do ruído, dividida por `sqrt(entrada)`. 0,5 é o valor do paper.
    """

    #: Sorteia um ruído **por linha do lote** em vez de um por passada.
    #:
    #: O paper (Fortunato et al.) sorteia `ε` uma vez por passada e o compartilha com o
    #: lote inteiro — e isso é fiel quando existe **um** ambiente, que é o caso dele. Com
    #: `num_envs=64` a mesma passada decide a ação dos 64, então os 64 seguem a *mesma*
    #: política perturbada: são 64 cópias correlacionadas, não 64 exploradores. Medido, o
    #: tamanho efetivo cai para ~8 de 64.
    #:
    #: As implementações distribuídas do próprio lineage (Ape-X, R2D2) dão a cada ator a
    #: sua rede e o seu ruído — ou seja, um sorteio por ambiente é a **vetorização
    #: correta** do paper, não um desvio dele. Fica desligado por padrão mesmo assim,
    #: porque muda a política de comportamento e isso é decisão declarada.
    por_amostra: bool = False

    def __init__(self, units, activation=None, sigma0=0.5, seed=None, **kw):
        super().__init__(**kw)
        self.units = int(units)
        self.activation = keras.activations.get(activation)
        self.sigma0 = float(sigma0)
        self.seed = seed
        self.seed_generator = keras.random.SeedGenerator(seed)
        #: `None` = segue `training`; `True`/`False` forçam. Ver `ruido_ligado`.
        self.ruido = None

    def build(self, input_shape):
        entrada = int(input_shape[-1])
        limite = 1.0 / (entrada ** 0.5)
        sigma_ini = self.sigma0 / (entrada ** 0.5)

        self.w_mu = self.add_weight(
            shape=(entrada, self.units), name="w_mu",
            initializer=keras.initializers.RandomUniform(-limite, limite))
        self.w_sigma = self.add_weight(
            shape=(entrada, self.units), name="w_sigma",
            initializer=keras.initializers.Constant(sigma_ini))
        self.b_mu = self.add_weight(
            shape=(self.units,), name="b_mu",
            initializer=keras.initializers.RandomUniform(-limite, limite))
        self.b_sigma = self.add_weight(
            shape=(self.units,), name="b_sigma",
            initializer=keras.initializers.Constant(sigma_ini))
        self._entrada = entrada

    @staticmethod
    def _f(x):
        """`sign(x) * sqrt(|x|)` — a transformação que fatora o ruído no paper."""
        return ops.sign(x) * ops.sqrt(ops.abs(x))

    def call(self, inputs, training=False):
        ativo = self.ruido if self.ruido is not None else training

        if ativo and self.por_amostra:
            # Um ε por linha do lote. Materializar 512 matrizes de peso seria proibitivo,
            # mas a fatoração dispensa: com w[b] = w_mu + w_sigma·(ε_in[b] ⊗ ε_out[b]),
            #     (x[b] @ w[b])_j = (x[b] @ w_mu)_j + ε_out[b,j]·((x[b]·ε_in[b]) @ w_sigma)_j
            # e o viés segue o mesmo ε_out. Custa dois matmuls em vez de um.
            lote = ops.shape(inputs)[0]
            eps_in = self._f(keras.random.normal((lote, self._entrada),
                                                 seed=self.seed_generator))
            eps_out = self._f(keras.random.normal((lote, self.units),
                                                 seed=self.seed_generator))
            y = ops.matmul(inputs, self.w_mu) + self.b_mu
            y = y + eps_out * (ops.matmul(inputs * eps_in, self.w_sigma) + self.b_sigma)
            return self.activation(y) if self.activation is not None else y

        if ativo:
            eps_in = self._f(keras.random.normal((self._entrada,),
                                                 seed=self.seed_generator))
            eps_out = self._f(keras.random.normal((self.units,),
                                                  seed=self.seed_generator))
            w = self.w_mu + self.w_sigma * ops.outer(eps_in, eps_out)
            b = self.b_mu + self.b_sigma * eps_out
        else:
            w, b = self.w_mu, self.b_mu

        y = ops.matmul(inputs, w) + b
        return self.activation(y) if self.activation is not None else y

    def compute_output_shape(self, input_shape):
        return (*input_shape[:-1], self.units)

    def ruido_medio(self):
        """σ médio dos pesos — cai conforme a rede fica confiante. Bom de registrar."""
        return float(ops.convert_to_numpy(ops.mean(ops.abs(self.w_sigma))))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "units": self.units,
            "activation": keras.activations.serialize(self.activation),
            "sigma0": self.sigma0,
            "seed": self.seed,
        })
        return cfg


@keras.saving.register_keras_serializable(package="snakeai")
class CentraNaMedia(layers.Layer):
    """`x − média(x)` ao longo de um eixo. Camada registrada, **não** `Lambda`.

    O `Lambda` com uma função anônima não sobrevive a `save`/`load` no Keras 3: o
    desserializador recusa recarregar um lambda Python por risco de execução arbitrária, e
    exige `safe_mode=False`. `snakeai/nets/muzero.py` já tinha aprendido isso; a cabeça
    dueling e a do C51 não.

    O custo do descuido não é teórico. `AgentBase.avaliar_melhor()` recarrega o checkpoint
    `best` **no fim do treino**, então o `ValueError` chegava depois do orçamento inteiro
    gasto — 8.931 s de GPU numa execução do Rainbow, perdida na última linha. E só o
    Rainbow batia nisto: é o único agente com `dueling=True` **e** `n_atoms>0`, os dois
    caminhos que usavam `Lambda`.
    """

    def __init__(self, eixo=-1, **kw):
        super().__init__(**kw)
        self.eixo = int(eixo)

    def call(self, x):
        return x - ops.mean(x, axis=self.eixo, keepdims=True)

    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        cfg = super().get_config()
        cfg["eixo"] = self.eixo
        return cfg


def _densa(tipo, unidades, ativacao=None, nome=None):
    if tipo == "noisy":
        return NoisyDense(unidades, activation=ativacao, name=nome)
    return layers.Dense(unidades, activation=ativacao, name=nome)


def dueling_head(x, n_actions, largura=256, densa="dense", nome="dueling"):
    """`Q(s,a) = V(s) + A(s,a) − média_a A(s,a)`.

    A subtração da média é o que torna a decomposição identificável: sem ela, somar uma
    constante a `V` e subtraí-la de `A` daria o mesmo `Q`, e as duas correntes poderiam
    derivar sem que a perda percebesse.

    O original usava a **média**; o paper também oferece o **máximo**. Ficamos na média,
    que é o padrão do Rainbow.
    """
    a = _densa(densa, largura, "relu", f"{nome}_a_h")(x)
    a = _densa(densa, n_actions, None, f"{nome}_a")(a)
    v = _densa(densa, largura, "relu", f"{nome}_v_h")(x)
    v = _densa(densa, 1, None, f"{nome}_v")(v)

    a_centrada = CentraNaMedia(eixo=-1, name=f"{nome}_center")(a)
    return layers.Add(name=f"{nome}_q")([v, a_centrada])


def distributional_head(x, n_actions, n_atoms=51, largura=256, densa="dense",
                        dueling=False, nome="c51"):
    """Cabeça categórica do C51: distribuição sobre `n_atoms` valores por ação.

    Em vez de estimar `Q(s,a)` — a média do retorno — o C51 estima a distribuição inteira.
    O ganho não é só estatístico: aprender uma distribuição dá um sinal de treino mais
    rico por transição, e é a peça que mais contribui no Rainbow.

    Devolve **logits** de forma `(lote, n_ações, n_átomos)`. A softmax e a projeção sobre
    o suporte ficam no agente, onde o `v_min`/`v_max` é conhecido.
    """
    if dueling:
        a = _densa(densa, largura, "relu", f"{nome}_a_h")(x)
        a = _densa(densa, n_actions * n_atoms, None, f"{nome}_a")(a)
        a = layers.Reshape((n_actions, n_atoms), name=f"{nome}_a_r")(a)

        v = _densa(densa, largura, "relu", f"{nome}_v_h")(x)
        v = _densa(densa, n_atoms, None, f"{nome}_v")(v)
        v = layers.Reshape((1, n_atoms), name=f"{nome}_v_r")(v)

        a_centrada = CentraNaMedia(eixo=1, name=f"{nome}_center")(a)
        return layers.Add(name=f"{nome}_logits")([v, a_centrada])

    h = _densa(densa, largura, "relu", f"{nome}_h")(x)
    h = _densa(densa, n_actions * n_atoms, None, f"{nome}_d")(h)
    return layers.Reshape((n_actions, n_atoms), name=f"{nome}_logits")(h)


def q_de_distribuicao(logits, suporte):
    """Colapsa a distribuição categórica em `Q(s,a)` — só para escolher a ação.

    `logits`: `(lote, n_ações, n_átomos)`. `suporte`: `(n_átomos,)`.
    """
    p = ops.softmax(logits, axis=-1)
    return ops.sum(p * ops.reshape(suporte, (1, 1, -1)), axis=-1)


def suporte_c51(v_min=-10.0, v_max=10.0, n_atoms=51):
    """Os `n_atoms` valores igualmente espaçados em `[v_min, v_max]`.

    Com recompensa `+1`/`−1` e γ = 0,995, o retorno de um episódio de Snake fica bem
    dentro de `[−2, 60]` — a faixa padrão de `[−10, 10]` do Atari é estreita demais aqui.
    O agente escolhe a sua; este é só o utilitário.
    """
    import numpy as np

    return np.linspace(v_min, v_max, n_atoms, dtype=np.float32)


@contextlib.contextmanager
def ruido_ligado(modelo, ativo=True, por_amostra=False):
    """Liga o ruído das `NoisyDense` de `modelo` dentro do bloco, e devolve como estava.

    Existe porque **coletar não é avaliar**. `NoisyDense.call` amarra o ruído a
    `training`, e ligar `training=True` na coleta traria junto tudo o que esse sinalizador
    significa nos outros troncos — o `Dropout` do `cnn_classic`, por exemplo. Este
    gerenciador mexe só nas camadas ruidosas.

    Uso **eager**, na escolha da ação. Dentro de uma `tf.function` o atributo é lido no
    traçado e vira constante no grafo, que não é o que se quer.
    """
    camadas = [c for c in _camadas(modelo) if isinstance(c, NoisyDense)]
    antes = [(c.ruido, c.por_amostra) for c in camadas]
    for c in camadas:
        c.ruido = ativo
        c.por_amostra = bool(por_amostra)
    try:
        yield camadas
    finally:
        for c, (r, pa) in zip(camadas, antes):
            c.ruido, c.por_amostra = r, pa


def _camadas(modelo):
    """Todas as camadas de `modelo`, inclusive as aninhadas em submodelos."""
    vistas, pilha = [], list(getattr(modelo, "layers", []))
    while pilha:
        c = pilha.pop()
        vistas.append(c)
        pilha.extend(getattr(c, "layers", []))
    return vistas


# --- snakeai/nets/registry.py ---
"""O registro de redes — qualquer tronco, para qualquer algoritmo, por string.

É isto que transforma "qual arquitetura é melhor?" numa ablação medida: o agente recebe
`net="cnn_vgg"` ou `net="resnet_small"` e o resto do experimento não muda. Sem isso, cada
comparação de rede viraria um notebook novo, que é como o repositório antigo acabou com
seis DQNs que ninguém conseguia comparar.

Todo modelo construído aqui obedece ao contrato: entrada `(B, B, 5)` egocêntrica,
saída de política com 3 ações relativas.
"""


import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers


__all__ = [
    "TRONCOS",
    "listar_troncos",
    "build_backbone",
    "build_actor_critic",
    "build_actor_critic_populacao",
    "build_option_actor_critic",
    "build_q_network",
    "build_policy_q",
    "resumo",
]

TRONCOS = {**TRONCOS_RESIDUAIS, **TRONCOS_CLASSICOS}

#: A cabeça densa do repositório antigo tinha **3136** unidades. O número não é arbitrário
#: — é exatamente o achatamento da `cnn_rainbow` num tabuleiro 10×10 (7×7×64), o mesmo
#: valor do DQN do Atari por coincidência de kernels. Só que uma camada de 3136 sobre uma
#: entrada de 3136 são **9,8 milhões de parâmetros**, e ela era replicada nas duas
#: correntes do dueling. Sobre a `cnn_vgg`, que entrega 64 features, a mesma camada liga
#: 64 entradas a 3136 unidades: quase toda a capacidade do modelo depois de o tronco já
#: ter descartado a informação espacial.
#: O padrão aqui é 256. `LARGURA_DENSA_LEGADA` continua disponível para reproduzir o
#: original quando a fidelidade importar mais que o bom senso.
LARGURA_DENSA_LEGADA = 3136
LARGURA_DENSA_PADRAO = 256


def listar_troncos():
    """Nomes aceitos, incluindo os apelidos numéricos do repositório antigo."""
    return sorted(TRONCOS) + sorted(APELIDOS_LEGADOS)


def _resolve(nome):
    if nome in TRONCOS:
        return TRONCOS[nome], nome
    if nome in APELIDOS_LEGADOS:
        canonico = APELIDOS_LEGADOS[nome]
        return TRONCOS[canonico], canonico
    raise ValueError(
        f"tronco desconhecido: {nome!r}. Disponíveis: {listar_troncos()}"
    )


def build_backbone(entrada, net="resnet_small"):
    """Aplica o tronco `net` a um tensor de entrada. Devolve `(saida, nome_canonico)`."""
    fn, canonico = _resolve(net)
    return fn(entrada), canonico


def _entrada(board_size, canais=N_CHANNELS):
    """A entrada do tronco. `canais` só sai de 5 numa ablação declarada.

    O contrato fixa 5 canais, e mudar isso muda a **entrada da rede** — nenhuma curva de 5
    canais é comparável a uma de 6. O parâmetro existe para `VecSnake(canal_fome=True)`,
    que é uma ablação `comparable=False`, e não para configuração casual.
    """
    return keras.Input(shape=(board_size, board_size, canais), name="board")


def _e_espacial(t):
    """True se o tronco devolveu um mapa `(H, W, C)` em vez de um vetor achatado."""
    return len(t.shape) == 4


def build_actor_critic(board_size=10, net="resnet_small", largura_densa=None,
                       n_actions=N_ACTIONS, nome=None, canais=N_CHANNELS):
    """Modelo de duas saídas `[logits, valor]` — o que PPO, A2C e ACER consomem.

    Em troncos que preservam a estrutura espacial (as ResNets), as cabeças são
    convoluções 1×1 seguidas de achatamento, como no AlphaZero: mais barato e mais
    informativo que jogar um `Dense` gigante em cima de um mapa achatado. Em troncos
    clássicos, que já achatam, usa-se a cabeça densa mesmo.

    O `Dense` final da política nasce com `kernel_initializer` de ganho pequeno: no início
    do treino a política precisa ser quase uniforme, senão o PPO gasta as primeiras
    iterações desfazendo uma preferência aleatória.
    """
    inp = _entrada(board_size, canais)
    x, canonico = build_backbone(inp, net)
    largura = LARGURA_DENSA_PADRAO if largura_densa is None else int(largura_densa)

    if _e_espacial(x):
        p = layers.Conv2D(4, 1, use_bias=False, name="pi_c")(x)
        p = layers.GroupNormalization(groups=2, name="pi_n")(p)
        p = layers.Activation("relu", name="pi_a")(p)
        p = layers.Flatten(name="pi_f")(p)

        v = layers.Conv2D(2, 1, use_bias=False, name="v_c")(x)
        v = layers.GroupNormalization(groups=2, name="v_n")(v)
        v = layers.Activation("relu", name="v_a")(v)
        v = layers.Flatten(name="v_f")(v)
        v = layers.Dense(largura, activation="relu", name="v_d")(v)
    else:
        p = layers.Dense(largura, activation="relu", name="pi_d")(x)
        v = layers.Dense(largura, activation="relu", name="v_d")(x)

    logits = layers.Dense(
        n_actions, name="logits",
        kernel_initializer=keras.initializers.Orthogonal(gain=0.01),
        bias_initializer="zeros",
    )(p)
    valor = layers.Dense(
        1, name="value",
        kernel_initializer=keras.initializers.Orthogonal(gain=1.0),
        bias_initializer="zeros",
    )(v)

    return keras.Model(inp, [logits, valor], name=nome or f"ac_{canonico}")


def build_actor_critic_populacao(board_size=10, net="resnet_small", n_politicas=3,
                                 largura_densa=None, n_actions=N_ACTIONS, nome=None,
                                 canais=N_CHANNELS):
    """`N` pares (política, valor) sobre um tronco **compartilhado** — o que o LBC consome.

    Saída `[logits, valor]` com formas `(lote, N, ações)` e `(lote, N)`: a população
    inteira num forward só. É essa forma que permite ao comportamento do LBC ser uma
    mistura sobre as `N` políticas sem `N` passadas pela rede.

    O tronco compartilhado é um **desvio declarado** do paper, que trata cada política como
    um modelo inteiro e independente (Assumption 1). A razão é o orçamento: o contrato deste
    repositório dá 5 M passos de ambiente a todos os algoritmos, e três ResNets separadas
    triplicariam o custo por passo — o LBC entraria na arena competindo com o mesmo
    orçamento de ambiente e três vezes mais computação, que é a comparação que este
    repositório existe para não fazer. Ver `docs/LBC.md`.

    O que se perde é diversidade de **representação**: as três políticas veem as mesmas
    features. O que se mantém é diversidade de **objetivo** — cada cabeça é treinada com o
    seu próprio γ e o seu próprio alvo V-trace — e é ela que constrói o espaço de
    comportamento não-degenerado do §4.1.
    """
    if int(n_politicas) < 1:
        raise ValueError("a população precisa de pelo menos uma política")
    n_politicas = int(n_politicas)

    inp = _entrada(board_size, canais)
    x, canonico = build_backbone(inp, net)
    largura = LARGURA_DENSA_PADRAO if largura_densa is None else int(largura_densa)
    espacial = _e_espacial(x)

    logits_por_politica, valores_por_politica = [], []
    for i in range(n_politicas):
        if espacial:
            p = layers.Conv2D(4, 1, use_bias=False, name=f"pi{i}_c")(x)
            p = layers.GroupNormalization(groups=2, name=f"pi{i}_n")(p)
            p = layers.Activation("relu", name=f"pi{i}_a")(p)
            p = layers.Flatten(name=f"pi{i}_f")(p)

            v = layers.Conv2D(2, 1, use_bias=False, name=f"v{i}_c")(x)
            v = layers.GroupNormalization(groups=2, name=f"v{i}_n")(v)
            v = layers.Activation("relu", name=f"v{i}_a")(v)
            v = layers.Flatten(name=f"v{i}_f")(v)
            v = layers.Dense(largura, activation="relu", name=f"v{i}_d")(v)
        else:
            p = layers.Dense(largura, activation="relu", name=f"pi{i}_d")(x)
            v = layers.Dense(largura, activation="relu", name=f"v{i}_d")(x)

        li = layers.Dense(
            n_actions, name=f"logits_{i}",
            kernel_initializer=keras.initializers.Orthogonal(gain=0.01),
            bias_initializer="zeros",
        )(p)
        vi = layers.Dense(
            1, name=f"value_{i}",
            kernel_initializer=keras.initializers.Orthogonal(gain=1.0),
            bias_initializer="zeros",
        )(v)
        logits_por_politica.append(
            layers.Reshape((1, n_actions), name=f"logits_{i}_r")(li))
        valores_por_politica.append(vi)

    # `Concatenate` recusa uma entrada só — e uma população de tamanho 1 é justamente a
    # ablação "reduzir H" da Fig. 5 do paper, então este caminho tem que existir.
    if n_politicas == 1:
        logits = logits_por_politica[0]
        valor = valores_por_politica[0]
    else:
        logits = layers.Concatenate(axis=1, name="logits")(logits_por_politica)
        valor = layers.Concatenate(axis=-1, name="value")(valores_por_politica)

    return keras.Model(inp, [logits, valor],
                       name=nome or f"lbc{n_politicas}_{canonico}")


def build_option_actor_critic(board_size=10, net="resnet_small", n_opcoes=4,
                              largura_densa=None, n_actions=N_ACTIONS, nome=None,
                              canais=N_CHANNELS):
    """Política com opções — o que o SOAP consome. Três saídas:

    * `logits_a` `(lote, Z, ações)` — a sub-política `π_θ(a|s,z)`, uma por opção;
    * `logits_z` `(lote, Z, ações, Z)` — a transição `π_ψ(z'|s,a,z)`;
    * `valor` `(lote, Z)` — o crítico condicionado à opção corrente.

    A transição depende de `(s, a, z)`, e não só de `s`: é a fatoração que o paper do SOAP
    propõe contra a do Option-Critic, e é ela que permite a uma opção **persistir** por
    conta própria em vez de ser re-sorteada a cada passo. O custo é um tensor de saída
    `Z × A × Z` — com `Z = 4` e `A = 3`, 48 números por estado, que é barato.

    Os logits da sub-política nascem com ganho pequeno, como no `build_actor_critic`: no
    começo do treino toda opção precisa ser quase uniforme. Os da transição também, e por
    um motivo mais forte — uma preferência inicial de troca de opção é um viés que o agente
    gasta as primeiras iterações desfazendo, e enquanto isso a crença `ζ` já colapsou.
    """
    if int(n_opcoes) < 1:
        raise ValueError("é preciso pelo menos uma opção")
    n_opcoes = int(n_opcoes)

    inp = _entrada(board_size, canais)
    x, canonico = build_backbone(inp, net)
    largura = LARGURA_DENSA_PADRAO if largura_densa is None else int(largura_densa)

    def projeta(nome_curto, filtros):
        if _e_espacial(x):
            h = layers.Conv2D(filtros, 1, use_bias=False, name=f"{nome_curto}_c")(x)
            h = layers.GroupNormalization(groups=2, name=f"{nome_curto}_n")(h)
            h = layers.Activation("relu", name=f"{nome_curto}_a")(h)
            h = layers.Flatten(name=f"{nome_curto}_f")(h)
            if nome_curto != "pi":
                h = layers.Dense(largura, activation="relu", name=f"{nome_curto}_d")(h)
            return h
        return layers.Dense(largura, activation="relu", name=f"{nome_curto}_d")(x)

    p = projeta("pi", 4)
    q = projeta("psi", 4)
    v = projeta("v", 2)

    logits_a = layers.Dense(
        n_opcoes * n_actions, name="logits_a_d",
        kernel_initializer=keras.initializers.Orthogonal(gain=0.01),
        bias_initializer="zeros",
    )(p)
    logits_a = layers.Reshape((n_opcoes, n_actions), name="logits_a")(logits_a)

    logits_z = layers.Dense(
        n_opcoes * n_actions * n_opcoes, name="logits_z_d",
        kernel_initializer=keras.initializers.Orthogonal(gain=0.01),
        bias_initializer="zeros",
    )(q)
    logits_z = layers.Reshape((n_opcoes, n_actions, n_opcoes),
                              name="logits_z")(logits_z)

    valor = layers.Dense(
        n_opcoes, name="value",
        kernel_initializer=keras.initializers.Orthogonal(gain=1.0),
        bias_initializer="zeros",
    )(v)

    return keras.Model(inp, [logits_a, logits_z, valor],
                       name=nome or f"soap{n_opcoes}_{canonico}")


def build_q_network(board_size=10, net="cnn_rainbow", largura_densa=None,
                    n_actions=N_ACTIONS, dueling=False, noisy=False, n_atoms=0,
                    nome=None, canais=N_CHANNELS):
    """A família DQN inteira num construtor só.

    `dueling`, `noisy` e `n_atoms` são os eixos que separam o DQN base do Rainbow — e são
    ortogonais de propósito, para que cada um possa ser medido isolado. Essa é a resposta
    aos seis notebooks quase idênticos do repositório antigo: uma função, seis chamadas.

    Saída
    -----
    `(lote, n_ações)` no modo normal; `(lote, n_ações, n_atoms)` de **logits** quando
    `n_atoms > 0` (C51).
    """
    inp = _entrada(board_size, canais)
    x, canonico = build_backbone(inp, net)
    largura = LARGURA_DENSA_PADRAO if largura_densa is None else int(largura_densa)
    densa = "noisy" if noisy else "dense"

    if _e_espacial(x):
        x = layers.Conv2D(8, 1, use_bias=False, name="q_c")(x)
        x = layers.GroupNormalization(groups=2, name="q_n")(x)
        x = layers.Activation("relu", name="q_a")(x)
        x = layers.Flatten(name="q_f")(x)

    if n_atoms:
        saida = distributional_head(x, n_actions, n_atoms=n_atoms, largura=largura,
                                    densa=densa, dueling=dueling)
    elif dueling:
        saida = dueling_head(x, n_actions, largura=largura, densa=densa)
    else:
        h = _densa(densa, largura, "relu", "q_d")(x)
        saida = _densa(densa, n_actions, None, "q")(h)

    partes = [p for p, on in (("dueling", dueling), ("noisy", noisy),
                              (f"c51x{n_atoms}", bool(n_atoms))) if on]
    sufixo = ("_" + "_".join(partes)) if partes else ""
    return keras.Model(inp, saida, name=nome or f"q_{canonico}{sufixo}")


def build_policy_q(board_size=10, net="resnet_small", largura_densa=None,
                   n_actions=N_ACTIONS, nome=None, canais=N_CHANNELS):
    """Modelo de duas saídas `[logits, Q(s,·)]` — o que o ACER consome.

    Diferente do actor-critic comum: aqui o crítico devolve **um valor por ação**, não um
    escalar. É disso que o Retrace precisa, e `V(s) = Σ_a π(a|s) Q(s,a)` sai de graça —
    sem uma terceira cabeça e sem inconsistência entre V e Q, que é uma fonte clássica de
    bug silencioso em ACER.
    """
    inp = _entrada(board_size, canais)
    x, canonico = build_backbone(inp, net)
    largura = LARGURA_DENSA_PADRAO if largura_densa is None else int(largura_densa)

    if _e_espacial(x):
        p = layers.Conv2D(4, 1, use_bias=False, name="pi_c")(x)
        p = layers.GroupNormalization(groups=2, name="pi_n")(p)
        p = layers.Activation("relu", name="pi_a")(p)
        p = layers.Flatten(name="pi_f")(p)

        q = layers.Conv2D(8, 1, use_bias=False, name="q_c")(x)
        q = layers.GroupNormalization(groups=2, name="q_n")(q)
        q = layers.Activation("relu", name="q_a")(q)
        q = layers.Flatten(name="q_f")(q)
        q = layers.Dense(largura, activation="relu", name="q_d")(q)
    else:
        p = layers.Dense(largura, activation="relu", name="pi_d")(x)
        q = layers.Dense(largura, activation="relu", name="q_d")(x)

    logits = layers.Dense(
        n_actions, name="logits",
        kernel_initializer=keras.initializers.Orthogonal(gain=0.01),
        bias_initializer="zeros",
    )(p)
    q_saida = layers.Dense(n_actions, name="q", bias_initializer="zeros")(q)
    return keras.Model(inp, [logits, q_saida], name=nome or f"acer_{canonico}")


def resumo(board_size=10, largura_densa=None):
    """Tabela comparativa dos troncos: parâmetros e formato de saída.

    Usada no notebook de ablação e no README. Revela, de graça, quais troncos colapsam o
    tabuleiro — a coluna `saída do tronco` mostra `1×1` para os que usam pooling.
    """
    linhas = []
    for nome in sorted(TRONCOS):
        inp = _entrada(board_size)
        saida, _ = build_backbone(inp, nome)
        forma = tuple(saida.shape[1:])
        modelo = build_actor_critic(board_size, nome, largura_densa)
        tronco = keras.Model(inp, saida)
        linhas.append({
            "tronco": nome,
            "saida_tronco": "×".join(str(d) for d in forma),
            "espacial": _e_espacial(saida),
            "params_tronco": tronco.count_params(),
            "params_actor_critic": modelo.count_params(),
        })
    return linhas


# --- snakeai/agents/base.py ---
"""Andaime comum a todos os agentes.

O que fica aqui é o que **precisa** ser idêntico entre algoritmos para que a comparação
valha: a cadência da avaliação, o formato do registro, o critério de "melhor checkpoint",
e os agendamentos lineares. O que varia — como o agente aprende — fica em cada módulo.

Foi essa separação que faltou no repositório antigo: cada notebook tinha o próprio laço de
treino, a própria noção de época e o próprio jeito de avaliar, e por isso as curvas nunca
puderam ser sobrepostas.
"""


import contextlib
import json
import os
from collections import deque
from dataclasses import asdict, dataclass, field

import numpy as np


__all__ = ["BaseConfig", "AgentBase"]


@dataclass
class BaseConfig:
    """Os campos que todo agente do benchmark tem. Cada algoritmo estende com os seus."""

    board_size: int = CONTRATO["board_size"]
    net: str = "resnet_small"
    seed: int = 0

    #: Orçamento oficial. O contrato exige o **mesmo** valor para todos os algoritmos.
    total_steps: int = 5_000_000

    #: Avaliação periódica durante o treino, no protocolo oficial.
    eval_every_steps: int = 250_000
    eval_episodes: int = CONTRATO["eval_episodes"]
    eval_envs: int = 250

    ckpt_dir: str = "checkpoints"
    runs_dir: str = "runs"
    log_every_steps: int = 50_000

    #: Artefatos gerados no fim do treino. O GIF custa segundos e responde a pergunta que
    #: nenhuma curva responde: *como* o agente joga.
    salvar_grafico: bool = True
    #: Sufixo acrescentado à variante. Serve para que uma execução que muda
    #: hiperparâmetros — e portanto **compete**, mas não é a mesma coisa — não divida a
    #: identidade `(algo, variant, seed)` com a do padrão. `load_all` agrupa por essa
    #: tripla, então identidade repetida vira uma curva só, com as duas misturadas.
    sufixo_variante: str = ""

    salvar_gif: bool = True
    gif_seeds: tuple = (7, 21, 42)

    #: Marque `False` numa execução que muda o ambiente ou o protocolo de propósito — uma
    #: ablação. Ela continua sendo gravada e plotada, mas **fora da arena**, e `caveat`
    #: passa a ser obrigatório: uma curva incomparável sem o motivo escrito é pior que
    #: nenhuma curva, porque alguém vai compará-la mesmo assim.
    comparable: bool = True
    caveat: str = ""

    def __post_init__(self):
        if not self.comparable and not self.caveat:
            raise ValueError(
                "comparable=False exige `caveat` dizendo por que esta execução não "
                "compete. Sem isso a curva vira uma armadilha para quem ler depois.")
        if self.board_size != CONTRATO["board_size"]:
            raise ValueError(
                f"board_size={self.board_size} viola o contrato "
                f"({CONTRATO['board_size']}). Mude o contrato conscientemente, "
                "não a execução."
            )


def proximo_multiplo(passo, cadencia):
    """O menor múltiplo de `cadencia` **estritamente acima** de `passo`.

    É a diferença entre uma grade absoluta e uma que reancora: `passo + cadencia` faz cada
    avaliação cair um bloco depois da anterior, e o desvio se acumula — na execução padrão
    do PPO a última avaliação aconteceu 513 mil passos além do ponto nominal, 10% do
    orçamento. Como cada algoritmo avança em blocos de tamanho diferente (8.192 no A2C,
    49.152 no PPO padrão), as grades divergem entre si e a coluna `passos até 40`, que o
    contrato lê **sem interpolação**, passa a comparar medições feitas em lugares
    diferentes. Ver `docs/REVISAO_ALGORITMOS.md` §1.7.
    """
    return (int(passo) // int(cadencia) + 1) * int(cadencia)


class AgentBase:
    """Laço de treino comum: agendamentos, avaliação, checkpoint e registro.

    A subclasse implementa `iterate()` — um passo de aprendizado, que devolve estatísticas
    do rollout — e o resto vem de graça, igual para todo mundo.
    """

    algo = "base"

    #: Tamanho da janela da média móvel do treino, **em episódios**. 500 é da mesma ordem
    #: dos 1.000 da avaliação oficial: grande o bastante para o número não pular com um
    #: episódio de sorte, pequeno o bastante para acompanhar o agente melhorando.
    JANELA_EPISODIOS = 500

    #: Sufixo da variante para execuções fora do contrato de observação. A identidade de
    #: uma execução é `(algo, variant, seed)` — é por ela que `load_all` agrupa, não pelo
    #: caminho. Sem o sufixo, uma execução com `canal_fome=True` fica com a **mesma
    #: identidade** da execução de contrato da mesma rede e semente: hoje elas só não se
    #: misturam porque `comparable=False` as tira da arena, o que é proteção por acidente,
    #: não por construção. Ver `docs/CANAL_DE_FOME.md`.
    SUFIXO_FOME = "_fome"

    def __init__(self, cfg, variant="default"):
        self.cfg = cfg
        if getattr(cfg, "canal_fome", False):
            variant = self._com_sufixo(variant, self.SUFIXO_FOME)
        self.variant = self._com_sufixo(variant, getattr(cfg, "sufixo_variante", ""))
        self.model = None
        self.global_step = 0
        self.episodes = 0
        self.iteration = 0
        self.history = []
        self.evals = []
        self.baseline = None
        self.melhor = -np.inf
        self._proximo_eval = 0
        self._proximo_log = 0
        self._atualizacoes = 0
        #: Janela de episódios recentes para a média móvel do treino. Sem ela, o log
        #: imprime a média dos episódios que por acaso terminaram **naquela** iteração —
        #: uma amostra de tamanho 0 a 3. É o que produzia a sequência
        #: `2,50 · 10,00 · — · — · 2,00 · 11,00`, que parece instabilidade do algoritmo e
        #: é só tamanho de amostra. O `—` é literalmente "nenhum episódio acabou agora".
        #:
        #: A janela é medida em **episódios**, não em iterações, e a diferença não é
        #: cosmética. Uma iteração de PPO são 512 × 96 = 49.152 passos e ~200 episódios;
        #: uma de DQN são ~1.000 passos e 2 ou 3 episódios. Um limite fixo de iterações
        #: cobriria a execução inteira num caso e alguns segundos no outro — e no primeiro
        #: a "média móvel" viraria **média acumulada**, arrastada para baixo pelos
        #: episódios ruins do começo para sempre.
        self._janela = deque()
        #: Totais acumulados desde o início e no instante do log anterior. A janela móvel
        #: descarta pela esquerda, então a diferença entre dois logs **não** dá para ser
        #: reconstruída dela — e é essa diferença que responde "o que aconteceu nos
        #: últimos N episódios", que a média móvel de 500 esconde por construção. Numa
        #: iteração que produz ~80 episódios por log, a móvel arrasta 6 logs de história:
        #: uma degradação em curso aparece nela achatada e com atraso.
        self._acumulado = {"n": 0.0, **{c: 0.0 for c in self.CAMPOS_JANELA}}
        self._acumulado_no_log = dict(self._acumulado)
        self._legenda_impressa = False
        self._registrou_causas = False
        os.makedirs(cfg.ckpt_dir, exist_ok=True)

    # ----------------------------------------------------------- agendamentos
    #: Os campos somados na janela. Todos são **contagens ou somas** por bloco, nunca
    #: médias: só assim a média da janela pode ser ponderada pelo número de episódios de
    #: cada bloco, que é o que impede uma iteração com 2 episódios de pesar igual a uma
    #: com 200.
    CAMPOS_JANELA = ("score", "vitorias", "fome", "colisao", "passos")

    def _registra_episodios(self, media, n, **somas):
        """Guarda `n` episódios de score médio `media` e descarta o que saiu da janela.

        Descarta pela esquerda enquanto o que sobra ainda cobre `JANELA_EPISODIOS`, e
        nunca esvazia: com um algoritmo cuja iteração já produz mais episódios que a
        janela inteira, o certo é a janela ser aquela iteração — e não ficar vazia.
        """
        bloco = {"n": n, "score": media * n}
        bloco.update({c: float(somas.get(c, 0.0)) for c in self.CAMPOS_JANELA
                      if c != "score"})
        self._janela.append(bloco)
        self._acumulado["n"] += n
        for c in self.CAMPOS_JANELA:
            self._acumulado[c] += bloco[c]
        total = sum(b["n"] for b in self._janela)
        while len(self._janela) > 1 and total - self._janela[0]["n"] >= self.JANELA_EPISODIOS:
            total -= self._janela.popleft()["n"]

    def registra_fim(self, info):
        """Contabiliza os episódios que acabaram neste passo, **por causa**.

        Chamado de dentro do laço de coleta de cada agente, com o `info` que `VecSnake.step`
        devolve. É o único lugar onde a causa da morte existe: a memória de treino guarda
        `cont = 0` e nada mais, e depois do reset não há como saber se a cobra bateu ou
        passou fome.

        E a diferença entre as duas é justamente o que o score sozinho esconde. Um agente
        preso em 1,2 pontos pode estar batendo em tudo (não aprendeu a sobreviver) ou
        andando em círculo até morrer de fome (aprendeu a sobreviver e não a comer) — dois
        problemas opostos, com o mesmo número na curva. Foi exatamente essa ambiguidade que
        custou horas no diagnóstico do Dreamer.
        """
        n = int(info["scores"].size)
        if not n:
            return
        self._registrou_causas = True
        self._registra_episodios(
            float(info["scores"].mean()), n,
            vitorias=info["wins"],
            fome=info["starved"],
            # `deaths` conta colisão; vitória e fome não entram nele
            colisao=info["deaths"],
            passos=float(info["lengths"].sum()),
        )

    def media_movel(self):
        """Score médio dos últimos ~`JANELA_EPISODIOS` episódios, ponderado.

        `None` só quando nenhum episódio terminou ainda.
        """
        n = self.episodios_na_janela()
        return sum(b["score"] for b in self._janela) / n if n else None

    def episodios_na_janela(self):
        return sum(b["n"] for b in self._janela)

    def resumo_janela(self):
        """As frações que o score sozinho não conta, sobre a mesma janela de episódios.

        `{}` enquanto nenhum episódio terminou. As causas só aparecem se o agente chamar
        `registra_fim` — quem ainda não chama continua reportando só o score, sem inventar
        um zero que pareceria "nunca morre de fome".
        """
        n = self.episodios_na_janela()
        if not n:
            return {}
        soma = {c: sum(b.get(c, 0.0) for b in self._janela) for c in self.CAMPOS_JANELA}
        r = {"janela_episodios": n, "train_score_mean": soma["score"] / n}
        if self._registrou_causas:
            r.update({
                "win_rate": soma["vitorias"] / n,
                "frac_fome": soma["fome"] / n,
                "frac_colisao": soma["colisao"] / n,
                "passos_por_episodio": soma["passos"] / n,
            })
        return r

    def resumo_bloco(self):
        """As mesmas médias de `resumo_janela`, mas só sobre os episódios **novos**.

        "Novos" = terminados desde o log anterior. É o número que mostra a direção: a
        média móvel de 500 episódios responde "onde o agente está", o bloco responde "para
        onde está indo", e num treino que degrada as duas discordam por muito tempo antes
        de a móvel virar. Prefixadas com `bloco_` no registro para não colidirem com as
        chaves da janela, que são as que a arena e as curvas leem.

        `{}` quando nenhum episódio terminou desde o log anterior.
        """
        n = self._acumulado["n"] - self._acumulado_no_log["n"]
        if n <= 0:
            return {}
        d = {c: self._acumulado[c] - self._acumulado_no_log[c] for c in self.CAMPOS_JANELA}
        r = {"bloco_episodios": int(n), "bloco_train_score_mean": d["score"] / n}
        if self._registrou_causas:
            r.update({
                "bloco_win_rate": d["vitorias"] / n,
                "bloco_frac_fome": d["fome"] / n,
                "bloco_frac_colisao": d["colisao"] / n,
                "bloco_passos_por_episodio": d["passos"] / n,
            })
        return r

    def _marcar_bloco(self):
        """Fecha o bloco atual. Chamado depois de cada log, e só de lá."""
        self._acumulado_no_log = dict(self._acumulado)

    def frac(self):
        """Fração do orçamento já gasta, em [0, 1]. Base de todo agendamento linear."""
        return min(1.0, self.global_step / max(1, self.cfg.total_steps))

    def linear(self, inicio, fim):
        return inicio + self.frac() * (fim - inicio)

    # ----------------------------------------------------------- truncamento
    @staticmethod
    def desfaz_truncamento(info, prox_obs, prox_mask, done):
        """Devolve `(prox_obs, prox_mask, done)` com a morte por fome tratada como o que
        ela é: **truncamento**, não terminação.

        O `VecSnake` marca `done` para fome porque o episódio de fato acaba ali, mas
        exporta `trunc_idx`, `final_obs` e `final_mask` justamente para que o agente possa
        continuar o valor. Quem guarda a transição crua — DQN, Rainbow — tinha dois
        problemas de uma vez: gravava `done=1`, jogando fora o `γ·V(s')`, e gravava como
        `s'` a observação **do episódio seguinte**, porque o ambiente já resetou. O
        segundo é o pior: não havia como corrigir depois, o estado certo não estava no
        buffer.

        Aqui os dois somem: `s'` volta a ser o estado final verdadeiro e `done` volta a
        ser 0, que é exatamente o alvo de TD correto. Ver `docs/REVISAO_ALGORITMOS.md`
        §1.1.

        Não altera as entradas — devolve cópias.
        """
        ti = info.get("trunc_idx")
        if ti is None or len(ti) == 0:
            return prox_obs, prox_mask, done
        prox_obs = np.array(prox_obs, copy=True)
        prox_mask = np.array(prox_mask, copy=True)
        done = np.array(done, copy=True)
        prox_obs[ti] = info["final_obs"]
        prox_mask[ti] = info["final_mask"]
        done[ti] = 0.0
        return prox_obs, prox_mask, done

    @staticmethod
    def bootstrap_truncados(info, recompensas, valores_finais, gamma):
        """Soma `γ·V(s_final)` à recompensa dos episódios truncados por fome.

        A outra metade do tratamento de truncamento, para quem guarda **retornos** em vez
        de transições soltas. O `desfaz_truncamento` serve a quem tem um buffer de
        `(s, a, r, s')` e pode simplesmente devolver o `s'` verdadeiro; num rollout ou num
        segmento, o passo seguinte já pertence a outro episódio, e o valor do estado final
        precisa entrar **na recompensa** — que é como o PPO faz desde sempre.

        O `done` continua 1: a fronteira do episódio é real dentro do buffer, e é ela que
        impede o retorno de atravessar para o episódio seguinte. O que muda é que o
        retorno daquele passo deixa de valer −0,5 e passa a valer −0,5 + γ·V(s_final).

        Devolve uma cópia; sem truncamento, devolve a entrada intacta. Ver
        `docs/REVISAO_ALGORITMOS.md` §1.1.
        """
        ti = info.get("trunc_idx")
        if ti is None or len(ti) == 0:
            return recompensas
        saida = np.array(recompensas, copy=True, dtype=np.float32)
        saida[ti] += float(gamma) * np.asarray(valores_finais, dtype=np.float32)
        return saida

    # -------------------------------------------------------------- avaliação
    def politica(self):
        """A função de política que `snakeai.eval` consome. Sobrescreva se precisar."""
        return keras_policy(self.model)

    def politica_do_modelo(self, modelo):
        """A política de um modelo que veio **de fora** — um checkpoint, tipicamente.

        Existe porque `avaliar_melhor` trocava `self.model` e chamava `avaliar()`, o que
        só funciona para quem joga por `self.model`. O MuZero declara `model` como
        propriedade com setter vazio e o DreamerV3 joga por `self.ator` dentro de uma
        política recorrente: nos dois, a troca não fazia nada e a coluna `melhor` do
        registro virava uma segunda medição do modelo **final**, gravada com o passo do
        checkpoint `best`. Ver `docs/REVISAO_ALGORITMOS.md` §1.4.

        Quem não consegue jogar a partir de um `.keras` sozinho deve levantar
        `NotImplementedError` com o motivo — `avaliar_melhor` transforma isso numa coluna
        ausente e explicada, que é honesto, em vez de um número errado.
        """
        return keras_policy(modelo)

    def avaliar(self, episodes=None, safety=False, politica=None):
        """Roda o protocolo oficial. **Nunca** com exploração — é o número honesto."""
        stats, _ = evaluate(
            politica or self.politica(),
            board_size=self.cfg.board_size,
            episodes=episodes or self.cfg.eval_episodes,
            num_envs=self.cfg.eval_envs,
            greedy=CONTRATO["eval_greedy"],
            safety=safety,
            seed=CONTRATO["eval_seed"],
            # o ambiente de avaliação tem que ter os mesmos canais que o de treino
            canal_fome=getattr(self.env, "canal_fome", False),
        )
        return stats

    def piso(self):
        if self.baseline is None:
            self.baseline = random_baseline(
                self.cfg.board_size, self.cfg.eval_episodes, self.cfg.eval_envs,
                seed=CONTRATO["eval_seed"],
            )
        return self.baseline

    # ------------------------------------------------------------- checkpoint
    def _caminho(self, tag, ext):
        return os.path.join(self.cfg.ckpt_dir, f"{self.algo}_{tag}.{ext}")

    def modelos_extra(self):
        """Modelos/camadas além de `self.model` sem os quais a execução não se reproduz.

        O `salvar()` grava `self.model`, o que basta para quem joga com uma rede só. O
        DreamerV3 não é assim: `self.model` é o **ator**, e um ator sem o modelo do mundo
        não joga nada — a pasta da execução guardava um `.keras` que não reproduz o número
        da curva, e `retomar()` voltava com o RSSM aleatório enquanto o `global_step`
        continuava contando. Ver `docs/REVISAO_ALGORITMOS.md` §1.4.

        Devolve `{nome: modelo}`. Os pesos vão para um `.npz` ao lado do `.keras`, em vez
        de um `.keras` por peça: preserva a identidade dos objetos, e portanto as
        `tf.function` já traçadas que os capturaram.
        """
        return {}

    def _pesos_extra(self):
        return {f"{nome}/{i}": np.asarray(v)
                for nome, m in self.modelos_extra().items()
                for i, v in enumerate(m.weights)}

    def _salvar_extra(self, tag):
        pesos = self._pesos_extra()
        if pesos:
            np.savez(self._caminho(tag, "npz"), **pesos)

    def _carregar_extra(self, tag):
        """Devolve `True` se havia pesos extras para carregar."""
        caminho = self._caminho(tag, "npz")
        extras = self.modelos_extra()
        if not extras or not os.path.exists(caminho):
            return False
        with np.load(caminho) as dados:
            for nome, m in extras.items():
                for i, v in enumerate(m.weights):
                    chave = f"{nome}/{i}"
                    if chave in dados:
                        v.assign(dados[chave])
        return True

    def salvar(self, tag="last"):
        self.model.save(self._caminho(tag, "keras"))
        self._salvar_extra(tag)
        estado = {
            "global_step": self.global_step, "episodes": self.episodes,
            "iteration": self.iteration, "history": self.history,
            "evals": self.evals, "baseline": self.baseline, "melhor": self.melhor,
            "config": asdict(self.cfg), "variant": self.variant,
        }
        with open(self._caminho(tag, "json"), "w", encoding="utf-8") as f:
            json.dump(estado, f, ensure_ascii=False)

    def retomar(self, tag="last"):
        """Retoma do checkpoint. O Colab derruba a sessão — é questão de quando."""
        import keras

        m, s = self._caminho(tag, "keras"), self._caminho(tag, "json")
        if not (os.path.exists(m) and os.path.exists(s)):
            return False
        self.model = keras.models.load_model(m)
        self._carregar_extra(tag)
        self.on_model_reloaded()
        with open(s, encoding="utf-8") as f:
            estado = json.load(f)
        self.global_step = estado["global_step"]
        self.episodes = estado["episodes"]
        self.iteration = estado["iteration"]
        self.history = estado["history"]
        self.evals = estado.get("evals", [])
        self.baseline = estado.get("baseline")
        self.melhor = estado.get("melhor", -np.inf)
        self._proximo_eval = proximo_multiplo(self.global_step,
                                              self.cfg.eval_every_steps)
        self._proximo_log = self.global_step
        return True

    @staticmethod
    def _com_sufixo(variant, sufixo):
        """Acrescenta `sufixo` à variante, sem duplicar quando ela já o traz."""
        if not sufixo:
            return variant
        sufixo = sufixo if sufixo.startswith("_") else f"_{sufixo}"
        return variant if variant.endswith(sufixo) else variant + sufixo

    def on_model_reloaded(self):
        """Gancho: o otimizador antigo aponta para as variáveis do modelo antigo."""

    # ------------------------------------------------------------------ treino
    def iterate(self):
        raise NotImplementedError

    def train(self, verbose=True, ate_passos=None):
        """Roda até o orçamento, avaliando na cadência oficial. Devolve o `RunRecord`."""
        alvo = ate_passos or self.cfg.total_steps
        # O `env_spec` descreve o ambiente que **de fato** rodou, não o contrato: uma
        # execução com `canal_fome=True` gravava `n_channels: 5` no registro, e o
        # arquivo mentia sobre a própria observação. Ele continua idêntico ao contrato
        # em qualquer execução de 5 canais, que é o caso normal.
        env_spec = dict(CONTRATO)
        canais = getattr(getattr(self, "env", None), "n_channels", None)
        if canais:
            env_spec["n_channels"] = int(canais)

        rec = Recorder(self.algo, variant=self.variant, seed=self.cfg.seed,
                       net=self.cfg.net,
                       params=self.model.count_params() if self.model else 0,
                       config=asdict(self.cfg), env_spec=env_spec,
                       root=self.cfg.runs_dir)
        self.piso()

        while self.global_step < alvo:
            stats = self.iterate()
            self.iteration += 1
            # Quantos passos de gradiente o orçamento de ambiente comprou. Fica no
            # metadado porque é o eixo do §2.1 da revisão e não dá para reconstruir do
            # `config` — o early-stop por KL corta épocas. Zero significa "o agente não
            # reporta", não "não atualizou".
            self._atualizacoes += int(stats.get("atualizacoes", 0) or 0)

            # Quem chama `registra_fim` no laço de coleta já contabilizou os episódios com
            # a causa da morte junto; registrar de novo aqui contaria cada um duas vezes e
            # a média móvel ficaria certa por acidente, mas as frações, não.
            m, k = stats.get("train_score_mean"), stats.get("n_episodes") or 0
            if not self._registrou_causas and m is not None and k:
                self._registra_episodios(m, k)

            if self.global_step >= self._proximo_log:
                self._proximo_log = self.global_step + self.cfg.log_every_steps
                # a curva registra a **média móvel**, não a iteração isolada: é o número
                # que responde "o treino está andando?" sem depender de quantos episódios
                # acabaram no exato momento do log
                bloco = self.resumo_bloco()
                ponto = {"episodes": self.episodes,
                         "train_score_mean": self.media_movel(),
                         "train_score_iter": stats.get("train_score_mean"),
                         **{k: v for k, v in stats.items() if k != "train_score_mean"},
                         **self.resumo_janela(), **bloco}
                self.history.append({"global_step": self.global_step, **ponto})
                rec.log(self.global_step, **ponto)
                if verbose:
                    self._imprimir(stats, bloco)
                self._marcar_bloco()

            if self.global_step >= self._proximo_eval:
                self._proximo_eval = proximo_multiplo(self.global_step,
                                                      self.cfg.eval_every_steps)
                av = self.avaliar()
                av["global_step"] = self.global_step
                av["episodes"] = self.episodes
                self.evals.append(av)
                rec.log(self.global_step, eval_score_mean=av["score_mean"],
                        eval_score_p95=av["score_p95"], episodes=self.episodes)
                if verbose:
                    print(f"  [eval] passo {self.global_step:,} · "
                          f"score {av['score_mean']:.2f} "
                          f"(piso {self.baseline:.2f})")
                if av["score_mean"] > self.melhor:
                    self.melhor = av["score_mean"]
                    self.salvar("best")
                self.salvar("last")

        final = self.avaliar()
        rec.log(self.global_step, eval_score_mean=final["score_mean"],
                eval_score_p95=final["score_p95"], episodes=self.episodes)

        # O melhor checkpoint é medido com o **mesmo** protocolo, e não reaproveita o
        # número da avaliação periódica: aquele veio de outra amostra, e comparar duas
        # medições ruidosas favorece sistematicamente quem foi medido mais vezes.
        melhor = self.avaliar_melhor(verbose=verbose)
        rec.finish(final, melhor_stats=melhor,
                   comparable=getattr(self.cfg, "comparable", True),
                   caveat=getattr(self.cfg, "caveat", ""))
        rec.record.meta["baseline"] = self.baseline
        # Onde este número foi produzido. Uma curva do Kaggle e outra do Colab são
        # comparáveis — o contrato garante isso — mas o **tempo de parede** não é, e
        # `wall_s_total` é lido com frequência como se fosse.
        rec.record.meta.update(resumo_plataforma())
        # Quantos canais a rede realmente viu. Fica no metadado porque é a diferença que
        # torna uma curva incomparável com outra, e "comparable=False + caveat em prosa"
        # não é conferível por máquina — este número é.
        if getattr(self, "env", None) is not None:
            rec.record.meta["obs_channels"] = int(
                getattr(self.env, "n_channels", CONTRATO["n_channels"]))
        if self._atualizacoes:
            rec.record.meta["atualizacoes"] = int(self._atualizacoes)
        self.salvar("last")

        # O registro é gravado SEMPRE. Estourar no fim de um treino de horas e perder a
        # curva seria o pior desfecho possível; o portão do contrato age na hora de
        # montar a arena, não na hora de escrever. As violações ficam no metadado e
        # `RunRecord.oficial` passa a ser False.
        problemas = validate(rec.record)
        if problemas:
            rec.record.meta["contract_violations"] = problemas
            if verbose:
                print("\n[contrato] esta execução NÃO entra na arena:")
                for p in problemas:
                    print(f"  - {p}")
        caminho = rec.save(skip_validation=True)
        if verbose:
            print(f"[registro] {caminho}")

        self.artefatos(rec, verbose=verbose)
        return rec

    # ---------------------------------------------------------------- artefatos
    def modelo_melhor(self):
        """O modelo do checkpoint `best`, ou `None` se ele não existe.

        Carrega numa instância separada de propósito: `self.model` continua sendo o do
        último passo, porque é ele que define a curva e o número oficial. Trocar em
        silêncio faria a última avaliação medir uma coisa e a curva outra.
        """
        import keras

        caminho = self._caminho("best", "keras")
        if not os.path.exists(caminho):
            return None
        return keras.models.load_model(caminho)

    @contextlib.contextmanager
    def politica_de_checkpoint(self, tag="best"):
        """Uma política que joga pelo checkpoint `tag`, válida dentro do bloco.

        `None` quando o checkpoint não existe. É um gerenciador de contexto porque há
        agentes — o DreamerV3 — que só conseguem jogar um checkpoint **trocando os pesos
        dos próprios submodelos**, e nesse caso a restauração precisa acontecer mesmo se a
        avaliação levantar.
        """
        m = self.modelo_melhor() if tag == "best" else None
        yield None if m is None else self.politica_do_modelo(m)

    def avaliar_melhor(self, verbose=True):
        """Roda o protocolo oficial sobre o melhor checkpoint. `{}` se não houver.

        Avalia **pelo modelo carregado**, sem tocar em `self.model`: a troca de atributo
        era silenciosamente ineficaz em dois agentes (ver `politica_do_modelo`).
        """
        try:
            with self.politica_de_checkpoint("best") as pol:
                if pol is None:
                    return {}
                stats = self.avaliar(politica=pol)
        except NotImplementedError as e:
            if verbose:
                print(f"  [melhor] não avaliado: {e}")
            return {"indisponivel": str(e),
                    "global_step": int(self._passo_do_melhor())}
        stats["global_step"] = int(self._passo_do_melhor())
        if verbose:
            print(f"  [melhor] checkpoint do passo {stats['global_step']:,} · "
                  f"score {stats['score_mean']:.2f} "
                  f"(último: {self.evals[-1]['score_mean']:.2f})"
                  if self.evals else
                  f"  [melhor] score {stats['score_mean']:.2f}")
        return stats

    def _passo_do_melhor(self):
        caminho = self._caminho("best", "json")
        if os.path.exists(caminho):
            with open(caminho, encoding="utf-8") as f:
                return json.load(f).get("global_step", 0)
        return 0

    def copiar_modelos(self, destino, verbose=True):
        """Leva `last.keras` e `best.keras` para dentro da pasta da execução.

        Os checkpoints vivem em `ckpt_dir`, que é compartilhado e sobrescrito pela
        execução seguinte. Sem esta cópia, o `history.json` afirma um score que ninguém
        consegue reproduzir nem inspecionar depois — e o GIF vira a única evidência de
        como o agente jogava.

        Os dois, e não só o melhor: `last` é o modelo que produziu o número **oficial**,
        então é ele que permite reconferir a curva; `best` é o que se leva para o jogo.
        """
        import shutil

        pasta = os.path.join(destino, "modelos")
        os.makedirs(pasta, exist_ok=True)
        copiados = {}
        for tag in ("last", "best"):
            # o `.npz` acompanha o `.keras`: para o DreamerV3 é ele que carrega o modelo
            # do mundo, e sem ele a pasta guarda um ator que não joga (§1.4 da revisão)
            for ext in ("keras", "npz"):
                origem = self._caminho(tag, ext)
                if os.path.exists(origem):
                    alvo = os.path.join(pasta, f"{tag}.{ext}")
                    shutil.copyfile(origem, alvo)
                    copiados[tag if ext == "keras" else f"{tag}+pesos"] = alvo
        if verbose and copiados:
            mb = sum(os.path.getsize(c) for c in copiados.values()) / 1e6
            print(f"  [modelos] {', '.join(sorted(copiados))} em {pasta} ({mb:.1f} MB)")
        return copiados

    def artefatos(self, rec, verbose=True):
        """Gráfico, GIFs e os modelos — tudo ao lado do `history.json`.

        A pasta da execução tem que ser autossuficiente: quem a recebe consegue ver a
        curva, ver o agente jogando e **rodar o modelo**, sem depender de nenhum estado
        que ficou na máquina de quem treinou.
        """
        import os

        destino = os.path.dirname(rec.save(skip_validation=True))
        saida = {}
        saida["modelos"] = self.copiar_modelos(destino, verbose=verbose)

        if self.cfg.salvar_grafico:
            try:
                import matplotlib
                matplotlib.use("Agg")

                fig, _ = plot_run(rec.record)
                caminho = os.path.join(destino, "curva.png")
                fig.savefig(caminho, dpi=150, facecolor=fig.get_facecolor())
                matplotlib.pyplot.close(fig)
                saida["grafico"] = caminho
            except Exception as e:                      # nunca derrubar o treino por isso
                saida["grafico_erro"] = repr(e)

        if self.cfg.salvar_gif:

            politica = self.politica()
            for seed in self.cfg.gif_seeds:
                try:
                    caminho, score, motivo = render_episode(
                        politica, caminho=os.path.join(destino, f"episodio_s{seed}.gif"),
                        board_size=self.cfg.board_size, seed=seed,
                        canal_fome=getattr(self.env, "canal_fome", False),
                    )
                    saida[f"gif_s{seed}"] = {"caminho": caminho, "score": score,
                                             "fim": motivo}
                    if verbose:
                        print(f"[gif] seed {seed}: score {score}, terminou por {motivo}")
                except Exception as e:
                    saida[f"gif_s{seed}_erro"] = repr(e)

        rec.record.meta["artefatos"] = saida
        rec.save(skip_validation=True)
        return saida

    #: Legenda impressa uma vez, antes da primeira linha de log.
    LEGENDA = ("[log] cada métrica sai como  janela | bloco  — a média móvel dos últimos "
               "~{janela} episódios\n"
               "      à esquerda, e só os episódios encerrados desde o log anterior à "
               "direita.\n"
               "      Elas discordam por muitos logs antes de a móvel virar: a da "
               "esquerda diz onde o\n"
               "      agente está, a da direita diz para onde ele está indo.")

    @staticmethod
    def _par(janela, bloco, fmt, largura):
        """`janela | bloco` no mesmo formato, com `—` quando o bloco está vazio."""
        esq = f"{janela:{fmt}}" if janela is not None else "—"
        dir_ = f"{bloco:{fmt}}" if bloco is not None else "—"
        return f"{esq:>{largura}}|{dir_:<{largura}}"

    def _imprimir(self, stats, bloco=None):
        """Uma linha por log: a janela móvel e o bloco novo, lado a lado.

        O score sozinho é ambíguo: 1,2 pontos pode ser "bate em tudo" ou "anda em círculo
        até morrer de fome", e a curva fica igual nos dois casos. Por isso a linha traz a
        **repartição das causas de fim**, que separa os dois de imediato, mais o
        comprimento médio do episódio, que é o sinal mais precoce de todos — uma cobra que
        aprende a sobreviver alonga os episódios antes de o score subir.

        E cada uma dessas medidas aparece **duas vezes**: sobre a janela de
        `JANELA_EPISODIOS` e sobre os episódios encerrados desde o log anterior. A média
        móvel existe para o número não pular com uma amostra de 3 episódios, mas o preço é
        atraso — com ~80 episódios por log ela carrega seis logs de passado. Numa
        degradação em curso as duas colunas discordam bem antes de a curva virar, e é
        exatamente essa discordância que se quer ver.
        """
        if not self._legenda_impressa:
            print(self.LEGENDA.format(janela=self.JANELA_EPISODIOS))
            self._legenda_impressa = True
        r = self.resumo_janela()
        b = bloco if bloco is not None else self.resumo_bloco()
        n_bloco = b.get("bloco_episodios", 0)
        partes = [
            f"passo {self.global_step:>10,}",
            f"ep {self.episodes:>7,} +{n_bloco:<4}",
            "score " + self._par(self.media_movel(), b.get("bloco_train_score_mean"),
                                 ".2f", 6),
        ]
        if "win_rate" in r:
            partes += [
                "fome " + self._par(r["frac_fome"], b.get("bloco_frac_fome"), ".1%", 6),
                "colisão " + self._par(r["frac_colisao"], b.get("bloco_frac_colisao"),
                                       ".1%", 6),
                "vit " + self._par(r["win_rate"], b.get("bloco_win_rate"), ".1%", 6),
                self._par(r["passos_por_episodio"], b.get("bloco_passos_por_episodio"),
                          ".0f", 4) + " passos/ep",
            ]
        partes.append(f"janela {self.episodios_na_janela()}")
        print(" · ".join(partes))


# --- snakeai/kfac.py ---
"""K-FAC — curvatura aproximada por fatores de Kronecker, em Keras 3.

O que é
-------
Descida de gradiente natural precisa de `F⁻¹∇`, onde `F` é a matriz de Fisher. Para uma
rede com 300 mil parâmetros, `F` tem 9×10¹⁰ entradas: não cabe, muito menos inverte. O
K-FAC (Martens & Grosse, 2015) aproxima `F` por **blocos, um por camada**, e aproxima cada
bloco por um **produto de Kronecker de duas matrizes pequenas**::

    F_ℓ  ≈  A_ℓ ⊗ G_ℓ

* `A_ℓ` = covariância das **ativações que entram** na camada — lado `(entrada × entrada)`;
* `G_ℓ` = covariância dos **gradientes na pré-ativação** que sai — lado `(saída × saída)`.

A conta que torna isso viável é a identidade `(A ⊗ G)⁻¹ vec(∇W) = vec(A⁻¹ ∇W G⁻¹)`: em vez
de inverter uma matriz de `(in·out)²`, inverte-se uma de `in²` e uma de `out²`. Numa camada
de 288×64, isso é 340 mil entradas em vez de 340 **bilhões**.

Por que ele não entrou no eixo `optimizer`
------------------------------------------
Um `keras.optimizers.Optimizer` recebe apenas pares `(gradiente, variável)`. O K-FAC precisa
das **ativações de entrada** e dos **gradientes de pré-ativação** de cada camada — coisas
que só existem durante o passo forward/backward e que nenhum otimizador do Keras enxerga.
A API Keras do `tensorflow/kfac` contornava isso recebendo `model=` e `loss=` e refazendo o
forward por dentro; aquele repositório foi arquivado em 19/04/2026 e depende de
`tensorflow.compat.v1`, então não é um caminho.

Aqui o K-FAC é um **pré-condicionador**, não um otimizador: ele se coloca entre o gradiente
e o `optimizer.apply_gradients`. Quem o usa é o `ACKTR` (`snakeai/agents/acktr.py`), que é
o uso historicamente correto do K-FAC em RL.

Fisher de verdade, não Fisher empírico
--------------------------------------
`G_ℓ` tem que ser a covariância dos gradientes do **log-likelihood do modelo com rótulos
amostrados do próprio modelo** — não dos gradientes da perda de RL. Usar a perda de RL dá o
*Fisher empírico*, que é uma matriz diferente e um pré-condicionador reconhecidamente pior:
perto de um ótimo ele colapsa, porque os gradientes vão a zero por acerto, não por
curvatura baixa. Então o `passo_kfac` faz **duas** retropropagações sobre o mesmo forward:

1. a perda real, que dá o gradiente a ser pré-condicionado;
2. a *perda de Fisher* — `log π(a')` com `a' ~ π(·|s)` amostrada, mais um alvo gaussiano
   para o crítico — que dá **só** as estatísticas de `G`.

Amortecimento
-------------
`A` e `G` são singulares na prática (mais parâmetros que amostras no lote). O amortecimento
de Tikhonov fatorado (Martens & Grosse, §6.3) distribui `λ` entre os dois fatores de forma
que `(A + √λ·π·I) ⊗ (G + √λ/π·I)` fique o mais perto possível de `A⊗G + λ·I`::

    π = sqrt( (tr(A)/dim A) / (tr(G)/dim G) )

Sem o `π`, o amortecimento cai desigual sobre os dois lados e a direção sai enviesada.

Cobertura
---------
Cobre `Dense` e `Conv2D` — que no `resnet_tiny` são 96% dos parâmetros. As camadas restantes
(`GroupNormalization`) recebem o gradiente cru. `KFac.resumo()` diz exatamente qual fração
dos parâmetros está sob pré-condicionamento, porque um K-FAC que cobre metade da rede e não
avisa é pior que nenhum.
"""


import contextlib
import types

import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
import numpy as np
import tensorflow as tf
from keras import layers

__all__ = ["KFac", "EKFac", "captura_kfac", "patches_de_entrada",
           "fatores_de_camada", "perda_fisher_categorica", "perda_fisher_gaussiana"]

REGISTRAVEIS = (layers.Dense, layers.Conv2D)


# --------------------------------------------------------------------- captura
def _call_dense(self, inputs, *a, **kw):
    z = keras.ops.matmul(inputs, self.kernel)
    if self.use_bias:
        z = z + self.bias
    _REGISTRO[-1].append((self, inputs, z))
    return self.activation(z) if self.activation is not None else z


def _call_conv(self, inputs, *a, **kw):
    z = tf.nn.convolution(
        inputs, self.kernel,
        strides=list(self.strides), padding=self.padding.upper(),
        dilations=list(self.dilation_rate),
    )
    if self.use_bias:
        z = tf.nn.bias_add(z, self.bias)
    _REGISTRO[-1].append((self, inputs, z))
    return self.activation(z) if self.activation is not None else z


#: Pilha de listas de captura. Pilha, e não uma lista só, para que um `captura_kfac`
#: aninhado (ou uma retraçagem do `tf.function` no meio de outra) não misture tensores de
#: escopos diferentes — que seria um bug silencioso, do tipo que dá números plausíveis.
_REGISTRO = []

_AUSENTE = object()


@contextlib.contextmanager
def captura_kfac(camadas):
    """Durante o bloco, cada camada de `camadas` registra `(camada, entrada, pré-ativação)`.

    Reimplementa o `call` de `Dense` e `Conv2D` porque a pré-ativação **não existe como
    tensor** quando a ativação vem fundida (`Dense(64, activation="relu")`): o Keras calcula
    `relu(x @ W + b)` de uma vez e só o resultado final vira nó do grafo. `G` precisa do
    gradiente em `x @ W + b`, antes da ativação.

    Duplicar a semântica de uma camada é arriscado — por isso
    `tests/test_kfac.py::test_captured_forward_is_bit_identical` compara a saída do modelo
    com e sem captura e exige igualdade exata.
    """
    _REGISTRO.append([])
    tocadas = []
    try:
        for c in camadas:
            novo = _call_dense if isinstance(c, layers.Dense) else _call_conv
            # Guarda o `call` de instância anterior — ou a ausência dele, que é o caso
            # normal. Restaurar com `c.call = c.call` deixaria um atributo de instância
            # sombreando o método da classe para sempre.
            tocadas.append((c, c.__dict__.get("call", _AUSENTE)))
            c.call = types.MethodType(novo, c)
        yield _REGISTRO[-1]
    finally:
        for c, anterior in tocadas:
            if anterior is _AUSENTE:
                c.__dict__.pop("call", None)
            else:
                c.call = anterior
        _REGISTRO.pop()


# ---------------------------------------------------------------- perda de Fisher
def perda_fisher_categorica(logits, mask=None, seed=None):
    """`log π(a')` com `a' ~ π(·|s)`. É daqui que sai `G` da cabeça de política.

    O detalhe que decide entre Fisher e Fisher empírico está na **origem da ação**: aqui
    ela é amostrada da política atual, não é a ação que o agente de fato tomou. Usar a ação
    tomada daria o Fisher empírico — outra matriz, e um pré-condicionador que degenera perto
    do ótimo, quando os gradientes vão a zero por acerto e não por curvatura baixa.
    """
    if mask is not None:
        logits = tf.where(mask, logits, tf.fill(tf.shape(logits), -1e9))
    amostra = tf.random.categorical(logits, 1, seed=seed)[:, 0]
    logp = tf.nn.log_softmax(logits)
    return tf.reduce_mean(tf.gather(logp, amostra, batch_dims=1))


def perda_fisher_gaussiana(valor, seed=None):
    """Alvo gaussiano de variância 1 em volta da própria predição — `G` do crítico.

    O crítico não tem distribuição de saída explícita; o K-FAC trata a regressão como uma
    gaussiana de variância unitária, que é o que faz a "Fisher" do valor ser a
    Gauss-Newton. Na prática o gradiente que chega na pré-ativação é ruído branco puro,
    então `G ≈ I` e o pré-condicionamento do crítico vira Newton sobre `A`.

    Normalização, que é onde isto costuma sair errado por um fator igual à dimensão de
    saída: a log-verossimilhança de uma gaussiana `d`-dimensional **soma** sobre as `d`
    componentes e só depois tira a média sobre o lote. Trocar essa soma por uma média
    encolhe `G` por `d` e o passo natural sai `d` vezes maior — o que num crítico escalar
    (`d = 1`) não faz diferença nenhuma e por isso passa despercebido até alguém usar uma
    saída vetorial.
    """
    ruido = tf.random.normal(tf.shape(valor), seed=seed)
    alvo = tf.stop_gradient(valor) + ruido
    quadrado = tf.square(valor - alvo)
    return 0.5 * tf.reduce_mean(tf.reduce_sum(
        tf.reshape(quadrado, [tf.shape(quadrado)[0], -1]), axis=-1))


# ---------------------------------------------------------------------- fatores
def patches_de_entrada(camada, entrada):
    """Achata a entrada de uma camada na forma `(amostras, dim_entrada)` que `A` espera.

    Para `Dense` é a própria entrada. Para `Conv2D` é o truque do KFC (Grosse & Martens,
    2016): cada posição espacial da saída consome um *patch* `kh × kw × cin` da entrada, e
    a convolução é uma `Dense` aplicada a cada patch. Extraindo os patches, a camada
    convolucional vira densa e o resto da conta é idêntico.

    A ordem do achatamento (`kh, kw, cin`) é a mesma de `kernel.reshape(-1, cout)` — o que
    não é óbvio e por isso está testado em `test_conv_patches_reproduce_the_convolution`.
    """
    if isinstance(camada, layers.Dense):
        return tf.reshape(entrada, [-1, tf.shape(entrada)[-1]])

    kh, kw = camada.kernel_size
    sh, sw = camada.strides
    dh, dw = camada.dilation_rate
    p = tf.image.extract_patches(
        entrada, sizes=[1, kh, kw, 1], strides=[1, sh, sw, 1],
        rates=[1, dh, dw, 1], padding=camada.padding.upper(),
    )
    return tf.reshape(p, [-1, kh * kw * entrada.shape[-1]])


def fatores_de_camada(camada, entrada, grad_pre):
    """Devolve `(A, G, n_amostras, n_posicoes)` para uma camada.

    Convenções de escala, que decidem se o pré-condicionamento está certo ou só parece
    certo. A perda é uma **média** sobre `N` amostras, então o gradiente de pré-ativação
    por amostra vale `N · g`. Daí::

        A = (1/N) Σ â âᵀ            â = [a, 1] se a camada tem viés
        G = (N/T) Σ g gᵀ            T = posições espaciais (1 em `Dense`)

    e `Δ = A⁻¹ ∇W G⁻¹`, com `∇W` na forma `(dim_entrada[+1], saída)`.
    """
    a = patches_de_entrada(camada, entrada)
    n = tf.cast(tf.shape(entrada)[0], tf.float32)
    t = tf.cast(tf.shape(a)[0], tf.float32) / n

    if camada.use_bias:
        a = tf.concat([a, tf.ones([tf.shape(a)[0], 1], a.dtype)], axis=1)

    g = tf.reshape(grad_pre, [-1, tf.shape(grad_pre)[-1]])

    A = tf.matmul(a, a, transpose_a=True) / n
    G = tf.matmul(g, g, transpose_a=True) * (n / t)
    return A, G, n, t


# ------------------------------------------------------------------------ K-FAC
class KFac:
    """Pré-condicionador K-FAC para um `keras.Model` funcional.

    Uso::

        kf = KFac(model, damping=1e-2)
        with captura_kfac(kf.camadas) as cap:
            with tf.GradientTape(persistent=True) as tape:
                ...  # forward
                perda, perda_fisher = ...
            grads = tape.gradient(perda, model.trainable_variables)
            gs = tape.gradient(perda_fisher, [z for _, _, z in cap])
        kf.acumula(cap, gs)
        nat = kf.precondiciona(grads)
    """

    def __init__(self, model, damping=1e-2, ema=0.95, inv_every=20, eps=1e-8):
        self.model = model
        self.damping = float(damping)
        self.ema = float(ema)
        self.inv_every = int(inv_every)
        self.eps = float(eps)

        self.camadas = [c for c in model.layers
                        if isinstance(c, REGISTRAVEIS) and c.trainable_weights]
        if not self.camadas:
            raise ValueError("nenhuma camada Dense ou Conv2D registrável no modelo")

        self._A, self._G = {}, {}
        self._cholA, self._cholG = {}, {}
        self._passos = 0
        #: Índice das variáveis de cada camada dentro de `model.trainable_variables`.
        #: Guardado por `id`, porque comparar `tf.Variable` com `==` faz broadcast.
        ordem = {id(v): i for i, v in enumerate(model.trainable_variables)}
        self._idx = {c.name: (ordem[id(c.kernel)],
                              ordem[id(c.bias)] if c.use_bias else None)
                     for c in self.camadas}

    # ------------------------------------------------------------- estatísticas
    def acumula(self, capturado, grads_pre):
        """Atualiza as médias móveis de `A` e `G` com um lote."""
        for (camada, entrada, _), gp in zip(capturado, grads_pre):
            if gp is None:
                continue
            A, G, _, _ = fatores_de_camada(camada, entrada, gp)
            nome = camada.name
            if nome in self._A:
                d = self.ema
                self._A[nome] = d * self._A[nome] + (1.0 - d) * A
                self._G[nome] = d * self._G[nome] + (1.0 - d) * G
            else:
                self._A[nome], self._G[nome] = A, G
        self._passos += 1
        if (self._passos - 1) % self.inv_every == 0:
            self.atualiza_inversos()

    def atualiza_inversos(self):
        """Refatora `A` e `G` amortecidos. Cholesky, não inversa explícita.

        `cholesky_solve` resolve o sistema com metade das operações de uma inversão e sem
        o erro de arredondamento de multiplicar por uma inversa formada explicitamente.
        """
        for nome in self._A:
            A, G = self._A[nome], self._G[nome]
            dA = tf.cast(tf.shape(A)[0], tf.float32)
            dG = tf.cast(tf.shape(G)[0], tf.float32)
            trA = tf.linalg.trace(A) / dA
            trG = tf.linalg.trace(G) / dG
            pi = tf.sqrt((trA + self.eps) / (trG + self.eps))
            raiz = np.sqrt(self.damping)
            self._cholA[nome] = tf.linalg.cholesky(
                A + tf.eye(tf.shape(A)[0]) * (raiz * pi))
            self._cholG[nome] = tf.linalg.cholesky(
                G + tf.eye(tf.shape(G)[0]) * (raiz / pi))

    # ----------------------------------------------------------- condicionamento
    def precondiciona(self, grads):
        """`grads` cru → direção natural. Camadas não cobertas passam intactas."""
        if not self._cholA:
            return list(grads)
        saida = list(grads)
        for camada in self.camadas:
            nome = camada.name
            if nome not in self._cholA:
                continue
            ik, ib = self._idx[nome]
            gk = grads[ik]
            if gk is None:
                continue

            forma = tf.shape(camada.kernel)
            cout = camada.kernel.shape[-1]
            plano = tf.reshape(gk, [-1, cout])
            if ib is not None:
                plano = tf.concat([plano, tf.reshape(grads[ib], [1, cout])], axis=0)

            # Δ = A⁻¹ ∇W G⁻¹, via dois sistemas triangulares
            x = tf.linalg.cholesky_solve(self._cholA[nome], plano)
            x = tf.transpose(tf.linalg.cholesky_solve(
                self._cholG[nome], tf.transpose(x)))

            if ib is not None:
                saida[ib] = x[-1]
                x = x[:-1]
            saida[ik] = tf.reshape(x, forma)
        return saida

    # ------------------------------------------------------------- região de confiança
    @staticmethod
    def escala_kl(naturais, crus, kl_max, lr_max):
        """Passo maior que respeita uma KL alvo — a parte "trust region" do ACKTR.

        Com `Δ = F⁻¹∇`, a KL de segunda ordem induzida por um passo `ηΔ` vale
        `½η² ΔᵀFΔ`, e `ΔᵀFΔ = Δᵀ∇` — um produto interno, sem tocar em `F`. Igualando a
        `kl_max` sai `η = sqrt(2·kl_max / Δᵀ∇)`, limitado por `lr_max`.

        É isto que permite ao ACKTR usar passos que derrubariam um A2C: o tamanho não é
        fixo, é o maior que ainda cabe dentro da KL pedida.
        """
        quad = tf.add_n([tf.reduce_sum(n * g)
                         for n, g in zip(naturais, crus) if n is not None and g is not None])
        quad = tf.maximum(quad, 1e-12)
        return tf.minimum(lr_max, tf.sqrt(2.0 * kl_max / quad))

    # ------------------------------------------------------------------- relato
    def resumo(self):
        """Quantos parâmetros estão de fato sob pré-condicionamento."""
        cobertos = sum(int(np.prod(v.shape))
                       for c in self.camadas for v in c.trainable_weights)
        total = sum(int(np.prod(v.shape)) for v in self.model.trainable_variables)
        return {
            "camadas": [c.name for c in self.camadas],
            "params_cobertos": cobertos,
            "params_total": total,
            "fracao": cobertos / max(1, total),
            "maior_fator": max((int(self._A[n].shape[0]) for n in self._A), default=0),
        }


# ----------------------------------------------------------------------- EK-FAC
class EKFac(KFac):
    """EK-FAC — K-FAC com os autovalores **medidos** em vez de fatorados.

    (George et al., 2018, *Fast Approximate Natural Gradient Descent in a
    Kronecker-factored Eigenbasis*.)

    A ideia em uma frase
    --------------------
    O K-FAC faz duas coisas ao mesmo tempo e só uma delas é boa. A decomposição
    `A ⊗ G = (U_A ⊗ U_G)(S_A ⊗ S_G)(U_A ⊗ U_G)ᵀ` dá **um sistema de coordenadas** — a base
    de autovetores, chamada de KFE — e **uma escala em cada eixo** dessa base. A base é uma
    aproximação razoável dos autovetores da Fisher de verdade; as escalas, não: elas são
    obrigadas a ter forma de produto de Kronecker, `λ_A(j)·λ_G(i)`, e essa restrição não
    tem justificativa nenhuma além de ter saído junto.

    O EK-FAC mantém a base e **joga fora as escalas**, medindo no lugar delas o segundo
    momento verdadeiro do gradiente projetado::

        s*_{ji} = E_n[ ((U_Aᵀ ∇W_n U_G)_{ji})² ]

    O Teorema 2 do paper diz que `s*` é a melhor escala diagonal possível **naquela base**,
    em norma de Frobenius; o Teorema 3 conclui que o EK-FAC nunca é pior que o K-FAC. Não é
    uma heurística com um `ε` a mais: é o mínimo de um problema de mínimos quadrados, e o
    K-FAC é um ponto qualquer do mesmo espaço de busca.

    Por que sai barato
    ------------------
    O gradiente **por amostra** de uma camada densa é o produto externo `a_n g_nᵀ`. Projetar
    um produto externo é projetar cada lado::

        U_Aᵀ (a_n g_nᵀ) U_G = (U_Aᵀ a_n)(U_Gᵀ g_n)ᵀ

    então o quadrado da entrada `(j,i)` é `(U_Aᵀa_n)_j² · (U_Gᵀg_n)_i²`, e a média sobre o
    lote inteiro é **um produto de matrizes** entre as projeções ao quadrado. Nada de
    materializar `N` gradientes por amostra, nada de laço em Python — e é por isso que a
    implementação de referência em PyTorch precisa de um laço sobre o lote e esta não.

    O que muda no custo em relação ao K-FAC:

    * **por atualização**: uma projeção a mais das ativações e dos gradientes de
      pré-ativação, `O(N·T·d²)` — da mesma ordem do que montar `A` já custa. O
      pré-condicionamento em si troca dois `cholesky_solve` por quatro produtos de matriz;
    * **a cada `inv_every`**: `eigh` em vez de `cholesky`, mais caro por uma constante.

    O paper propõe **amortizar**: recalcular a base raramente (50 a 500 passos) e as escalas
    a cada passo. Aqui `inv_every` continua com o padrão do ACKTR — ver `docs/EKFAC.md`
    para por que o padrão fica assim e o que se ganha ao subi-lo.

    O amortecimento, e por que a primeira atualização é idêntica à do K-FAC
    -----------------------------------------------------------------------
    O amortecimento de Tikhonov fatorado do K-FAC dá aos eixos da KFE a escala
    `(λ_A + √λ·π)(λ_G + √λ/π)`, que expandida é
    `λ_Aλ_G + λ_A·√λ/π + λ_G·√λ·π + λ`. O apêndice C do paper prescreve reproduzir
    exatamente essa estrutura em torno de `s*`::

        denominador_{ji} = s*_{ji} + λ_A(j)·√λ/π + λ_G(i)·√λ·π + λ

    Isso tem uma consequência que vale como teste: com `s*` inicializado em `λ_A·λ_G` — que
    é o que o EK-FAC assume antes de medir qualquer coisa —, o denominador é **idêntico** ao
    do K-FAC amortecido, e as duas direções coincidem até o último bit. O EK-FAC começa
    exatamente onde o K-FAC está e se afasta conforme mede; a diferença entre as duas curvas
    não inclui "uma começou de um lugar diferente da outra".

    A convolução, e o que "exato" quer dizer nela
    ---------------------------------------------
    Numa `Dense`, `s*` é o segundo momento exato — sem aproximação nenhuma. Numa `Conv2D`, o
    gradiente por amostra é a **soma sobre as posições espaciais** de produtos externos, e o
    quadrado de uma soma não se decompõe. Aqui, como no KFC, cada posição é tratada como uma
    amostra independente, e `s*` é o segundo momento exato **sob essa hipótese** — a mesma
    que o `A ⊗ G` do K-FAC para convolução já faz. Ou seja: o EK-FAC corrige os autovalores
    dentro da hipótese de homogeneidade espacial, não a hipótese. Está registrado aqui
    porque "autovalores exatos" numa camada convolucional é uma frase que promete mais do
    que entrega.
    """

    def __init__(self, model, damping=1e-2, ema=0.95, inv_every=10, eps=1e-8,
                 ema_escalas=0.5):
        super().__init__(model, damping=damping, ema=ema, inv_every=inv_every, eps=eps)
        #: Autovetores e autovalores dos dois fatores — a KFE.
        self._UA, self._UG = {}, {}
        self._lamA, self._lamG = {}, {}
        #: `s*`, o segundo momento medido na KFE. Forma `(entrada[+1], saída)`, a mesma de
        #: `∇W`, porque cada entrada dele é a escala de **um** eixo da base.
        self._m2 = {}
        #: `π` do amortecimento fatorado, guardado por camada para o denominador.
        self._pi = {}
        self.ema_escalas = float(ema_escalas)

    # ------------------------------------------------------------- estatísticas
    def acumula(self, capturado, grads_pre):
        """Médias móveis de `A` e `G` (do K-FAC) **e** de `s*` (o que o EK-FAC acrescenta).

        A ordem importa: a base tem que existir antes de projetar. `KFac.acumula` já chama
        `atualiza_inversos` — que aqui virou a construção da KFE — no passo certo, então
        basta medir as escalas depois.
        """
        super().acumula(capturado, grads_pre)
        self._atualiza_escalas(capturado, grads_pre)

    def atualiza_inversos(self):
        """Constrói a KFE e **reinicia** `s*` nos autovalores do K-FAC.

        Reiniciar é obrigatório, não uma escolha: `s*` são escalas de eixos de uma base
        específica, e quando a base muda os números antigos passam a descrever eixos que
        não existem mais. Reaproveitá-los daria um pré-condicionador que mistura duas
        bases — plausível, silencioso e errado.

        O valor de partida é `λ_A ⊗ λ_G`, que é a hipótese do K-FAC. É o prior honesto:
        antes de medir, o EK-FAC não sabe mais do que o K-FAC sabia.
        """
        for nome in self._A:
            A, G = self._A[nome], self._G[nome]
            dA = tf.cast(tf.shape(A)[0], tf.float32)
            dG = tf.cast(tf.shape(G)[0], tf.float32)
            trA = tf.linalg.trace(A) / dA
            trG = tf.linalg.trace(G) / dG
            self._pi[nome] = tf.sqrt((trA + self.eps) / (trG + self.eps))

            # `eigh` devolve autovalores em ordem crescente e uma base ortonormal. O
            # `relu` corta os autovalores levemente negativos que aparecem por
            # arredondamento numa matriz que é PSD por construção — deixá-los passar
            # inverteria o sinal daquele eixo do pré-condicionamento.
            lamA, UA = tf.linalg.eigh(A)
            lamG, UG = tf.linalg.eigh(G)
            self._lamA[nome], self._UA[nome] = tf.nn.relu(lamA), UA
            self._lamG[nome], self._UG[nome] = tf.nn.relu(lamG), UG
            self._m2[nome] = (self._lamA[nome][:, None]
                              * self._lamG[nome][None, :])

    def _atualiza_escalas(self, capturado, grads_pre):
        """Mede `s*` neste lote e mistura na média móvel.

        Tudo acontece em dois produtos de matriz por camada, pelo argumento do produto
        externo no docstring da classe. A escala segue a convenção documentada em
        `fatores_de_camada`: o gradiente de pré-ativação **por amostra** da perda somada
        vale `N·g`, e a soma sobre as `T` posições espaciais entra dividindo por `N`, não
        por `N·T` — é o que faz `s*` nascer na mesma escala de `λ_A·λ_G` e o amortecimento
        do apêndice C fechar.
        """
        for (camada, entrada, _), gp in zip(capturado, grads_pre):
            nome = camada.name
            if gp is None or nome not in self._UA:
                continue

            a = patches_de_entrada(camada, entrada)
            n = tf.cast(tf.shape(entrada)[0], tf.float32)
            if camada.use_bias:
                a = tf.concat([a, tf.ones([tf.shape(a)[0], 1], a.dtype)], axis=1)
            g = tf.reshape(gp, [-1, tf.shape(gp)[-1]]) * n

            pa = tf.square(tf.matmul(a, self._UA[nome]))
            pg = tf.square(tf.matmul(g, self._UG[nome]))
            s = tf.matmul(pa, pg, transpose_a=True) / n

            d = self.ema_escalas
            self._m2[nome] = d * self._m2[nome] + (1.0 - d) * s

    # ----------------------------------------------------------- condicionamento
    def precondiciona(self, grads):
        """`grads` cru → direção natural, com as escalas medidas. Não coberto passa intacto."""
        if not self._UA:
            return list(grads)
        saida = list(grads)
        raiz = np.sqrt(self.damping)

        for camada in self.camadas:
            nome = camada.name
            if nome not in self._UA:
                continue
            ik, ib = self._idx[nome]
            gk = grads[ik]
            if gk is None:
                continue

            forma = tf.shape(camada.kernel)
            cout = camada.kernel.shape[-1]
            plano = tf.reshape(gk, [-1, cout])
            if ib is not None:
                plano = tf.concat([plano, tf.reshape(grads[ib], [1, cout])], axis=0)

            UA, UG = self._UA[nome], self._UG[nome]
            pi = self._pi[nome]
            # o amortecimento do apêndice C: a mesma forma do Tikhonov fatorado do K-FAC,
            # escrita na base — ver o docstring da classe
            denom = (self._m2[nome]
                     + self._lamA[nome][:, None] * (raiz / pi)
                     + self._lamG[nome][None, :] * (raiz * pi)
                     + self.damping)

            proj = tf.matmul(tf.matmul(UA, plano, transpose_a=True), UG)
            x = tf.matmul(tf.matmul(UA, proj / denom), UG, transpose_b=True)

            if ib is not None:
                saida[ib] = x[-1]
                x = x[:-1]
            saida[ik] = tf.reshape(x, forma)
        return saida

    # ------------------------------------------------------------------- relato
    def desvio_de_kronecker(self):
        """Quanto `s*` já se afastou do palpite do K-FAC — o número que diz se isto serve.

        `‖s* − λ_A⊗λ_G‖_F / ‖λ_A⊗λ_G‖_F`, média sobre as camadas: **o tamanho da correção
        que o EK-FAC está aplicando** em relação ao que o K-FAC teria feito. Zero significa
        que ele não está fazendo nada, e a curva dele tem que coincidir com a do ACKTR.
        Sem esta medida, um resultado nulo na arena seria indistinguível de um bug — e o bug
        é a explicação mais provável das duas.

        Ele mede duas coisas somadas, e vale saber quais: quanto a Fisher deste problema
        deixa de ser um produto de Kronecker **naquela base**, e quanto a base envelheceu
        desde que foi construída. As duas são exatamente o que o EK-FAC existe para
        absorver — a segunda é o argumento de amortização do §"update frequency" do paper —
        mas elas não se separam neste número.

        O formato é um **dente de serra**: cai a zero em cada reconstrução da base (é lá que
        `s*` é reiniciado no palpite do K-FAC) e cresce até a próxima. Ler uma atualização
        isolada não diz nada; o que interessa é o pico antes de cada reinício.
        """
        if not self._m2:
            return 0.0
        desvios = []
        for nome, m2 in self._m2.items():
            ref = self._lamA[nome][:, None] * self._lamG[nome][None, :]
            n = tf.norm(ref)
            desvios.append(float(tf.norm(m2 - ref) / tf.maximum(n, 1e-12)))
        return float(np.mean(desvios))

    def resumo(self):
        r = super().resumo()
        r["escalas_medidas"] = len(self._m2)
        r["desvio_de_kronecker"] = self.desvio_de_kronecker()
        return r


# --- snakeai/agents/ppo.py ---
"""PPO — a implementação de referência do benchmark.

Enxuta, mas com os detalhes que decidem se um PPO aprende ou vira ruído (a lista do
*"37 Implementation Details of PPO"*):

* **GAE(λ)** com bootstrap correto no truncamento por fome — que é diferente de morte;
* **clipping** da razão **e** do valor;
* normalização de vantagem **por minibatch**;
* **early stop por KL aproximado**, que impede o colapso quando o LR está alto demais;
* **entropia com decaimento** — explora cedo, fica determinística no fim;
* **máscara de ação aplicada aos logits no rollout _e_ no update.** Este é o detalhe que
  mais silenciosamente destrói um PPO com máscara: se o update não reaplica a máscara, o
  `log_prob` calculado lá não bate com o que gerou a ação, a razão vira lixo e o algoritmo
  otimiza uma coisa que não existe.

Sobre o truncamento por fome
----------------------------
Morrer e ficar sem comida são coisas diferentes. Morte é terminação: o retorno acabou, e o
valor do estado seguinte é zero. Fome é **truncamento**: o episódio continuaria, e cortar
ali sem fazer bootstrap ensina o agente que sobreviver muito tempo é ruim. O `VecSnake`
devolve a observação terminal dos truncados justamente para isso, e o `collect` soma
`γ · V(s_final)` à recompensa daquele passo.
"""


import os
from dataclasses import dataclass

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
import numpy as np
import tensorflow as tf


__all__ = ["PPOConfig", "PPO", "compute_gae", "variancia_explicada"]


@dataclass
class PPOConfig(BaseConfig):
    num_envs: int = 512
    #: 32, e não 96, desde a ablação de orçamento: com o mesmo orçamento de ambiente, o
    #: rollout curto multiplica por ~16 as atualizações de gradiente. Ver
    #: `docs/ORCAMENTO_DE_GRADIENTE.md` e `PPOConfig.esparso`.
    rollout: int = 32

    gamma: float = 0.995
    gae_lambda: float = 0.95
    clip_eps: float = 0.2
    vf_coef: float = 0.5
    vf_clip: float = 0.2
    ent_coef_start: float = 0.02
    ent_coef_end: float = 0.002
    max_grad_norm: float = 0.5
    lr_start: float = 3e-4
    lr_end: float = 5e-5
    #: O eixo que substitui o K-FAC. Ver `snakeai/otimizadores.py`.
    optimizer: str = "adam"
    epochs: int = 4
    minibatches: int = 32
    target_kl: float = 0.03

    #: Shaping potencial, com coeficiente que decai a zero em `shaping_frac` do treino.
    #: Decair a zero é o que garante que a política ótima final seja a do problema real.
    shaping_start: float = 0.5
    shaping_frac: float = 0.25

    #: Sexto canal com o relógio da fome. **Fora do contrato** — ver `VecSnake`. Ligar isto
    #: sem marcar `comparable=False` levanta erro, porque a entrada da rede muda e a curva
    #: deixa de ser comparável com qualquer outra do repositório.
    canal_fome: bool = False

    @classmethod
    def esparso(cls, **kw):
        """O orçamento de gradiente **anterior** — a configuração que a ablação aposentou.

        Existe para reproduzir o braço de controle, não para uso normal. Os 5 M passos do
        contrato são de *ambiente*; quantas atualizações de gradiente se tira deles é
        escolha livre, e a escolha antiga gastava pouquíssimo:

        =========================  ==============  ==============
        \                          `esparso()`     padrão
        =========================  ==============  ==============
        `rollout`                  96              32
        amostras por iteração      49.152          16.384
        iterações em 5 M passos    ~102            ~305
        `epochs` × `minibatches`   3 × 8           4 × 32
        tamanho do minilote        6.144           512
        **atualizações no total**  **~2.400**      **~38.300**
        =========================  ==============  ==============

        Medido em três sementes: 62,19 contra 80,90 de score, 4,4% contra 60,1% de
        tabuleiro cheio, e desvio entre sementes de 9,79 contra 1,80 — o orçamento não só
        levantou a média como **colapsou a dispersão** por um fator de 5,4. Ver
        `docs/ORCAMENTO_DE_GRADIENTE.md`.

        A variante ganha o sufixo `_esparso`: as duas configurações competem no mesmo
        contrato, mas não são a mesma coisa, e identidade `(algo, variant, seed)` repetida
        vira uma curva só na arena.
        """
        kw.setdefault("sufixo_variante", "esparso")
        return cls(rollout=96, epochs=3, minibatches=8, **kw)

    def __post_init__(self):
        super().__post_init__()
        if self.canal_fome and self.comparable:
            raise ValueError(
                "canal_fome=True muda a observação de 5 para 6 canais e portanto a "
                "entrada da rede. Marque comparable=False e escreva o caveat.")

    @property
    def batch_size(self):
        return self.num_envs * self.rollout


# ------------------------------------------------------------------------- GAE
def compute_gae(rewards, values, dones, last_value, gamma, lam):
    """GAE(λ) padrão, em NumPy.

    O bootstrap de truncamento já foi somado à recompensa em `collect`, então aqui todo
    `done` pode ser tratado como terminal — sem isso o valor do episódio *seguinte*
    vazaria para o anterior.
    """
    T, N = rewards.shape
    adv = np.zeros((T, N), dtype=np.float32)
    ultimo = np.zeros(N, dtype=np.float32)
    for t in reversed(range(T)):
        v_prox = last_value if t == T - 1 else values[t + 1]
        continua = 1.0 - dones[t]
        delta = rewards[t] + gamma * v_prox * continua - values[t]
        ultimo = delta + gamma * lam * continua * ultimo
        adv[t] = ultimo
    return adv, adv + values


def variancia_explicada(valor, retorno):
    """`1 − Var(retorno − valor) / Var(retorno)` — o crítico explica quanto do retorno?

    1 é um crítico perfeito, 0 é um crítico que não vale mais que prever a média, e
    negativo é um crítico que atrapalha. É a métrica que diz se a vantagem do GAE está
    sendo calculada sobre uma baseline útil ou sobre ruído — e é a evidência que decide se
    o `vf_clip` em unidades absolutas está travando o crítico, porque com o valor preso a
    ±`vf_clip` por iteração ele nunca alcança a escala do retorno.
    Ver `docs/REVISAO_ALGORITMOS.md` §2.2.
    """
    retorno = np.asarray(retorno, dtype=np.float64)
    var = retorno.var()
    if var < 1e-12:                      # retorno constante: a razão não significa nada
        return float("nan")
    return float(1.0 - (retorno - np.asarray(valor, dtype=np.float64)).var() / var)


# ------------------------------------------------------------------ forward TF
@tf.function(reduce_retracing=True)
def policy_forward(model, obs, mask):
    logits, valor = model(obs, training=False)
    logits = tf.where(mask, logits, tf.fill(tf.shape(logits), MASK_NEG))
    return logits, tf.squeeze(valor, -1)


@tf.function(reduce_retracing=True)
def sample_actions(model, obs, mask):
    logits, valor = policy_forward(model, obs, mask)
    acoes = tf.random.categorical(logits, 1, dtype=tf.int32)[:, 0]
    logp_all = tf.nn.log_softmax(logits)
    logp = tf.gather(logp_all, acoes, batch_dims=1)
    return acoes, logp, valor


def make_optimizer(cfg, model):
    """Cria o Adam e **constrói os slots na hora**.

    Sem o `build()` explícito, o Adam só cria os momentos na primeira chamada de
    `apply_gradients` — que acontece dentro de uma `tf.function` já traçada, e aí estoura
    *"tf.function only supports singleton tf.Variables created on the first call"*.
    Na prática isso quebrava o segundo `PPO(...)` da sessão: retomar de um checkpoint, ou
    simplesmente rodar a célula de treino duas vezes no Colab.
    """
    opt = cria_otimizador(getattr(cfg, "optimizer", "adam"), cfg.lr_start,
                          clipnorm=cfg.max_grad_norm)
    opt.build(model.trainable_variables)
    return opt


class PPO(AgentBase):
    algo = "ppo"

    def __init__(self, cfg: PPOConfig = None, model=None, variant=None):
        cfg = cfg or PPOConfig()
        super().__init__(cfg, variant=variant or cfg.net)
        keras.utils.set_random_seed(cfg.seed)
        self.env = VecSnake(cfg.num_envs, cfg.board_size,
                            rng=np.random.default_rng(cfg.seed),
                            canal_fome=getattr(cfg, "canal_fome", False))
        # A rede é construída a partir do **ambiente**, não de uma constante: se as duas
        # fontes discordarem, o erro aparece só na primeira multiplicação de matriz, com
        # uma mensagem sobre formas que não diz nada sobre canal de fome.
        self.model = model or build_actor_critic(cfg.board_size, cfg.net,
                                                 canais=self.env.n_channels)
        self.optimizer = make_optimizer(cfg, self.model)
        self.obs, self.mask = self.env.reset()

    def on_model_reloaded(self):
        self.optimizer = make_optimizer(self.cfg, self.model)

    # ------------------------------------------------------------ agendamentos
    def lr(self):
        return self.linear(self.cfg.lr_start, self.cfg.lr_end)

    def ent_coef(self):
        return self.linear(self.cfg.ent_coef_start, self.cfg.ent_coef_end)

    def shaping(self):
        f = self.frac()
        return max(0.0, self.cfg.shaping_start * (1.0 - f / self.cfg.shaping_frac))

    # ----------------------------------------------------------------- rollout
    def collect(self):
        cfg = self.cfg
        T, N = cfg.rollout, cfg.num_envs
        # do ambiente, não da constante: com `canal_fome` são 6, e um buffer de 5 falharia
        # só aqui, com uma mensagem sobre formas que não menciona o canal de fome
        b, c = cfg.board_size, self.env.n_channels

        obs_buf = np.empty((T, N, b, b, c), dtype=np.float32)
        mask_buf = np.empty((T, N, N_ACTIONS), dtype=bool)
        act_buf = np.empty((T, N), dtype=np.int32)
        logp_buf = np.empty((T, N), dtype=np.float32)
        val_buf = np.empty((T, N), dtype=np.float32)
        rew_buf = np.empty((T, N), dtype=np.float32)
        done_buf = np.empty((T, N), dtype=np.float32)

        shaping = self.shaping()
        scores, passos_ep, vitorias = [], [], 0

        for t in range(T):
            obs_buf[t] = self.obs
            mask_buf[t] = self.mask
            a, lp, v = sample_actions(self.model,
                                      tf.convert_to_tensor(self.obs),
                                      tf.convert_to_tensor(self.mask))
            a = a.numpy()
            act_buf[t], logp_buf[t], val_buf[t] = a, lp.numpy(), v.numpy()

            self.obs, self.mask, r, d, info = self.env.step(a, shaping, cfg.gamma)
            self.registra_fim(info)
            rew_buf[t] = r
            done_buf[t] = d.astype(np.float32)

            if info["trunc_idx"].size:       # fome é truncamento, não terminação
                _, vf = policy_forward(self.model,
                                       tf.convert_to_tensor(info["final_obs"]),
                                       tf.convert_to_tensor(info["final_mask"]))
                rew_buf[t] = self.bootstrap_truncados(info, rew_buf[t], vf.numpy(),
                                                      cfg.gamma)

            scores.extend(info["scores"].tolist())
            passos_ep.extend(info["lengths"].tolist())
            vitorias += info["wins"]

        _, ultimo_v = policy_forward(self.model,
                                     tf.convert_to_tensor(self.obs),
                                     tf.convert_to_tensor(self.mask))
        adv, ret = compute_gae(rew_buf, val_buf, done_buf, ultimo_v.numpy(),
                               cfg.gamma, cfg.gae_lambda)

        self.global_step += T * N
        self.episodes += len(scores)

        def achata(x, forma):
            return x.reshape((T * N,) + forma)

        lote = {
            "obs": achata(obs_buf, (b, b, c)),
            "mask": achata(mask_buf, (N_ACTIONS,)),
            "act": achata(act_buf, ()),
            "logp": achata(logp_buf, ()),
            "adv": achata(adv, ()),
            "ret": achata(ret, ()),
            "val": achata(val_buf, ()),
        }
        stats = {
            "train_score_mean": float(np.mean(scores)) if scores else None,
            "train_score_p95": float(np.percentile(scores, 95)) if scores else None,
            "train_ep_steps": float(np.mean(passos_ep)) if passos_ep else None,
            "n_episodes": len(scores),
            "wins": vitorias,
            "shaping": shaping,
        }
        return lote, stats

    # ------------------------------------------------------------------ update
    @staticmethod
    @tf.function(reduce_retracing=True)
    def _train_step(model, optimizer, obs, mask, act, old_logp, adv, ret, old_val,
                    clip_eps, vf_coef, vf_clip, ent_coef):
        adv = (adv - tf.reduce_mean(adv)) / (tf.math.reduce_std(adv) + 1e-8)
        with tf.GradientTape() as tape:
            logits, valor = model(obs, training=True)
            valor = tf.squeeze(valor, -1)
            # a máscara TEM que ser reaplicada aqui: sem isso o log_prob do update não
            # bate com o do rollout e a razão do PPO vira ruído
            logits = tf.where(mask, logits, tf.fill(tf.shape(logits), MASK_NEG))
            logp_all = tf.nn.log_softmax(logits)
            logp = tf.gather(logp_all, act, batch_dims=1)

            razao = tf.exp(logp - old_logp)
            pg1 = -adv * razao
            pg2 = -adv * tf.clip_by_value(razao, 1.0 - clip_eps, 1.0 + clip_eps)
            pg_loss = tf.reduce_mean(tf.maximum(pg1, pg2))

            v_clip = old_val + tf.clip_by_value(valor - old_val, -vf_clip, vf_clip)
            v_loss = 0.5 * tf.reduce_mean(
                tf.maximum(tf.square(valor - ret), tf.square(v_clip - ret))
            )

            probs = tf.exp(logp_all)
            seguro = tf.where(mask, logp_all, tf.zeros_like(logp_all))
            entropia = -tf.reduce_mean(tf.reduce_sum(probs * seguro, axis=-1))

            perda = pg_loss + vf_coef * v_loss - ent_coef * entropia

        grads = tape.gradient(perda, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        log_razao = logp - old_logp
        # estimador k3 do KL: não-negativo e de baixa variância, ao contrário de -log_ratio
        kl = tf.reduce_mean(tf.exp(log_razao) - 1.0 - log_razao)
        clipfrac = tf.reduce_mean(
            tf.cast(tf.greater(tf.abs(razao - 1.0), clip_eps), tf.float32)
        )
        return pg_loss, v_loss, entropia, kl, clipfrac

    def update(self, lote):
        cfg = self.cfg
        self.optimizer.learning_rate.assign(self.lr())
        ent = self.ent_coef()
        n = lote["act"].shape[0]
        mb = max(1, n // cfg.minibatches)
        idx = np.arange(n)
        rng = np.random.default_rng(cfg.seed + self.iteration)

        # Os escalares vão como **tensores**, não como floats Python: float Python entra
        # na assinatura da `tf.function` e o `ent_coef` muda a cada iteração, então cada
        # iteração recompilava o grafo inteiro e retinha mais uma `ConcreteFunction`. O
        # `reduce_retracing=True` relaxa formas de tensor, não escalares Python. Ver
        # `docs/REVISAO_ALGORITMOS.md` §2.6.
        escalares = [tf.constant(v, tf.float32)
                     for v in (cfg.clip_eps, cfg.vf_coef, cfg.vf_clip, ent)]
        tensores = {k: tf.convert_to_tensor(v) for k, v in lote.items()}
        logs = {"pg": [], "vf": [], "ent": [], "kl": [], "clipfrac": []}
        parar = False
        epocas_feitas = 0
        atualizacoes = 0
        for _ in range(cfg.epochs):
            rng.shuffle(idx)
            for s in range(0, n, mb):
                sl = tf.convert_to_tensor(idx[s:s + mb])
                pg, vf, e, kl, cf = self._train_step(
                    self.model, self.optimizer,
                    tf.gather(tensores["obs"], sl), tf.gather(tensores["mask"], sl),
                    tf.gather(tensores["act"], sl), tf.gather(tensores["logp"], sl),
                    tf.gather(tensores["adv"], sl), tf.gather(tensores["ret"], sl),
                    tf.gather(tensores["val"], sl), *escalares,
                )
                logs["pg"].append(float(pg)); logs["vf"].append(float(vf))
                logs["ent"].append(float(e)); logs["kl"].append(float(kl))
                logs["clipfrac"].append(float(cf))
                atualizacoes += 1
                if float(kl) > cfg.target_kl * 1.5:
                    parar = True
                    break
            epocas_feitas += 1
            if parar:
                break
        saida = {k: float(np.mean(v)) for k, v in logs.items()}
        saida["epochs_done"] = epocas_feitas
        saida["atualizacoes"] = int(atualizacoes)
        saida["ev"] = variancia_explicada(lote["val"], lote["ret"])
        saida["lr"] = float(self.lr())
        saida["ent_coef"] = ent
        return saida

    # ------------------------------------------------------------------ passo
    def iterate(self):
        lote, stats = self.collect()
        stats.update(self.update(lote))
        return stats


# --- snakeai/agents/a2c.py ---
"""A2C — actor-critic síncrono, o controle experimental do PPO.

O A2C é o PPO sem as duas coisas que definem o PPO: **sem clipping da razão** e **uma
única passada de gradiente por rollout**. Tudo o mais é igual — mesmo ambiente, mesma
rede, mesmo GAE, mesmo bootstrap de truncamento, mesmo agendamento de entropia.

Por isso ele é mais que "mais um algoritmo": é o **controle experimental**. A diferença
entre a curva do PPO e a do A2C mede exatamente quanto valem o clipping e o reaproveitamento
do rollout, com todo o resto congelado. Sem esse controle, o ganho do PPO poderia ser do
ambiente novo, da rede residual ou do shaping — e não haveria como saber.

A herança direta de `PPO` é deliberada: garante que o `collect()` seja *literalmente* o
mesmo código, não uma cópia que diverge com o tempo. Uma correção no rollout vale para os
dois na hora.
"""


from dataclasses import dataclass

import numpy as np
import tensorflow as tf


__all__ = ["A2CConfig", "A2C"]


@dataclass
class A2CConfig(PPOConfig):
    #: Rollouts curtos são o normal em A2C: sem clipping, dar passos grandes com dados
    #: velhos desestabiliza. **5 é o valor canônico** (o `t_max` do A3C de Mnih et al.), e
    #: aqui ele é também o que maximiza o orçamento de gradiente sem descaracterizar o
    #: algoritmo.
    #:
    #: Este é o ponto delicado da comparação com o PPO. O A2C dá **uma** atualização por
    #: rollout, por definição — reaproveitar o rollout em várias épocas é o que o PPO faz,
    #: e fazer isso aqui transformaria o controle no tratado. Então o orçamento dele tem
    #: teto estrutural: 1.953 atualizações com `rollout=5`, contra 38.300 do PPO. A
    #: ablação de orçamento mostrou que esse eixo vale ~18 pontos no PPO, então a
    #: diferença entre as duas curvas mede clipping **mais** orçamento, e o artigo precisa
    #: dizer isso — não há como igualar. Ver `docs/ORCAMENTO_DE_GRADIENTE.md`.
    rollout: int = 5
    lr_start: float = 7e-4
    lr_end: float = 1e-4
    ent_coef_start: float = 0.02
    ent_coef_end: float = 0.002
    vf_coef: float = 0.5

    @classmethod
    def esparso(cls, **kw):
        """O `rollout=16` que era o padrão antes do orçamento virar eixo declarado.

        Sobrescreve o `esparso()` do PPO, que mexe em `epochs` e `minibatches` — botões
        que no A2C não existem. Aqui o único botão é o rollout.
        """
        kw.setdefault("sufixo_variante", "esparso")
        return cls(rollout=16, **kw)

    #: Campos do PPO que não existem aqui. Ficam para o `dataclass` não brigar, mas o
    #: `A2C` ignora — e o teste `test_a2c_ignores_ppo_only_knobs` garante que ignora.
    epochs: int = 1
    minibatches: int = 1
    clip_eps: float = 0.0
    vf_clip: float = 0.0
    target_kl: float = 0.0


class A2C(PPO):
    algo = "a2c"

    def __init__(self, cfg: A2CConfig = None, model=None, variant=None):
        super().__init__(cfg or A2CConfig(), model=model, variant=variant)

    @staticmethod
    @tf.function(reduce_retracing=True)
    def _train_step_a2c(model, optimizer, obs, mask, act, adv, ret, ent_coef, vf_coef):
        adv = (adv - tf.reduce_mean(adv)) / (tf.math.reduce_std(adv) + 1e-8)
        with tf.GradientTape() as tape:
            logits, valor = model(obs, training=True)
            valor = tf.squeeze(valor, -1)
            # mesma regra do PPO: a máscara vale no update também
            logits = tf.where(mask, logits, tf.fill(tf.shape(logits), MASK_NEG))
            logp_all = tf.nn.log_softmax(logits)
            logp = tf.gather(logp_all, act, batch_dims=1)

            # o gradiente de política puro: sem razão, sem clipping
            pg_loss = -tf.reduce_mean(logp * adv)
            v_loss = 0.5 * tf.reduce_mean(tf.square(valor - ret))

            probs = tf.exp(logp_all)
            seguro = tf.where(mask, logp_all, tf.zeros_like(logp_all))
            entropia = -tf.reduce_mean(tf.reduce_sum(probs * seguro, axis=-1))

            perda = pg_loss + vf_coef * v_loss - ent_coef * entropia

        grads = tape.gradient(perda, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return pg_loss, v_loss, entropia

    def update(self, lote):
        """Uma passada de gradiente sobre o rollout inteiro, e o dado é descartado.

        É esta linha que separa o A2C do PPO: sem clipping, reaproveitar o rollout por
        várias épocas faria a política se afastar demais dos dados que a geraram.
        """
        cfg = self.cfg
        self.optimizer.learning_rate.assign(self.lr())
        ent = self.ent_coef()

        # escalares como tensores — ver a nota em `PPO.update` (§2.6 da revisão). Aqui
        # dói mais: o A2C dá uma atualização por iteração, então eram 610 recompilações
        # com `rollout=16` e 1.953 com o rollout canônico de 5.
        pg, vf, e = self._train_step_a2c(
            self.model, self.optimizer,
            tf.convert_to_tensor(lote["obs"]), tf.convert_to_tensor(lote["mask"]),
            tf.convert_to_tensor(lote["act"]), tf.convert_to_tensor(lote["adv"]),
            tf.convert_to_tensor(lote["ret"]),
            tf.constant(ent, tf.float32), tf.constant(cfg.vf_coef, tf.float32),
        )
        return {
            "pg": float(pg), "vf": float(vf), "ent": float(e),
            "lr": float(self.lr()), "ent_coef": ent, "epochs_done": 1,
            "atualizacoes": 1,
            "ev": variancia_explicada(lote["val"], lote["ret"]),
        }


# --- snakeai/agents/acktr.py ---
"""ACKTR — A2C com gradiente natural via K-FAC e região de confiança.

*Actor-Critic using Kronecker-factored Trust Region* (Wu et al., 2017). É o A2C com uma
única troca: onde o A2C anda na direção `∇`, o ACKTR anda em `F⁻¹∇`, com o tamanho do passo
escolhido para que a divergência KL entre a política velha e a nova não passe de um alvo.

Por que ele fecha uma dívida deste repositório
----------------------------------------------
Quatro notebooks do `colab-rl` tentaram K-FAC — `snakeai_dqn_kfac_cnn3`,
`snakeai_dqn_kfac_kl_divergence_cnn3`, `kfac_optimizer_test`, `new_kfac`. Nenhum roda hoje:
dependiam de `tensorflow.contrib.kfac`, que sumiu no TF2. A pergunta por trás deles — *vale
a pena aproximar a curvatura?* — ficou sem resposta por sete anos.

Aqui ela tem resposta medida, e de graça, por causa de uma escolha de projeto anterior: o
`A2C` já existe e já é o controle experimental do PPO. **ACKTR é o A2C com K-FAC ligado, e
nada mais.** Herda `collect`, herda o GAE, herda o agendamento de entropia, herda o
bootstrap de truncamento. A diferença entre as duas curvas na arena é atribuível ao
gradiente natural, e a mais nada.

Dois detalhes que decidem se funciona
-------------------------------------
**A KL escolhe o passo, não o learning rate.** Com `Δ = F⁻¹∇`, a KL induzida por um passo
`ηΔ` vale `½η²·Δᵀ∇`. Igualando ao alvo sai `η = √(2·kl_max / Δᵀ∇)`. Isso é o que permite ao
ACKTR usar passos que derrubariam um A2C: quando a curvatura é baixa ele anda muito, quando
é alta ele encolhe sozinho. O `lr` vira apenas um **teto**.

**A curvatura vem de uma perda separada.** As estatísticas de `G` saem de `log π(a')` com
`a'` amostrada da política — não da perda de RL. Ver `snakeai/kfac.py`, seção "Fisher de
verdade".

O que a primeira execução longa mostrou sobre a região de confiança
-------------------------------------------------------------------
Numa execução de 5 M passos (`resnet_small`, semente 0), a KL **medida depois do passo**
ficou sistematicamente acima do alvo — e o registro está aqui porque a primeira leitura que
fizemos destes números estava errada em duas frentes ao mesmo tempo.

============  ==============  ========  ==========
quinto        KL mediana      × alvo    entropia
============  ==============  ========  ==========
1             0,0237          11,8      0,158
2             0,0248          12,4      0,069
3             0,0150           7,5      0,053
4             0,0105           5,2      0,041
5             0,0088           4,4      0,041
============  ==============  ========  ==========

O estouro é **maior no começo e diminui ao longo do treino** — o contrário do que se lê
olhando as últimas linhas do log, que são pontos isolados de 0,03–0,06 e não a mediana. E a
correlação entre `log(KL)` e entropia é fraca (−0,26), então "a política ficou determinística
demais" **não** explica: o pior estouro acontece justamente quando a entropia é a mais alta
da execução.

A explicação que sobra é a própria aproximação. `Δᵀ∇ = ΔᵀF̃Δ`, com `F̃` a Fisher *aproximada*
— bloco-diagonal por camada, e cada bloco um produto de Kronecker. A KL medida é a da
política de verdade. Onde `F̃` subestima a curvatura real, `Δ` fica grande demais naquelas
direções e a KL prevista sai baixa. Que o erro encolha conforme a média móvel dos fatores
amadurece é consistente com isso. (Não é o amortecimento: com `Δ = (F̃ + λI)⁻¹∇`, tem-se
`Δᵀ∇ = ΔᵀF̃Δ + λ‖Δ‖²`, que **super**estima a forma quadrática e portanto *encolhe* o passo.)

Duas consequências práticas, ambas medidas:

* Em **100% das atualizações** o passo veio da fórmula da KL, nunca do teto do `lr`. Os
  `lr_start`/`lr_end` do ACKTR não limitaram nada nesta execução — quem governa é `kl_max`.
* `kl_max = 0,002` entrega, na prática, KL ≈ 0,01. O parâmetro é um alvo *aproximado* com um
  fator de escala que depende da qualidade de `F̃`. Apertá-lo encolhe todo passo por `√k`.

`stats["kl"]` existe exatamente para que isso seja visível em vez de suposto — e a lição de
método é que a mediana por fase diz uma coisa que as últimas linhas do log dizem ao contrário.

Custo
-----
Uma retropropagação extra por atualização (a perda de Fisher) e as fatorações de Cholesky
a cada `inv_every` passos. Como a atualização acontece uma vez por rollout — 512 × 16 =
8.192 passos de ambiente — o custo se dilui. `stats["kfac_ms"]` mede quanto sobrou.
"""


import time
from dataclasses import dataclass

import numpy as np
import tensorflow as tf


__all__ = ["ACKTRConfig", "ACKTR"]


@dataclass
class ACKTRConfig(A2CConfig):
    #: Teto do passo. No ACKTR o tamanho normalmente é decidido pela KL; o `lr` só impede
    #: que um lote de curvatura quase nula peça um passo absurdo.
    lr_start: float = 0.5
    lr_end: float = 0.1

    #: Alvo de KL por atualização. Wu et al. usam 0,001–0,002 no Atari; aqui o alvo
    #: **entregue** é o que importa, porque o pedido passa pela Fisher aproximada antes de
    #: virar passo — e é isso que `kl_calibrado` fecha.
    #:
    #: Três sementes com este valor, e uma leitura que **não** funcionou:
    #:
    #: =======  =============  =======  ========
    #: semente  KL entregue    final    cheio
    #: =======  =============  =======  ========
    #: 0        0,0068         89,78    89,7%
    #: 1        0,0097         70,67    43,7%
    #: 2        0,0143         78,13    60,7%
    #: =======  =============  =======  ========
    #:
    #: Média 79,52, desvio **9,63**. Cheguei a escrever aqui que existia um ótimo interior
    #: perto de 0,0068; a semente 2 desmente — ela entrega o **dobro** de KL e joga melhor
    #: que a semente 1. A KL entregue não explica a dispersão, e o que sobra é semente.
    #:
    #: Para comparação, o PPO no orçamento padrão faz 80,90 com desvio 1,80: mesma média,
    #: **5,3× menos dispersão**. É o resultado honesto sobre o ACKTR neste ambiente — não
    #: que ele seja pior, e sim que ele é imprevisível. Ver
    #: `docs/ORCAMENTO_DE_GRADIENTE.md`.
    kl_max: float = 1.5e-2

    #: Amortecimento de Tikhonov. Alto demais e o ACKTR vira A2C com passo esquisito;
    #: baixo demais e a inversa amplifica direções que o lote mal estimou.
    damping: float = 1e-2

    #: Média móvel dos fatores entre atualizações. Absorve o ruído de amostragem da Fisher.
    kfac_ema: float = 0.95

    #: A cada quantas atualizações refatorar. As Cholesky custam O(d³) nos fatores, que
    #: são pequenos — mas `A` da primeira convolução ainda é 288×288.
    inv_every: int = 10

    #: Peso da parte gaussiana na perda de Fisher. Wu et al. usam 1,0 quando ator e crítico
    #: compartilham tronco, que é o caso aqui.
    fisher_vf_coef: float = 1.0

    #: **Calibra a região de confiança pela KL que de fato aconteceu.**
    #:
    #: Sem isto, `kl_max` é um alvo nominal: a execução de 5 M passos pediu 0,002 e
    #: entregou ~0,01, porque `Δᵀ∇` usa a Fisher *aproximada* e a KL medida é a da política
    #: de verdade. Ligado, o agente estima o fator sistemático `c = KL_medida / alvo_pedido`
    #: por média móvel e pede `kl_max / c` — de modo que a KL **entregue** convirja para
    #: `kl_max`.
    #:
    #: **Ligado por padrão desde a medição.** Era eixo de ablação e virou o comportamento
    #: oficial: sem calibrar, a mesma configuração e a mesma semente entregaram 83,91 num
    #: Colab de agosto e 64,53 num Kaggle depois — o fator não controlado entre a Fisher
    #: aproximada e a KL real muda com o hardware, e o resultado deixa de ser reprodutível.
    #: Desligar isto é a ablação, e a variante ganha `+kl_nominal` para dizer isso.
    kl_calibrado: bool = True

    #: Média móvel do fator. Alta porque `c` é ruidoso lote a lote.
    kl_cal_ema: float = 0.98

    #: Limites do fator, para um lote patológico não travar a calibração num extremo.
    kl_cal_min: float = 0.05
    kl_cal_max: float = 200.0

    optimizer: str = "sgd"


class ACKTR(A2C):
    """A2C + K-FAC. Ver o docstring do módulo para o porquê da herança direta."""

    algo = "acktr"

    def __init__(self, cfg: ACKTRConfig = None, model=None, variant=None):
        super().__init__(cfg or ACKTRConfig(), model=model, variant=variant)
        c = self.cfg
        self.kfac = self._cria_precondicionador()
        # `on_model_reloaded` recria o otimizador; o K-FAC tem que acompanhar, senão os
        # índices de `trainable_variables` apontam para o modelo antigo.
        self._ultimo = {}
        #: Fator sistemático entre a KL pedida e a entregue. Começa em 1 — ou seja, a
        #: primeira atualização é idêntica à da versão não calibrada, e a correção só
        #: aparece conforme a medição chega.
        self._fator_kl = 1.0
        if variant is None:
            self.variant = self._com_sufixo(self._variante_da_regiao(c),
                                            getattr(c, "sufixo_variante", ""))

    def _cria_precondicionador(self):
        """Qual curvatura este agente usa.

        Existe como método, e não como uma linha dentro do `__init__`, porque é o **único**
        ponto que o `ACEKTR` sobrescreve. Enquanto for só isto, a diferença entre as duas
        curvas na arena é atribuível à correção de autovalores e a mais nada — e
        `tests/test_ekfac.py` confere que continua sendo só isto.
        """
        c = self.cfg
        return KFac(self.model, damping=c.damping, ema=c.kfac_ema,
                    inv_every=c.inv_every)

    @staticmethod
    def _variante_da_regiao(cfg):
        """A variante diz em que região de confiança a execução rodou.

        O padrão — calibrado no alvo medido — não ganha marca nenhuma: é o ACKTR oficial.
        Qualquer desvio aparece no nome, porque `load_all` agrupa por
        `(algo, variant, seed)` e duas regiões de confiança diferentes com a mesma
        identidade viram uma curva só. Foi assim que o ACKTR de 12/08 e o de agora quase
        se fundiram na arena.
        """
        marcas = []
        if not cfg.kl_calibrado:
            marcas.append("kl_nominal")
        if cfg.kl_max != type(cfg).kl_max:
            marcas.append(f"kl{cfg.kl_max:g}")
        return "+".join([cfg.net] + marcas)

    # ------------------------------------------------------------------ um passo
    def _forward_e_gradientes(self, obs, mask, act, adv, ret, ent_coef, vf_coef):
        """Um forward, duas retropropagações: a da perda real e a da perda de Fisher.

        Não é `tf.function`: a captura do K-FAC precisa reexecutar `call` a cada chamada e
        o `precondiciona` roda em eager de qualquer forma. Como isto acontece **uma vez por
        rollout**, o overhead de Python se dilui em milhares de passos de ambiente — o que
        foi medido, não suposto (`tools/perfil_dispositivo.py`).
        """
        adv = (adv - tf.reduce_mean(adv)) / (tf.math.reduce_std(adv) + 1e-8)

        with captura_kfac(self.kfac.camadas) as cap:
            with tf.GradientTape(persistent=True) as tape:
                logits, valor = self.model(obs, training=True)
                valor = tf.squeeze(valor, -1)
                logits = tf.where(mask, logits, tf.fill(tf.shape(logits), MASK_NEG))

                logp_all = tf.nn.log_softmax(logits)
                logp = tf.gather(logp_all, act, batch_dims=1)
                pg = -tf.reduce_mean(logp * adv)
                vl = 0.5 * tf.reduce_mean(tf.square(valor - ret))

                probs = tf.exp(logp_all)
                seguro = tf.where(mask, logp_all, tf.zeros_like(logp_all))
                ent = -tf.reduce_mean(tf.reduce_sum(probs * seguro, axis=-1))

                perda = pg + vf_coef * vl - ent_coef * ent

                # A perda de Fisher **não** é a perda de RL: ela define a métrica, não o
                # objetivo. O sinal negativo é porque queremos o gradiente do
                # log-likelihood, e `perda_fisher_categorica` já devolve a média de log π.
                pf = (-perda_fisher_categorica(logits, mask)
                      + self.cfg.fisher_vf_coef * perda_fisher_gaussiana(valor[:, None]))

            grads = tape.gradient(perda, self.model.trainable_variables)
            gs = tape.gradient(pf, [z for _, _, z in cap])
        del tape

        return grads, cap, gs, pg, vl, ent, logp_all

    def update(self, lote):
        cfg = self.cfg
        ent_coef = self.ent_coef()
        t0 = time.perf_counter()

        grads, cap, gs, pg, vl, ent, logp_velho = self._forward_e_gradientes(
            tf.convert_to_tensor(lote["obs"]), tf.convert_to_tensor(lote["mask"]),
            tf.convert_to_tensor(lote["act"]), tf.convert_to_tensor(lote["adv"]),
            tf.convert_to_tensor(lote["ret"]), ent_coef, cfg.vf_coef,
        )
        t_fwd = time.perf_counter() - t0

        t0 = time.perf_counter()
        self.kfac.acumula(cap, gs)
        naturais = self.kfac.precondiciona(grads)
        alvo_efetivo = cfg.kl_max / self._fator_kl if cfg.kl_calibrado else cfg.kl_max
        eta = self.kfac.escala_kl(naturais, grads, alvo_efetivo, self.lr())
        t_kfac = time.perf_counter() - t0

        self.optimizer.learning_rate.assign(float(eta))
        self.optimizer.apply_gradients(zip(naturais, self.model.trainable_variables))

        kl = self._kl_medida(lote, logp_velho)

        if cfg.kl_calibrado:
            # `c` é medido contra o que foi **pedido** nesta atualização, não contra
            # `kl_max`: pedir `kl_max/c` e depois comparar com `kl_max` realimentaria a
            # própria correção e a faria divergir.
            c = kl / max(alvo_efetivo, 1e-12)
            d = cfg.kl_cal_ema
            self._fator_kl = float(np.clip(d * self._fator_kl + (1 - d) * c,
                                           cfg.kl_cal_min, cfg.kl_cal_max))

        return {
            "pg": float(pg), "vf": float(vl), "ent": float(ent),
            "lr": float(eta), "lr_teto": float(self.lr()), "ent_coef": ent_coef,
            "kl": kl, "kl_alvo": cfg.kl_max, "kl_alvo_efetivo": float(alvo_efetivo),
            "kl_fator": self._fator_kl, "epochs_done": 1,
            "kfac_ms": t_kfac * 1e3, "fwd_ms": t_fwd * 1e3,
        }

    def _kl_medida(self, lote, logp_velho):
        """KL real depois do passo. O alvo é de segunda ordem; isto é o que aconteceu.

        Sem esta medida, `kl_max` seria um parâmetro que ninguém sabe se está sendo
        respeitado — e a aproximação quadrática se degrada exatamente quando o passo é
        grande, que é quando importa.
        """
        logits, _ = self.model(tf.convert_to_tensor(lote["obs"]), training=False)
        mask = tf.convert_to_tensor(lote["mask"])
        logits = tf.where(mask, logits, tf.fill(tf.shape(logits), MASK_NEG))
        novo = tf.nn.log_softmax(logits)
        p_velho = tf.exp(logp_velho)
        kl = tf.reduce_sum(tf.where(mask, p_velho * (logp_velho - novo),
                                    tf.zeros_like(novo)), axis=-1)
        return float(tf.reduce_mean(kl))

    # -------------------------------------------------------------------- relato
    def on_model_reloaded(self):
        super().on_model_reloaded()
        self.kfac = self._cria_precondicionador()

    def resumo_kfac(self):
        return self.kfac.resumo()


ASSINATURA_PACOTE = "9e11e1827d746f73"

# ==== FIM DO CÓDIGO GERADO ====

## Configuração

Os padrões abaixo são os do **contrato**: tabuleiro 10×10, 5 M passos de orçamento,
avaliação de 1.000 episódios com semente 123. Mexer neles é legítimo para experimentar,
mas o resultado só entra na arena se o contrato for respeitado — o `Recorder` recusa
qualquer outra coisa e diz o motivo.


In [ ]:
# @title Parâmetros
SEMENTE = 0        # @param {type:"integer"}
PASSOS = 5000000   # @param {type:"integer"}
REDE = "resnet_small"  # @param ["resnet_tiny", "resnet_small", "resnet_base", "cnn_rainbow", "cnn_alphazero", "cnn_vgg", "cnn_vgg_dropout", "cnn_vgg_sem_pool"]

# Armazenamento: nada para configurar. Detecta Colab, Kaggle ou máquina local e escolhe a
# pasta que **persiste** em cada um — Drive, /kaggle/working ou o diretório atual. Se a
# montagem do Drive falhar, avisa e segue, em vez de parar.
PASTA = pasta_de_trabalho()

# No Kaggle a sessão nova nasce com /kaggle/working vazio: o que sobreviveu está montado
# somente-leitura em /kaggle/input. Isto traz os checkpoints de volta — e nunca sobrescreve
# um checkpoint desta sessão, senão o treino andaria para trás.
semear_checkpoints(os.path.join(PASTA, "checkpoints"))

cfg = ACKTRConfig(
    seed=SEMENTE,
    net=REDE,
    total_steps=PASSOS,
    kl_calibrado=False,
    kl_max=2e-3,
    ckpt_dir=os.path.join(PASTA, "checkpoints"),
    runs_dir=os.path.join(PASTA, "runs"),
)
print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))

## Treino

**Retomável, e é requisito, não conveniência.** Um treino de 5 M passos não cabe numa
sessão gratuita sem cair pelo menos uma vez. Rode a célula de novo e ela continua do último
checkpoint.

* **Colab** — os checkpoints vão para o Drive e sobrevivem à queda da sessão.
* **Kaggle** — `/kaggle/working` vira a **saída** desta versão. Para continuar depois:
  *Save Version → Save & Run All* (roda headless, sem aba aberta), e na execução seguinte
  *Add Input → Your Work → Notebook Output* apontando para esta. A célula de parâmetros
  recupera os checkpoints sozinha.


In [ ]:
# @title Treinar
agente = ACKTR(cfg)
if agente.retomar("last"):
    print("retomando do checkpoint")
print("parâmetros:", f"{agente.model.count_params():,}")

registro = agente.train(verbose=True)

## Veredito — os dois modelos

Duas perguntas diferentes, dois números:

* **`last`** — o modelo do último passo. É ele que entra na curva e na arena, porque é o
  estado final do algoritmo, instabilidade inclusa.
* **`best`** — o melhor checkpoint já visto. É ele que você levaria para o jogo.

Os dois existem porque **RL profundo não melhora monotonicamente**: fora do caso tabular
não há garantia nenhuma, e uma execução pode terminar pior do que já esteve. Na primeira
execução longa do ACKTR, 8 das 21 avaliações tinham um checkpoint anterior melhor que o
modelo daquele momento — numa delas, 21,7 pontos melhor.

Dentro de cada um, três regimes: piso aleatório, política pura e política com o filtro de
segurança. Se a coluna do meio não estiver bem acima do piso, não aprendeu — e aí o
problema é hiperparâmetro ou tempo de treino, não código.


In [ ]:
# @title Veredito
print("=== last · modelo do último passo (é o que entra na arena) ===")
_fome = getattr(agente.env, "canal_fome", False)
resultado = verdict(agente.politica(), episodes=1000, canal_fome=_fome)
print(format_verdict(resultado))

melhor = agente.modelo_melhor()
if melhor is not None:
    print()
    print(f"=== best · checkpoint do passo "
          f"{registro.record.melhor.get('global_step', 0):,} ===")
    _guardado, agente.model = agente.model, melhor
    try:
        print(format_verdict(verdict(agente.politica(), episodes=1000,
                                     canal_fome=_fome)))
    finally:
        agente.model = _guardado

fig, _ = plot_run(registro.record)
plt.show()

## O agente jogando

Um GIF vale mais que a curva para entender *como* o agente perde. Morrer preso no próprio
corpo e morrer de fome dão a mesma linha no gráfico e são problemas completamente
diferentes.


In [ ]:
# @title GIF
from IPython.display import Image, display

for semente in (7, 21, 42):
    caminho, score, motivo = render_episode(
        agente.politica(), caminho=f"episodio_last_s{semente}.gif", seed=semente,
        canal_fome=getattr(agente.env, "canal_fome", False))
    print(f"last · semente {semente}: score {score}, terminou por {motivo}")
    display(Image(filename=caminho))

## Exportar — os dois

`.keras` para retomar treino, TFLite fp16/int8 para embarcar no jogo. A paridade de **ação**
contra o `.keras` é conferida — diferença numérica de quantização é aceitável, ação
diferente não é.

A conferência é pulada quando a política **tem memória** (o SOAP, com a crença de opção; o
DreamerV3, com o latente do modelo do mundo). Não é um detalhe de implementação: um
`.tflite` que recebe só a observação não consegue reproduzir uma política cuja ação depende
de estado interno, então "as ações batem" seria uma afirmação sobre outra coisa. Os arquivos
continuam sendo gerados e medidos; o que não se afirma é a paridade.

Exporta `last` **e** `best`, em pastas separadas. Exportar é para usar, e o que você leva
para o jogo é o melhor; mas o `last` vai junto porque é ele que corresponde ao número da
arena, e misturar os dois é como se perde a rastreabilidade entre o gráfico e o arquivo.


In [ ]:
# @title Exportar
relatorios = {}
# `apos_passo` é o contrato das políticas com memória (ver `snakeai/eval.py`). Quem o
# expõe não pode ter a paridade de ação conferida contra um `.tflite` sem estado.
_COM_MEMORIA = hasattr(agente.politica(), "apos_passo")
if _COM_MEMORIA:
    print("política com memória: TFLite exportado, paridade de ação não conferida")

relatorios["last"] = export_model(
    agente.model, out_dir=os.path.join(PASTA, "export", "last"),
    validar=not _COM_MEMORIA)

_melhor = agente.modelo_melhor()
if _melhor is not None:
    relatorios["best"] = export_model(
        _melhor, out_dir=os.path.join(PASTA, "export", "best"),
        validar=not _COM_MEMORIA)

print(json.dumps(relatorios, indent=2, ensure_ascii=False))

## Onde ficou o resultado

O `history.json` da execução vai para `runs/<algo>/<variante>/seed<N>/`, junto com a curva e
os GIFs. Essa pasta é o que entra na arena: coloque em `runs/` do repositório e rode
`python -m snakeai.arena --all`.

Ele carrega os dois resultados: `final` (o modelo do último passo, que é o número oficial)
e `melhor` (o melhor checkpoint, com o passo em que apareceu). Junto vão `modelos/last.keras`
e `modelos/best.keras` — a pasta é autossuficiente, quem a recebe consegue rodar o agente
sem depender de nada que ficou nesta máquina.

Sobre versionar isso no GitHub: o registro vai (`history.json`, `curva.png` e os GIFs), os
**pesos não**. Um `.keras` vai de 0,8 MB (`resnet_small`) a 6,7 MB (`cnn_rainbow` com dueling
e C51), e a arena inteira passa de 100 MB só de modelo — binário em git **nunca some do
histórico**, então cada re-execução deixaria mais uma cópia lá para sempre. O `.gitignore` já
tira `runs/**/*.keras` e `runs/**/*.npz`; o lugar deles é um *Release* do GitHub, que é feito
para binário e não entra no clone. Os arquivos continuam na sua pasta — o que muda é só o que
o git carrega.


In [ ]:
# @title Conferir o contrato
CAMINHO_REGISTRO = registro.save(skip_validation=True)
print("registro:", CAMINHO_REGISTRO)

problemas = validate(registro.record)
print("entra na arena?" , "sim" if not problemas else "NÃO:")
for p in problemas:
    print("  -", p)

_f = registro.record.final.get("score_mean")
_m = registro.record.melhor.get("score_mean")
if _f is not None and _m is not None:
    print()
    print(f"last  {_f:.2f}   (passo {registro.record.steps()[-1]:,})")
    print(f"best  {_m:.2f}   (passo {registro.record.melhor.get('global_step', 0):,})")
    if _m > _f:
        print(f"→ a execução terminou {_m - _f:.2f} abaixo do melhor que já esteve. "
              "Normal: RL profundo não melhora monotonicamente.")

## Baixar o resultado

Um `.zip` só, com a pasta inteira da execução — registro, curva, GIFs e o modelo exportado.

**Um arquivo, e não vários downloads**, por dois motivos: o navegador bloqueia downloads
múltiplos disparados em sequência, e a pasta da execução só faz sentido inteira — o
`history.json` sem a curva e sem os GIFs perde metade do que ela responde.

A entrega muda com a plataforma, e o `.zip` existe nos dois casos:

* **Colab** — dispara o download pelo navegador, o que exige a aba aberta. Se ela não
  estiver, a célula imprime o caminho em vez de falhar: o download é conveniência, o
  arquivo é o resultado.
* **Kaggle** — não há o que disparar, e é por isso que ele aguenta execução headless: o
  que está em `/kaggle/working` aparece sozinho no painel **Output**, à direita, e é
  baixável de lá com a aba fechada.


In [ ]:
# @title Baixar tudo num .zip
import shutil

PASTA_EXECUCAO = os.path.dirname(CAMINHO_REGISTRO)

# o export mora fora da pasta da execução; copiamos para dentro antes de zipar,
# senão o .zip sai sem o modelo — que é justamente o que se leva para o jogo
_export = os.path.join(PASTA, "export")
if os.path.isdir(_export):
    shutil.copytree(_export, os.path.join(PASTA_EXECUCAO, "export"), dirs_exist_ok=True)

_nome = "_".join([registro.record.algo, registro.record.variant,
                  f"seed{registro.record.seed}"])
ZIP = shutil.make_archive(os.path.join(PASTA, _nome), "zip", PASTA_EXECUCAO)
print(f"{ZIP}  ({os.path.getsize(ZIP) / 1e6:.1f} MB)")
for _raiz, _, _arqs in os.walk(PASTA_EXECUCAO):
    for _a in sorted(_arqs):
        print("   ", os.path.relpath(os.path.join(_raiz, _a), PASTA_EXECUCAO))

entregar_arquivo(ZIP)